In [137]:
import numpy as np
import math
import cvxpy as cp
import itertools
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, PillowWriter

sin = math.sin
cos = math.cos
tan = math.tan
th_max = np.pi/2
ph_max = np.pi/2
Ds=0.5

# coefficients for pairwise ECBF constraint
lam_pair = 5.1  # tune
alpha3 = 4.0 * lam_pair
alpha2 = 6.0 * lam_pair**2
alpha1 = 4.0 * lam_pair**3
alpha0 = lam_pair**4


def PelicanModelDirect(time, dt, state, control_inputs, system_parameters, sim_parameters):
    """
    One-step quad dynamics driven directly by control inputs:
    control_inputs = [up, uq, ur, u_thrust]
    state = [ph, th, ps, p, q, r, x, y, z, vx, vy, vz]
    """
    m = system_parameters["m"]
    g = system_parameters["g"]
    Ix = system_parameters["Ix"]
    Iy = system_parameters["Iy"]
    Iz = system_parameters["Iz"]
    kr = system_parameters.get("kr", 0.01)
    kt = system_parameters.get("kt", 0.01)

    dist_x = 0.1 * np.sin(time) if sim_parameters.get("disturbanceX", 0) else 0.0
    dist_y = 0.1 * np.sin(time) if sim_parameters.get("disturbanceY", 0) else 0.0
    dist_z = 0.1 * np.sin(time) if sim_parameters.get("disturbanceZ", 0) else 0.0

    up, uq, ur, u_thrust = control_inputs
    ph, th, ps, p, q, r, x, y, z, vx, vy, vz = state

    # attitude kinematics
    ph += dt * (p + (r * np.cos(ph) * np.sin(th)) / np.cos(th) + (q * np.sin(ph) * np.sin(th)) / np.cos(th))
    th += dt * (q * np.cos(ph) - r * np.sin(ph))
    ps += dt * ((r * np.cos(ph)) / np.cos(th) + (q * np.sin(ph)) / np.cos(th))

    # angular dynamics
    p += dt * (up - kr * p + (Iy - Iz) * q * r) / Ix
    q += dt * (uq - kr * q + (Iz - Ix) * p * r) / Iy
    r += dt * (ur - kr * r + (Ix - Iy) * p * q) / Iz

    # position
    x += dt * vx
    y += dt * vy
    z += dt * vz

    # translational dynamics
    vx += dt * (
        dist_x / m
        - (kt * vx) / m
        + (u_thrust * (np.sin(ph) * np.sin(ps) + np.cos(ph) * np.cos(ps) * np.sin(th))) / m
    )
    vy += -dt * (
        (kt * vy) / m
        - dist_y / m
        + (u_thrust * (np.cos(ps) * np.sin(ph) - np.cos(ph) * np.sin(ps) * np.sin(th))) / m
    )
    vz += -dt * (
        g
        - dist_z / m
        + (kt * vz) / m
        - (u_thrust * np.cos(ph) * np.cos(th)) / m
    )

    return np.array([ph, th, ps, p, q, r, x, y, z, vx, vy, vz], dtype=float)


# ----------------------------
# Per-quad configuration
# ----------------------------

class QuadConfig:
    def __init__(self, name, des_radius, des_x, des_y, des_height, des_yaw, des_speed,
                 gains, th_max, ph_max, cbf_lambdas, qp_weights):
        self.name = name

        # desired trajectory parameters (your circle)
        self.des_radius = float(des_radius)
        self.des_x = float(des_x)
        self.des_y = float(des_y)
        self.des_height = float(des_height)
        self.des_yaw = float(des_yaw)
        self.des_speed = float(des_speed)

        # gains dict: k1..k14 (or anything you need)
        self.gains = dict(gains)

        # angle constraints
        self.th_max = float(th_max)
        self.ph_max = float(ph_max)

        # cbf lambdas (for your single-quad roll/pitch cbfs)
        self.cbf_lambdas = dict(cbf_lambdas)  # expects {'lam_th':..., 'lam_ph':...}

        # QP weights
        self.qp_weights = dict(qp_weights)    # expects {'Q_u':..., 'P_delta':...}


# ----------------------------
# Environment (shared)
# ----------------------------

class Environment:
    def __init__(self, system_params, sim_params, M_map):
        self.system = dict(system_params)
        self.sim = dict(sim_params)
        self.M_map = np.array(M_map, dtype=float)

    def disturbances(self, t):
        # You can expand this to spatially varying or agent-dependent disturbances
        return np.array([0.0, 0.0, 0.0], dtype=float)


# ----------------------------
# Controller wrapper
# ----------------------------

def compute_tracking_terms(agent_id, state, zeta1, zeta2, cfg: QuadConfig, env: Environment, t, world_snapshot):
    """
    YOU IMPLEMENT THIS ONCE by moving your long Lie-derivative block here.

    Must return a dictionary with:
      - D: (4,4) numpy array
      - rhs: (4,) numpy array
      - cbf_terms: dict with the needed scalars to build your constraints:
            th: Lf2h_th, Lg3Lfh_th, Lg4Lfh_th, Lfh_th, h_th
            ph: Lf2h_ph, Lg2Lfh_ph, Lg3Lfh_ph, Lg4Lfh_ph, Lfh_ph, h_ph

    Also anything else you want to log.
    """
    # ---- unpack ----
    ph, th, ps, p, q, r, x, y, z, vx, vy, vz = state
    m = env.system["m"]
    kt = env.system.get("kt", 0.01)
    kr = env.system.get("kr", 0.01)
    Ix = env.system["Ix"]
    Iy = env.system["Iy"]
    Iz = env.system["Iz"]
    gains = cfg.gains
    g = env.system["g"]  
    

    # desired parameters
    des_radius = cfg.des_radius
    des_x = cfg.des_x
    des_y = cfg.des_y
    des_height = cfg.des_height
    des_yaw = cfg.des_yaw
    des_speed = cfg.des_speed

    # gains
    k1, k2, k3, k4 = gains["k1"], gains["k2"], gains["k3"], gains["k4"]
    k5, k6, k7, k8 = gains["k5"], gains["k6"], gains["k7"], gains["k8"]
    k9, k10, k11, k12 = gains["k9"], gains["k10"], gains["k11"], gains["k12"]
    k13, k14 = gains["k13"], gains["k14"]

    # NOTE:
    # Disturbances in your long expressions were dist_x/dist_y/dist_z.
    # Use env.disturbances(t) or keep them zero.
    dist_x, dist_y, dist_z = env.disturbances(t)

    # Transformed states
    xi1_1 = -des_radius**2 + (-des_x + x)**2 + (-des_y + y)**2
    xi1_2 = vx*(-2*des_x + 2*x) + vy*(-2*des_y + 2*y)
    xi1_3 = 2*vx**2 + 2*vy**2 + (-2*des_x + 2*x)*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(dist_y - kt*vy - zeta1*sin(ph)*cos(ps) + zeta1*sin(ps)*sin(th)*cos(ph))/m
    xi1_4 = zeta2*((-2*des_x + 2*x)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))/m) + (q*cos(ph) - r*sin(ph))*(zeta1*(-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m) + (-kt*(-2*des_x + 2*x)/m + 4*vx)*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m) + (-kt*(-2*des_y + 2*y)/m + 4*vy)*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m) + ((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th)) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th)) + 2*vx*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m + 2*vy*(dist_y - kt*vy - zeta1*sin(ph)*cos(ps) + zeta1*sin(ps)*sin(th)*cos(ph))/m

    Lg1Lf3S1 = (-2*des_x + 2*x)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))/m

    Lg2Lf3S1 = ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)/Ix

    Lg3Lf3S1 = (((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*sin(ph)/cos(th) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*sin(ph)*sin(th)/cos(th) + (zeta1*(-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m)*cos(ph))/Iy
    
    Lg4Lf3S1 = 0
    
    Lf4S1 = vx*(-2*kt*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/m + 2*zeta1*(q*cos(ph) - r*sin(ph))*cos(ph)*cos(ps)*cos(th)/m + 2*zeta2*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m + 2*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + 2*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/m) + vy*(-2*kt*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/m + 2*zeta1*(q*cos(ph) - r*sin(ph))*sin(ps)*cos(ph)*cos(th)/m + 2*zeta2*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))/m + 2*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m + 2*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/m) + zeta2*((q*cos(ph) - r*sin(ph))*((-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + (-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m) + ((-2*des_x + 2*x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th)) + ((-2*des_x + 2*x)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-sin(ph)*sin(ps)*sin(th) - cos(ph)*cos(ps))/m)*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th)) + 2*vx*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m + 2*vy*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))/m + (sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))*(-kt*(-2*des_x + 2*x)/m + 4*vx)/m - (sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*(-kt*(-2*des_y + 2*y)/m + 4*vy)/m) + (q*cos(ph) - r*sin(ph))*(zeta2*((-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + (-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m) + (q*cos(ph) - r*sin(ph))*(-zeta1*(-2*des_x + 2*x)*sin(th)*cos(ph)*cos(ps)/m - zeta1*(-2*des_y + 2*y)*sin(ps)*sin(th)*cos(ph)/m) + ((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*(q*sin(ph)*sin(th)/cos(th)**2 + r*sin(th)*cos(ph)/cos(th)**2) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*(q*sin(ph)*sin(th)**2/cos(th)**2 + q*sin(ph) + r*sin(th)**2*cos(ph)/cos(th)**2 + r*cos(ph)) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(-zeta1*(-2*des_x + 2*x)*sin(ps)*cos(ph)*cos(th)/m + zeta1*(-2*des_y + 2*y)*cos(ph)*cos(ps)*cos(th)/m) + (-zeta1*(-2*des_x + 2*x)*sin(ph)*cos(ps)*cos(th)/m - zeta1*(-2*des_y + 2*y)*sin(ph)*sin(ps)*cos(th)/m)*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th)) + 2*vx*zeta1*cos(ph)*cos(ps)*cos(th)/m + 2*vy*zeta1*sin(ps)*cos(ph)*cos(th)/m + zeta1*(-kt*(-2*des_x + 2*x)/m + 4*vx)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-kt*(-2*des_y + 2*y)/m + 4*vy)*sin(ps)*cos(ph)*cos(th)/m) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta2*((-2*des_x + 2*x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m) + (q*cos(ph) - r*sin(ph))*(-zeta1*(-2*des_x + 2*x)*sin(ps)*cos(ph)*cos(th)/m + zeta1*(-2*des_y + 2*y)*cos(ph)*cos(ps)*cos(th)/m) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(ps) - zeta1*sin(th)*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th)) + ((-2*des_x + 2*x)*(zeta1*sin(ph)*sin(ps)*sin(th) + zeta1*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m)*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th)) + 2*vx*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + 2*vy*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m - zeta1*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))*(-kt*(-2*des_y + 2*y)/m + 4*vy)/m + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*(-kt*(-2*des_x + 2*x)/m + 4*vx)/m) + (p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))*(zeta2*((-2*des_x + 2*x)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-sin(ph)*sin(ps)*sin(th) - cos(ph)*cos(ps))/m) + (-q*sin(ph) - r*cos(ph))*(zeta1*(-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m) + (q*cos(ph) - r*sin(ph))*(-zeta1*(-2*des_x + 2*x)*sin(ph)*cos(ps)*cos(th)/m - zeta1*(-2*des_y + 2*y)*sin(ph)*sin(ps)*cos(th)/m) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(ps) - zeta1*sin(th)*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m)*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th)) + ((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*(q*cos(ph)/cos(th) - r*sin(ph)/cos(th)) + ((-2*des_x + 2*x)*(zeta1*sin(ph)*sin(ps)*sin(th) + zeta1*cos(ph)*cos(ps))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th)) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*(q*sin(th)*cos(ph)/cos(th) - r*sin(ph)*sin(th)/cos(th)) + 2*vx*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + 2*vy*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m + zeta1*(-kt*(-2*des_x + 2*x)/m + 4*vx)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))/m - zeta1*(-kt*(-2*des_y + 2*y)/m + 4*vy)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps))/m) + (dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)*(4*dist_x/m - 6*kt*vx/m - kt*(-kt*(-2*des_x + 2*x)/m + 4*vx)/m + 4*zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m + 2*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m) + (dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)*(4*dist_y/m - 6*kt*vy/m - kt*(-kt*(-2*des_y + 2*y)/m + 4*vy)/m - 4*zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m + 2*(dist_y - kt*vy - zeta1*sin(ph)*cos(ps) + zeta1*sin(ps)*sin(th)*cos(ph))/m) + (Ix*p*q - Iy*p*q - kr*r)*(((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*cos(ph)/cos(th) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*sin(th)*cos(ph)/cos(th) - (zeta1*(-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m)*sin(ph))/Iz + (-Ix*p*r + Iz*p*r - kr*q)*(((-2*des_x + 2*x)*(zeta1*sin(ph)*cos(ps) - zeta1*sin(ps)*sin(th)*cos(ph))/m + (-2*des_y + 2*y)*(zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m)*sin(ph)/cos(th) + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*sin(ph)*sin(th)/cos(th) + (zeta1*(-2*des_x + 2*x)*cos(ph)*cos(ps)*cos(th)/m + zeta1*(-2*des_y + 2*y)*sin(ps)*cos(ph)*cos(th)/m)*cos(ph))/Iy + ((-2*des_x + 2*x)*(-zeta1*sin(ph)*sin(th)*cos(ps) + zeta1*sin(ps)*cos(ph))/m + (-2*des_y + 2*y)*(-zeta1*sin(ph)*sin(ps)*sin(th) - zeta1*cos(ph)*cos(ps))/m)*(Iy*q*r - Iz*q*r - kr*p)/Ix
    
    xi2_1 = z - des_height
    xi2_2 = vz
    xi2_3 = (dist_z - g*m - kt*vz + zeta1*cos(ph)*cos(th))/m
    xi2_4 = (-dist_z*kt + g*kt*m + kt**2*vz - kt*zeta1*cos(ph)*cos(th) - m*p*zeta1*sin(ph)*cos(th) - m*q*zeta1*sin(th) + m*zeta2*cos(ph)*cos(th))/m**2
    
    Lg1Lf3S2 = cos(ph)*cos(th)/m
    Lg2Lf3S2 = -(zeta1*cos(th)*sin(ph))/(Ix*m)
    Lg3Lf3S2 = -(zeta1*sin(th))/(Iy*m)
    Lg4Lf3S2 = 0
    Lf4S2 = kt**2*(dist_z/m - g - kt*vz/m + zeta1*cos(ph)*cos(th)/m)/m**2 + zeta2*(-kt*cos(ph)*cos(th) - m*p*sin(ph)*cos(th) - m*q*sin(th))/m**2 + (q*cos(ph) - r*sin(ph))*(kt*zeta1*sin(th)*cos(ph) + m*p*zeta1*sin(ph)*sin(th) - m*q*zeta1*cos(th) - m*zeta2*sin(th)*cos(ph))/m**2 + (p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))*(kt*zeta1*sin(ph)*cos(th) - m*p*zeta1*cos(ph)*cos(th) - m*zeta2*sin(ph)*cos(th))/m**2 - zeta1*(-Ix*p*r + Iz*p*r - kr*q)*sin(th)/(Iy*m) - zeta1*(Iy*q*r - Iz*q*r - kr*p)*sin(ph)*cos(th)/(Ix*m)
    
    eta1_1 = np.arctan2(y-des_y,x-des_x)
    eta1_2 = (vx*(des_y - y) - vy*(des_x - x))/((des_x - x)**2 + (des_y - y)**2)
    eta1_3 = vx*(vy/((des_x - x)**2 + (des_y - y)**2) + (2*des_x - 2*x)*(vx*(des_y - y) - vy*(des_x - x))/((des_x - x)**2 + (des_y - y)**2)**2) + vy*(-vx/((des_x - x)**2 + (des_y - y)**2) + (2*des_y - 2*y)*(vx*(des_y - y) - vy*(des_x - x))/((des_x - x)**2 + (des_y - y)**2)**2) + (-des_x + x)*(dist_y - kt*vy - zeta1*sin(ph)*cos(ps) + zeta1*sin(ps)*sin(th)*cos(ph))/(m*((des_x - x)**2 + (des_y - y)**2)) + (des_y - y)*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/(m*((des_x - x)**2 + (des_y - y)**2))
    eta1_4 = vx*((4*des_x - 4*x)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(vx*(-2*vx*(des_y - y) + vy*(-2*des_x + 2*x) + 4*vy*(des_x - x)) - vy*(vx*(-2*des_x + 2*x) - 2*vy*(des_y - y))) + (-2*des_x + 2*x)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(vx*(-2*vx*(des_x - x) + vy*(-2*des_y + 2*y)) - vy*(vx*(-2*des_y + 2*y) + 4*vx*(des_y - y) - 2*vy*(des_x - x))) + (-2*des_y + 2*y)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta2*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*cos(ph) - r*sin(ph))*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2))
    
    Lg1Lf3P1 = ((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2))
    Lg2Lf3P1 = (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))/(Ix*m*((des_x - x)**2 + (des_y - y)**2))
    Lg3Lf3P1 = ((zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))*sin(ph)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*sin(ph)*sin(th)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) + (-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))*cos(ph)/(m*((des_x - x)**2 + (des_y - y)**2)))/Iy
    Lg4Lf3P1 = 0
    Lf4P1 = vx*(vx*((4*des_x - 4*x)*(6*des_x - 6*x)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**4) + 2*(4*des_x - 4*x)*(m*(vx*(-2*vx*(des_y - y) + vy*(-2*des_x + 2*x) + 4*vy*(des_x - x)) - vy*(vx*(-2*des_x + 2*x) - 2*vy*(des_y - y))) + (-2*des_x + 2*x)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) - 4*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (-4*m*vx*vy + 2*(-2*des_x + 2*x)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + 2*(des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + 2*(des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_x - 4*x)*(m*(vx*(-2*vx*(des_x - x) + vy*(-2*des_y + 2*y)) - vy*(vx*(-2*des_y + 2*y) + 4*vx*(des_y - y) - 2*vy*(des_x - x))) + (-2*des_y + 2*y)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (6*des_x - 6*x)*(4*des_y - 4*y)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**4) + (4*des_y - 4*y)*(m*(vx*(-2*vx*(des_y - y) + vy*(-2*des_x + 2*x) + 4*vy*(des_x - x)) - vy*(vx*(-2*des_x + 2*x) - 2*vy*(des_y - y))) + (-2*des_x + 2*x)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(2*vx**2 - 2*vy**2) + (-2*des_x + 2*x)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))) + (-2*des_y + 2*y)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta1*(q*cos(ph) - r*sin(ph))*sin(ps)*cos(ph)*cos(th)/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + zeta2*(2*des_x - 2*x)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + zeta2*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))/(m*((des_x - x)**2 + (des_y - y)**2)) + (2*des_x - 2*x)*(q*cos(ph) - r*sin(ph))*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (2*des_x - 2*x)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (2*des_x - 2*x)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (4*des_x - 4*x)*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (4*des_x - 4*x)*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (-kt*(-2*des_x + 2*x)*(des_y - y) + m*(-4*vx*(des_y - y) + 4*vy*(des_x - x)))*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)*(kt*(-2*des_x + 2*x)*(des_x - x) - kt*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*(-2*des_x + 2*x) + vx*(2*des_x - 2*x) + 4*vy*(des_y - y)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*(vx*((4*des_x - 4*x)*(6*des_y - 6*y)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**4) + (4*des_x - 4*x)*(m*(vx*(-2*vx*(des_x - x) + vy*(-2*des_y + 2*y)) - vy*(vx*(-2*des_y + 2*y) + 4*vx*(des_y - y) - 2*vy*(des_x - x))) + (-2*des_y + 2*y)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (4*des_y - 4*y)*(m*(vx*(-2*vx*(des_y - y) + vy*(-2*des_x + 2*x) + 4*vy*(des_x - x)) - vy*(vx*(-2*des_x + 2*x) - 2*vy*(des_y - y))) + (-2*des_x + 2*x)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(2*vx**2 - 2*vy**2) + (-2*des_x + 2*x)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))) + (-2*des_y + 2*y)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(6*des_y - 6*y)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**4) + 2*(4*des_y - 4*y)*(m*(vx*(-2*vx*(des_x - x) + vy*(-2*des_y + 2*y)) - vy*(vx*(-2*des_y + 2*y) + 4*vx*(des_y - y) - 2*vy*(des_x - x))) + (-2*des_y + 2*y)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) - 4*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (4*m*vx*vy + 2*(des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + 2*(-2*des_y + 2*y)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))) + 2*(des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) - zeta1*(q*cos(ph) - r*sin(ph))*cos(ph)*cos(ps)*cos(th)/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + zeta2*(2*des_y - 2*y)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + zeta2*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))/(m*((des_x - x)**2 + (des_y - y)**2)) + (2*des_y - 2*y)*(q*cos(ph) - r*sin(ph))*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (2*des_y - 2*y)*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (2*des_y - 2*y)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (4*des_y - 4*y)*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (4*des_y - 4*y)*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (kt*(des_x - x)*(-2*des_y + 2*y) + m*(-4*vx*(des_y - y) - 2*vy*(-des_x + x) + 2*vy*(des_x - x)))*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)*(-kt*(-2*des_y + 2*y)*(des_y - y) + kt*((des_x - x)**2 + (des_y - y)**2) + m*(-4*vx*(des_x - x) + vy*(-2*des_y + 2*y) - vy*(2*des_y - 2*y)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta2*(vx*((4*des_x - 4*x)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + ((-2*des_x + 2*x)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))) + (-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + ((-2*des_y + 2*y)*((des_x - x)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)) + (des_y - y)*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))) + (-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + (q*cos(ph) - r*sin(ph))*(-(des_x - x)*sin(ps)*cos(ph)*cos(th) + (des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + ((des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + (des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))*(q*sin(ph)/cos(th) + r*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + ((des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + (des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) - (sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2)) + (q*cos(ph) - r*sin(ph))*(vx*((4*des_x - 4*x)*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (zeta1*((des_x - x)**2 + (des_y - y)**2)*sin(ps)*cos(ph)*cos(th) + (-2*des_x + 2*x)*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-zeta1*((des_x - x)**2 + (des_y - y)**2)*cos(ph)*cos(ps)*cos(th) + (-2*des_y + 2*y)*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta2*(-(des_x - x)*sin(ps)*cos(ph)*cos(th) + (des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*cos(ph) - r*sin(ph))*(zeta1*(des_x - x)*sin(ps)*sin(th)*cos(ph) - zeta1*(des_y - y)*sin(th)*cos(ph)*cos(ps))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(-zeta1*(des_x - x)*cos(ph)*cos(ps)*cos(th) - zeta1*(des_y - y)*sin(ps)*cos(ph)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))*(q*sin(ph)*sin(th)/cos(th)**2 + r*sin(th)*cos(ph)/cos(th)**2)/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(q*sin(ph)*sin(th)**2/cos(th)**2 + q*sin(ph) + r*sin(th)**2*cos(ph)/cos(th)**2 + r*cos(ph))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*sin(ph)*sin(ps)*cos(th) - zeta1*(des_y - y)*sin(ph)*cos(ps)*cos(th))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + zeta1*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))*sin(ps)*cos(ph)*cos(th)/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) + zeta1*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))*cos(ph)*cos(ps)*cos(th)/(m**2*((des_x - x)**2 + (des_y - y)**2)**2)) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(vx*((4*des_x - 4*x)*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-zeta1*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))*((des_x - x)**2 + (des_y - y)**2) + (-2*des_x + 2*x)*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*((des_x - x)**2 + (des_y - y)**2) + (-2*des_y + 2*y)*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta2*((des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + (des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*cos(ph) - r*sin(ph))*(-zeta1*(des_x - x)*cos(ph)*cos(ps)*cos(th) - zeta1*(des_y - y)*sin(ps)*cos(ph)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*(des_x - x)*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph)) + zeta1*(des_y - y)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(sin(ph)*sin(th)*cos(ps) - sin(ps)*cos(ph)) + zeta1*(des_y - y)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps))*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2)) + (p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))*(vx*((4*des_x - 4*x)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-zeta1*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps))*((des_x - x)**2 + (des_y - y)**2) + (-2*des_x + 2*x)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)**2) + (-zeta1*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))*((des_x - x)**2 + (des_y - y)**2) + (-2*des_y + 2*y)*(zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + zeta2*((des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + (des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (-q*sin(ph) - r*cos(ph))*(-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*cos(ph) - r*sin(ph))*(zeta1*(des_x - x)*sin(ph)*sin(ps)*cos(th) - zeta1*(des_y - y)*sin(ph)*cos(ps)*cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*sin(ph)/cos(th) + r*cos(ph)/cos(th))*(zeta1*(des_x - x)*(sin(ph)*sin(th)*cos(ps) - sin(ps)*cos(ph)) + zeta1*(des_y - y)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (q*cos(ph)/cos(th) - r*sin(ph)/cos(th))*(zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(-sin(ph)*cos(ps) + sin(ps)*sin(th)*cos(ph)) + zeta1*(des_y - y)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(q*sin(th)*cos(ph)/cos(th) - r*sin(ph)*sin(th)/cos(th))/(m*((des_x - x)**2 + (des_y - y)**2)) - zeta1*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) + zeta1*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2)) + (dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)*(-kt*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) + vx*((4*des_x - 4*x)*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (-kt*(-2*des_x + 2*x)*(des_y - y) + m*(vx*(-2*des_y + 2*y) - 2*vx*(des_y - y) + 4*vy*(des_x - x)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(-kt*(des_y - y)*((des_x - x)**2 + (des_y - y)**2) + m*(2*vx*(des_x - x)*(des_y - y) - vy*((des_x - x)**2 - (des_y - y)**2) + vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (-kt*(-2*des_y + 2*y)*(des_y - y) + kt*((des_x - x)**2 + (des_y - y)**2) + m*(vx*(-2*des_x + 2*x) - 2*vx*(des_x - x) + vy*(-2*des_y + 2*y) - vy*(2*des_y - 2*y)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + 4*(des_x - x)*(des_y - y)*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/((des_x - x)**2 + (des_y - y)**2)**2 + (2*(-des_x + x)*(des_x - x) + 2*(des_y - y)**2)*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/((des_x - x)**2 + (des_y - y)**2)**2 + (4*des_x - 4*x)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(vx*(-2*vx*(des_y - y) + vy*(-2*des_x + 2*x) + 4*vy*(des_x - x)) - vy*(vx*(-2*des_x + 2*x) - 2*vy*(des_y - y))) + (-2*des_x + 2*x)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(dist_y - kt*vy - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + (dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)*(-kt*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))/(m**2*((des_x - x)**2 + (des_y - y)**2)**2) + vx*((4*des_x - 4*x)*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (kt*(-2*des_x + 2*x)*(des_x - x) - kt*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*(-2*des_x + 2*x) + vx*(2*des_x - 2*x) - vy*(-2*des_y + 2*y) + 2*vy*(des_y - y)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + vy*((4*des_y - 4*y)*(kt*(des_x - x)*((des_x - x)**2 + (des_y - y)**2) + m*(-vx*((des_x - x)**2 + (des_y - y)**2) + vx*(2*(-des_x + x)*(des_x - x) + (des_x - x)**2 + (des_y - y)**2) + 2*vy*(-des_x + x)*(des_y - y) + 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x))))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (kt*(des_x - x)*(-2*des_y + 2*y) + m*(-4*vx*(des_y - y) - vy*(-2*des_x + 2*x) + 2*vy*(des_x - x)))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + 4*(-des_x + x)*(des_y - y)*(dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m)/((des_x - x)**2 + (des_y - y)**2)**2 + (2*(-des_x + x)*(des_x - x) + 2*(des_y - y)**2)*(dist_x/m - kt*vx/m + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m)/((des_x - x)**2 + (des_y - y)**2)**2 + (4*des_y - 4*y)*(m*(vx*(vy*((des_x - x)**2 + (des_y - y)**2) + 2*(des_x - x)*(vx*(des_y - y) - vy*(des_x - x))) - vy*(vx*((des_x - x)**2 + (des_y - y)**2) - 2*(des_y - y)*(vx*(des_y - y) - vy*(des_x - x)))) + ((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))*((des_x - x)**2 + (des_y - y)**2))/(m*((des_x - x)**2 + (des_y - y)**2)**3) + (m*(vx*(-2*vx*(des_x - x) + vy*(-2*des_y + 2*y)) - vy*(vx*(-2*des_y + 2*y) + 4*vx*(des_y - y) - 2*vy*(des_x - x))) + (-2*des_y + 2*y)*((des_x - x)*(-dist_y + kt*vy + zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))) + (des_y - y)*(dist_x - kt*vx + zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps)))) + ((des_x - x)**2 + (des_y - y)**2)*(-dist_x + kt*vx - zeta1*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))))/(m*((des_x - x)**2 + (des_y - y)**2)**2)) + (Ix*p*q - Iy*p*q - kr*r)*((zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))*cos(ph)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*sin(th)*cos(ph)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) - (-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))*sin(ph)/(m*((des_x - x)**2 + (des_y - y)**2)))/Iz + (-Ix*p*r + Iz*p*r - kr*q)*((zeta1*(des_x - x)*(-sin(ph)*sin(ps) - sin(th)*cos(ph)*cos(ps)) + zeta1*(des_y - y)*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph)))*sin(ph)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*sin(ph)*sin(th)/(m*((des_x - x)**2 + (des_y - y)**2)*cos(th)) + (-zeta1*(des_x - x)*sin(ps)*cos(ph)*cos(th) + zeta1*(des_y - y)*cos(ph)*cos(ps)*cos(th))*cos(ph)/(m*((des_x - x)**2 + (des_y - y)**2)))/Iy + (zeta1*(des_x - x)*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps)) + zeta1*(des_y - y)*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph)))*(Iy*q*r - Iz*q*r - kr*p)/(Ix*m*((des_x - x)**2 + (des_y - y)**2))
    
    eta2_1 = ps - des_yaw;
    eta2_2 = (r*cos(ph) + q*sin(ph))/cos(th)
    
    Lg1LfP2 = 0
    Lg2LfP2 = 0
    Lg3LfP2 = sin(ph)/(Iy*cos(th))
    Lg4LfP2 = cos(ph)/(Iz*cos(th))
    Lf2P2 = (q*sin(ph) + r*cos(ph))*(q*cos(ph) - r*sin(ph))*sin(th)/cos(th)**2 + (q*cos(ph) - r*sin(ph))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/cos(th) + (Ix*p*q - Iy*p*q - kr*r)*cos(ph)/(Iz*cos(th)) + (-Ix*p*r + Iz*p*r - kr*q)*sin(ph)/(Iy*cos(th))
    
    # Control inputs
    D = np.array([
        [Lg1Lf3S1, Lg2Lf3S1, Lg3Lf3S1, Lg4Lf3S1],
        [Lg1Lf3S2, Lg2Lf3S2, Lg3Lf3S2, Lg4Lf3S2],
        [Lg1Lf3P1, Lg2Lf3P1, Lg3Lf3P1, Lg4Lf3P1],
        [Lg1LfP2,  Lg2LfP2,  Lg3LfP2,  Lg4LfP2]
    ])


    v1_tran = -Lf4S1 - k1*xi1_1 - k2*xi1_2 - k3*xi1_3 - k4*xi1_4
    v2_tran = -Lf4S2 - k5*xi2_1 - k6*xi2_2 - k7*xi2_3 - k8*xi2_4

    v1_tang = -Lf4P1 - k10*(eta1_2 - des_speed) - k11*eta1_3 - k12*eta1_4 
    # v1_tang = -Lf4P1 - k9*(eta1_1 + np.pi/80) - k10*(eta1_2) - k11*eta1_3 - k12*eta1_4

    v2_tang = -Lf2P2 - k13*(eta2_1) - k14*eta2_2
    # v2_tang = -Lf2P2

    rhs = np.array([v1_tran, v2_tran, v1_tang, v2_tang])
    
    # Singularity Avoidance ECBF Terms
    h_th = th_max**2 - th**2
    Lfh_th = -2*th*(q*cos(ph) - r*sin(ph))
    Lf2h_th = 2*th*(q*sin(ph) + r*cos(ph))*(p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th)) - 2*(q*cos(ph) - r*sin(ph))**2 + 2*th*(Ix*p*q - Iy*p*q - kr*r)*sin(ph)/Iz - 2*th*(-Ix*p*r + Iz*p*r - kr*q)*cos(ph)/Iy
    Lg3Lfh_th = -2*th*cos(ph)/Iy
    Lg4Lfh_th = 2*th*sin(ph)/Iz
    
    h_ph = ph_max**2 - ph**2
    Lfh_ph = -2*ph*(p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th))
    Lf2h_ph = -2*ph*(q*sin(ph) + r*cos(ph))*(q*cos(ph) - r*sin(ph))/cos(th)**2 + (p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th))*(-2*p - 2*ph*(q*cos(ph) - r*sin(ph))*tan(th) - 2*q*sin(ph)*tan(th) - 2*r*cos(ph)*tan(th)) - 2*ph*(Ix*p*q - Iy*p*q - kr*r)*cos(ph)*tan(th)/Iz - 2*ph*(-Ix*p*r + Iz*p*r - kr*q)*sin(ph)*tan(th)/Iy - 2*ph*(Iy*q*r - Iz*q*r - kr*p)/Ix
    Lg2Lfh_ph = -2*ph/Ix
    Lg3Lfh_ph = -2*ph*sin(ph)*tan(th)/Iy
    Lg4Lfh_ph = -2*ph*cos(ph)*tan(th)/Iz
    
    
    # -------------------------
    # Pairwise rd=4 collision ECBF terms
    # -------------------------

    ph_i, th_i, ps_i, p_i, q_i, r_i, x_i, y_i, z_i, vx_i, vy_i, vz_i = state
    zeta1_i = float(zeta1)
    zeta2_i = float(zeta2)

    pair_cbf_terms = {}

    for j_id, sj in world_snapshot.items():
        if j_id == agent_id:
            continue

        xj = np.array(sj["state"], dtype=float).reshape(12)
        ph_j, th_j, ps_j, p_j, q_j, r_j, x_j, y_j, z_j, vx_j, vy_j, vz_j = xj
        zeta1_j = float(sj["zeta1"])
        zeta2_j = float(sj["zeta2"])

        B = (x_i - x_j)**2 + (y_i - y_j)**2 + (z_i - z_j)**2 - Ds**2

        LfB = 2*vx_i*(x_i - x_j) - 2*vx_j*(x_i - x_j) + 2*vy_i*(y_i - y_j) - 2*vy_j*(y_i - y_j) + 2*vz_i*(z_i - z_j) - 2*vz_j*(z_i - z_j)
        Lf2B = 2*(m*(vx_i*(vx_i - vx_j) - vx_j*(vx_i - vx_j) + vy_i*(vy_i - vy_j) - vy_j*(vy_i - vy_j) + vz_i*(vz_i - vz_j) - vz_j*(vz_i - vz_j)) + (-x_i + x_j)*(kt*vx_i - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i))) + (x_i - x_j)*(kt*vx_j - zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + (-y_i + y_j)*(kt*vy_i + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i))) + (y_i - y_j)*(kt*vy_j + zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) + (-z_i + z_j)*(g*m + kt*vz_i - zeta1_i*cos(ph_i)*cos(th_i)) + (z_i - z_j)*(g*m + kt*vz_j - zeta1_j*cos(ph_j)*cos(th_j)))/m
        Lf3B = 2*(m*zeta1_i*((q_i*sin(ph_i) + r_i*cos(ph_i))*((x_i - x_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i))) - (p_i*cos(th_i) + q_i*sin(ph_i)*sin(th_i) + r_i*sin(th_i)*cos(ph_i))*((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i)))*cos(th_j) + m*zeta1_j*(-(q_j*sin(ph_j) + r_j*cos(ph_j))*((x_i - x_j)*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)) + (y_i - y_j)*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + (p_j*cos(th_j) + q_j*sin(ph_j)*sin(th_j) + r_j*sin(th_j)*cos(ph_j))*((x_i - x_j)*(sin(ph_j)*sin(th_j)*cos(ps_j) - sin(ps_j)*cos(ph_j)) + (y_i - y_j)*(sin(ph_j)*sin(ps_j)*sin(th_j) + cos(ph_j)*cos(ps_j)) + (z_i - z_j)*sin(ph_j)*cos(th_j)))*cos(th_i) + m*(-vx_i*(kt*vx_i - kt*vx_j - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + vx_j*(kt*vx_i - kt*vx_j - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) - vy_i*(kt*vy_i - kt*vy_j + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) - zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) + vy_j*(kt*vy_i - kt*vy_j + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) - zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) - vz_i*(kt*vz_i - kt*vz_j - zeta1_i*cos(ph_i)*cos(th_i) + zeta1_j*cos(ph_j)*cos(th_j)) + vz_j*(kt*vz_i - kt*vz_j - zeta1_i*cos(ph_i)*cos(th_i) + zeta1_j*cos(ph_j)*cos(th_j)) + zeta1_i*(q_i*cos(ph_i) - r_i*sin(ph_i))*((x_i - x_j)*cos(ps_i)*cos(th_i) + (y_i - y_j)*sin(ps_i)*cos(th_i) + (-z_i + z_j)*sin(th_i))*cos(ph_i) - zeta1_j*(q_j*cos(ph_j) - r_j*sin(ph_j))*((x_i - x_j)*cos(ps_j)*cos(th_j) + (y_i - y_j)*sin(ps_j)*cos(th_j) + (-z_i + z_j)*sin(th_j))*cos(ph_j) + zeta2_i*((x_i - x_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + (-y_i + y_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (z_i - z_j)*cos(ph_i)*cos(th_i)) - zeta2_j*((x_i - x_j)*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j)) + (-y_i + y_j)*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)) + (z_i - z_j)*cos(ph_j)*cos(th_j)))*cos(th_i)*cos(th_j) + ((kt*vx_i - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)))*(kt*(x_i - x_j) - 2*m*(vx_i - vx_j)) + (kt*vx_j - zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j)))*(-kt*(x_i - x_j) + 2*m*(vx_i - vx_j)) + (kt*vy_i + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)))*(kt*(y_i - y_j) - 2*m*(vy_i - vy_j)) + (kt*vy_j + zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)))*(-kt*(y_i - y_j) + 2*m*(vy_i - vy_j)) + (-kt*(z_i - z_j) + 2*m*(vz_i - vz_j))*(g*m + kt*vz_j - zeta1_j*cos(ph_j)*cos(th_j)) + (kt*(z_i - z_j) - 2*m*(vz_i - vz_j))*(g*m + kt*vz_i - zeta1_i*cos(ph_i)*cos(th_i)))*cos(th_i)*cos(th_j))/(m**2*cos(th_i)*cos(th_j))
        Lf4B = -2*Ix*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) - 2*Ix*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*z_i*sin(th_i)/(Iy*m) - 2*Ix*zeta1_i*p_i*r_i*z_j*sin(th_i)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*z_i*sin(th_j)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*z_j*sin(th_j)/(Iy*m) + 6*zeta1_i**2/m**2 - 12*zeta1_i*zeta1_j*sin(ph_i)*sin(ph_j)*cos(ps_i - ps_j)/m**2 - 12*zeta1_i*zeta1_j*sin(ph_i)*sin(th_j)*sin(ps_i - ps_j)*cos(ph_j)/m**2 + 12*zeta1_i*zeta1_j*sin(ph_j)*sin(th_i)*sin(ps_i - ps_j)*cos(ph_i)/m**2 - 12*zeta1_i*zeta1_j*sin(th_i)*sin(th_j)*cos(ph_i)*cos(ph_j)*cos(ps_i - ps_j)/m**2 - 12*zeta1_i*zeta1_j*cos(ph_i)*cos(ph_j)*cos(th_i)*cos(th_j)/m**2 + 2*zeta1_i*kt**2*x_i*sin(ph_i)*sin(ps_i)/m**3 + 2*zeta1_i*kt**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*x_j*sin(ph_i)*sin(ps_i)/m**3 - 2*zeta1_i*kt**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*y_i*sin(ph_i)*cos(ps_i)/m**3 + 2*zeta1_i*kt**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**3 + 2*zeta1_i*kt**2*y_j*sin(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**3 + 2*zeta1_i*kt**2*z_i*cos(ph_i)*cos(th_i)/m**3 - 2*zeta1_i*kt**2*z_j*cos(ph_i)*cos(th_i)/m**3 + 2*zeta1_i*kt*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m**2 - 2*zeta1_i*kt*p_i*x_i*sin(ps_i)*cos(ph_i)/m**2 - 2*zeta1_i*kt*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m**2 + 2*zeta1_i*kt*p_i*x_j*sin(ps_i)*cos(ph_i)/m**2 + 2*zeta1_i*kt*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m**2 + 2*zeta1_i*kt*p_i*y_i*cos(ph_i)*cos(ps_i)/m**2 - 2*zeta1_i*kt*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m**2 - 2*zeta1_i*kt*p_i*y_j*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta1_i*kt*p_i*z_i*sin(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*p_i*z_j*sin(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*q_i*x_i*cos(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*x_j*cos(ps_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*q_i*y_i*sin(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*y_j*sin(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*z_i*sin(th_i)/m**2 - 2*zeta1_i*kt*q_i*z_j*sin(th_i)/m**2 - 20*zeta1_i*kt*vx_i*sin(ph_i)*sin(ps_i)/m**2 - 20*zeta1_i*kt*vx_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vx_j*sin(ph_i)*sin(ps_i)/m**2 + 20*zeta1_i*kt*vx_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vy_i*sin(ph_i)*cos(ps_i)/m**2 - 20*zeta1_i*kt*vy_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 20*zeta1_i*kt*vy_j*sin(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vy_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 20*zeta1_i*kt*vz_i*cos(ph_i)*cos(th_i)/m**2 + 20*zeta1_i*kt*vz_j*cos(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*p_i**2*x_i*sin(ph_i)*sin(ps_i)/m - 2*zeta1_i*p_i**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*x_j*sin(ph_i)*sin(ps_i)/m + 2*zeta1_i*p_i**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*y_i*sin(ph_i)*cos(ps_i)/m - 2*zeta1_i*p_i**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*p_i**2*y_j*sin(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*p_i**2*z_i*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*p_i**2*z_j*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/m + 2*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*z_i*sin(th_i)/m + 2*zeta1_i*p_i*r_i*z_j*sin(th_i)/m - 8*zeta1_i*p_i*vx_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 8*zeta1_i*p_i*vx_i*sin(ps_i)*cos(ph_i)/m + 8*zeta1_i*p_i*vx_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 8*zeta1_i*p_i*vx_j*sin(ps_i)*cos(ph_i)/m - 8*zeta1_i*p_i*vy_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 8*zeta1_i*p_i*vy_i*cos(ph_i)*cos(ps_i)/m + 8*zeta1_i*p_i*vy_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 8*zeta1_i*p_i*vy_j*cos(ph_i)*cos(ps_i)/m - 8*zeta1_i*p_i*vz_i*sin(ph_i)*cos(th_i)/m + 8*zeta1_i*p_i*vz_j*sin(ph_i)*cos(th_i)/m - 2*zeta1_i*q_i**2*x_i*sin(ph_i)*sin(ps_i)/m - 2*zeta1_i*q_i**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*x_j*sin(ph_i)*sin(ps_i)/m + 2*zeta1_i*q_i**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*y_i*sin(ph_i)*cos(ps_i)/m - 2*zeta1_i*q_i**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*q_i**2*y_j*sin(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*q_i**2*z_i*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*q_i**2*z_j*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 2*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/m - 2*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 2*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/m + 2*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 2*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/m - 2*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 2*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/m - 2*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/m + 8*zeta1_i*q_i*vx_i*cos(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vx_j*cos(ps_i)*cos(th_i)/m + 8*zeta1_i*q_i*vy_i*sin(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vy_j*sin(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vz_i*sin(th_i)/m + 8*zeta1_i*q_i*vz_j*sin(th_i)/m + 6*zeta1_j**2/m**2 - 2*zeta1_j*kt**2*x_i*sin(ph_j)*sin(ps_j)/m**3 - 2*zeta1_j*kt**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*x_j*sin(ph_j)*sin(ps_j)/m**3 + 2*zeta1_j*kt**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*y_i*sin(ph_j)*cos(ps_j)/m**3 - 2*zeta1_j*kt**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**3 - 2*zeta1_j*kt**2*y_j*sin(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**3 - 2*zeta1_j*kt**2*z_i*cos(ph_j)*cos(th_j)/m**3 + 2*zeta1_j*kt**2*z_j*cos(ph_j)*cos(th_j)/m**3 - 2*zeta1_j*kt*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m**2 + 2*zeta1_j*kt*p_j*x_i*sin(ps_j)*cos(ph_j)/m**2 + 2*zeta1_j*kt*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m**2 - 2*zeta1_j*kt*p_j*x_j*sin(ps_j)*cos(ph_j)/m**2 - 2*zeta1_j*kt*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m**2 - 2*zeta1_j*kt*p_j*y_i*cos(ph_j)*cos(ps_j)/m**2 + 2*zeta1_j*kt*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m**2 + 2*zeta1_j*kt*p_j*y_j*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta1_j*kt*p_j*z_i*sin(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*p_j*z_j*sin(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*q_j*x_i*cos(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*x_j*cos(ps_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*q_j*y_i*sin(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*y_j*sin(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*z_i*sin(th_j)/m**2 + 2*zeta1_j*kt*q_j*z_j*sin(th_j)/m**2 + 20*zeta1_j*kt*vx_i*sin(ph_j)*sin(ps_j)/m**2 + 20*zeta1_j*kt*vx_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vx_j*sin(ph_j)*sin(ps_j)/m**2 - 20*zeta1_j*kt*vx_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vy_i*sin(ph_j)*cos(ps_j)/m**2 + 20*zeta1_j*kt*vy_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 20*zeta1_j*kt*vy_j*sin(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vy_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 20*zeta1_j*kt*vz_i*cos(ph_j)*cos(th_j)/m**2 - 20*zeta1_j*kt*vz_j*cos(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*p_j**2*x_i*sin(ph_j)*sin(ps_j)/m + 2*zeta1_j*p_j**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*x_j*sin(ph_j)*sin(ps_j)/m - 2*zeta1_j*p_j**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*y_i*sin(ph_j)*cos(ps_j)/m + 2*zeta1_j*p_j**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*p_j**2*y_j*sin(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*p_j**2*z_i*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*p_j**2*z_j*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/m - 2*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*z_i*sin(th_j)/m - 2*zeta1_j*p_j*r_j*z_j*sin(th_j)/m + 8*zeta1_j*p_j*vx_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 8*zeta1_j*p_j*vx_i*sin(ps_j)*cos(ph_j)/m - 8*zeta1_j*p_j*vx_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 8*zeta1_j*p_j*vx_j*sin(ps_j)*cos(ph_j)/m + 8*zeta1_j*p_j*vy_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 8*zeta1_j*p_j*vy_i*cos(ph_j)*cos(ps_j)/m - 8*zeta1_j*p_j*vy_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 8*zeta1_j*p_j*vy_j*cos(ph_j)*cos(ps_j)/m + 8*zeta1_j*p_j*vz_i*sin(ph_j)*cos(th_j)/m - 8*zeta1_j*p_j*vz_j*sin(ph_j)*cos(th_j)/m + 2*zeta1_j*q_j**2*x_i*sin(ph_j)*sin(ps_j)/m + 2*zeta1_j*q_j**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*x_j*sin(ph_j)*sin(ps_j)/m - 2*zeta1_j*q_j**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*y_i*sin(ph_j)*cos(ps_j)/m + 2*zeta1_j*q_j**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*q_j**2*y_j*sin(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*q_j**2*z_i*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*q_j**2*z_j*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 2*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/m + 2*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 2*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/m - 2*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 2*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/m + 2*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 2*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/m + 2*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/m - 8*zeta1_j*q_j*vx_i*cos(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vx_j*cos(ps_j)*cos(th_j)/m - 8*zeta1_j*q_j*vy_i*sin(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vy_j*sin(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vz_i*sin(th_j)/m - 8*zeta1_j*q_j*vz_j*sin(th_j)/m - 2*zeta2_i*kt*x_i*sin(ph_i)*sin(ps_i)/m**2 - 2*zeta2_i*kt*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*x_j*sin(ph_i)*sin(ps_i)/m**2 + 2*zeta2_i*kt*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*y_i*sin(ph_i)*cos(ps_i)/m**2 - 2*zeta2_i*kt*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 2*zeta2_i*kt*y_j*sin(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 2*zeta2_i*kt*z_i*cos(ph_i)*cos(th_i)/m**2 + 2*zeta2_i*kt*z_j*cos(ph_i)*cos(th_i)/m**2 - 4*zeta2_i*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 4*zeta2_i*p_i*x_i*sin(ps_i)*cos(ph_i)/m + 4*zeta2_i*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 4*zeta2_i*p_i*x_j*sin(ps_i)*cos(ph_i)/m - 4*zeta2_i*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 4*zeta2_i*p_i*y_i*cos(ph_i)*cos(ps_i)/m + 4*zeta2_i*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 4*zeta2_i*p_i*y_j*cos(ph_i)*cos(ps_i)/m - 4*zeta2_i*p_i*z_i*sin(ph_i)*cos(th_i)/m + 4*zeta2_i*p_i*z_j*sin(ph_i)*cos(th_i)/m + 4*zeta2_i*q_i*x_i*cos(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*x_j*cos(ps_i)*cos(th_i)/m + 4*zeta2_i*q_i*y_i*sin(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*y_j*sin(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*z_i*sin(th_i)/m + 4*zeta2_i*q_i*z_j*sin(th_i)/m + 8*zeta2_i*vx_i*sin(ph_i)*sin(ps_i)/m + 8*zeta2_i*vx_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m - 8*zeta2_i*vx_j*sin(ph_i)*sin(ps_i)/m - 8*zeta2_i*vx_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m - 8*zeta2_i*vy_i*sin(ph_i)*cos(ps_i)/m + 8*zeta2_i*vy_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m + 8*zeta2_i*vy_j*sin(ph_i)*cos(ps_i)/m - 8*zeta2_i*vy_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m + 8*zeta2_i*vz_i*cos(ph_i)*cos(th_i)/m - 8*zeta2_i*vz_j*cos(ph_i)*cos(th_i)/m + 2*zeta2_j*kt*x_i*sin(ph_j)*sin(ps_j)/m**2 + 2*zeta2_j*kt*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*x_j*sin(ph_j)*sin(ps_j)/m**2 - 2*zeta2_j*kt*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*y_i*sin(ph_j)*cos(ps_j)/m**2 + 2*zeta2_j*kt*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 2*zeta2_j*kt*y_j*sin(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 2*zeta2_j*kt*z_i*cos(ph_j)*cos(th_j)/m**2 - 2*zeta2_j*kt*z_j*cos(ph_j)*cos(th_j)/m**2 + 4*zeta2_j*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 4*zeta2_j*p_j*x_i*sin(ps_j)*cos(ph_j)/m - 4*zeta2_j*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 4*zeta2_j*p_j*x_j*sin(ps_j)*cos(ph_j)/m + 4*zeta2_j*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 4*zeta2_j*p_j*y_i*cos(ph_j)*cos(ps_j)/m - 4*zeta2_j*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 4*zeta2_j*p_j*y_j*cos(ph_j)*cos(ps_j)/m + 4*zeta2_j*p_j*z_i*sin(ph_j)*cos(th_j)/m - 4*zeta2_j*p_j*z_j*sin(ph_j)*cos(th_j)/m - 4*zeta2_j*q_j*x_i*cos(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*x_j*cos(ps_j)*cos(th_j)/m - 4*zeta2_j*q_j*y_i*sin(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*y_j*sin(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*z_i*sin(th_j)/m - 4*zeta2_j*q_j*z_j*sin(th_j)/m - 8*zeta2_j*vx_i*sin(ph_j)*sin(ps_j)/m - 8*zeta2_j*vx_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m + 8*zeta2_j*vx_j*sin(ph_j)*sin(ps_j)/m + 8*zeta2_j*vx_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m + 8*zeta2_j*vy_i*sin(ph_j)*cos(ps_j)/m - 8*zeta2_j*vy_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m - 8*zeta2_j*vy_j*sin(ph_j)*cos(ps_j)/m + 8*zeta2_j*vy_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m - 8*zeta2_j*vz_i*cos(ph_j)*cos(th_j)/m + 8*zeta2_j*vz_j*cos(ph_j)*cos(th_j)/m - 2*kt**3*vx_i*x_i/m**3 + 2*kt**3*vx_i*x_j/m**3 + 2*kt**3*vx_j*x_i/m**3 - 2*kt**3*vx_j*x_j/m**3 - 2*kt**3*vy_i*y_i/m**3 + 2*kt**3*vy_i*y_j/m**3 + 2*kt**3*vy_j*y_i/m**3 - 2*kt**3*vy_j*y_j/m**3 - 2*kt**3*vz_i*z_i/m**3 + 2*kt**3*vz_i*z_j/m**3 + 2*kt**3*vz_j*z_i/m**3 - 2*kt**3*vz_j*z_j/m**3 + 14*kt**2*vx_i**2/m**2 - 28*kt**2*vx_i*vx_j/m**2 + 14*kt**2*vx_j**2/m**2 + 14*kt**2*vy_i**2/m**2 - 28*kt**2*vy_i*vy_j/m**2 + 14*kt**2*vy_j**2/m**2 + 14*kt**2*vz_i**2/m**2 - 28*kt**2*vz_i*vz_j/m**2 + 14*kt**2*vz_j**2/m**2 + 2*Iz*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) + 2*Iz*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*z_i*sin(th_i)/(Iy*m) + 2*Iz*zeta1_i*p_i*r_i*z_j*sin(th_i)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*z_i*sin(th_j)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*z_j*sin(th_j)/(Iy*m) - 2*zeta1_i*kr*q_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) - 2*zeta1_i*kr*q_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*z_i*sin(th_i)/(Iy*m) - 2*zeta1_i*kr*q_i*z_j*sin(th_i)/(Iy*m) + 2*zeta1_j*kr*q_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) + 2*zeta1_j*kr*q_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*z_i*sin(th_j)/(Iy*m) + 2*zeta1_j*kr*q_j*z_j*sin(th_j)/(Iy*m) - 2*Iy*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m) + 2*zeta1_i*kr*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*zeta1_i*kr*p_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*zeta1_i*kr*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*zeta1_i*kr*p_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*zeta1_i*kr*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*zeta1_i*kr*p_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*zeta1_i*kr*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*zeta1_i*kr*p_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*zeta1_i*kr*p_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) - 2*zeta1_i*kr*p_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) - 2*zeta1_j*kr*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*zeta1_j*kr*p_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*zeta1_j*kr*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*zeta1_j*kr*p_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*zeta1_j*kr*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*zeta1_j*kr*p_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*zeta1_j*kr*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*zeta1_j*kr*p_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*zeta1_j*kr*p_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) + 2*zeta1_j*kr*p_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m)

        Lg1_iLf3B = 2*((x_i - x_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + (-y_i + y_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (z_i - z_j)*cos(ph_i)*cos(th_i))/m
        Lg2_iLf3B = -2*zeta1_i*((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i))/(Ix*m)
        Lg3_iLf3B = 2*(zeta1_i*m*(((x_i - x_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)))*sin(ph_i) - ((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i))*sin(ph_i)*sin(th_i))*cos(th_j) + zeta1_i*m*((x_i - x_j)*cos(ps_i)*cos(th_i) + (y_i - y_j)*sin(ps_i)*cos(th_i) + (-z_i + z_j)*sin(th_i))*cos(ph_i)**2*cos(th_i)*cos(th_j))/(Iy*m**2*cos(th_i)*cos(th_j))
        Lg4_iLf3B = 0

        # ------------------------------------------------
        pair_cbf_terms[j_id] = {

            # rd=4 ECBF core terms
            "B": float(B),
            "LfB": float(LfB),
            "Lf2B": float(Lf2B),
            "Lf3B": float(Lf3B),
            "Lf4B": float(Lf4B),

            # input directions (agent i only, decentralized form)
            "Lg1_iLf3B": float(Lg1_iLf3B),
            "Lg2_iLf3B": float(Lg2_iLf3B),
            "Lg3_iLf3B": float(Lg3_iLf3B),
            "Lg4_iLf3B": float(Lg4_iLf3B),

            # (optional but very useful for debugging)
            "j_name": sj.get("name", f"agent_{j_id}"),
        }
    
    
    return {
    "D": np.array(D, dtype=float).reshape(4, 4),
    "rhs": np.array(rhs, dtype=float).reshape(4,),
    "cbf_terms": {
        "th": {
            "Lf2h": float(Lf2h_th),
            "Lg3Lfh": float(Lg3Lfh_th),
            "Lg4Lfh": float(Lg4Lfh_th),
            "Lfh": float(Lfh_th),
            "h": float(h_th),
        },
        "ph": {
            "Lf2h": float(Lf2h_ph),
            "Lg2Lfh": float(Lg2Lfh_ph),
            "Lg3Lfh": float(Lg3Lfh_ph),
            "Lg4Lfh": float(Lg4Lfh_ph),
            "Lfh": float(Lfh_ph),
            "h": float(h_ph),
        },
    },
    "pair_cbf_terms": pair_cbf_terms,
    "tracking_vars": {
        "eta1_2": float(eta1_2),
        "eta2_1": float(eta2_1),
        "xi1_1": float(xi1_1),
        "xi2_1": float(xi2_1),
    },
}


def solve_qp_for_u(D, rhs, cbf_terms, pair_cbf_terms, cfg: QuadConfig):
    """
    Solves your per-quad QP:
      min u^T Q u + delta^T P delta
      s.t. D u == rhs + [0,0,delta0,delta1]
           roll/pitch CBF constraints
    Returns:
      u (4,), delta(2,)
    """
    lam_th = float(cfg.cbf_lambdas["lam_th"])
    lam_ph = float(cfg.cbf_lambdas["lam_ph"])
    k1_th, k0_th = 2*lam_th, lam_th**2
    k1_ph, k0_ph = 2*lam_ph, lam_ph**2

    Q_u = 1
    P_delta = 10

    u = cp.Variable(4)
    delta = cp.Variable(2)

    objective = cp.Minimize(Q_u*cp.sum_squares(u) + P_delta*cp.sum_squares(delta))

    delta4 = cp.hstack([0, 0, delta[0], delta[1]])

    # roll/pitch cbf constraints (same structure as your code)
    th = cbf_terms["th"]
    ph = cbf_terms["ph"]

    th_cbf = (th["Lf2h"] + th["Lg3Lfh"]*u[2] + th["Lg4Lfh"]*u[3] + k1_th*th["Lfh"] + k0_th*th["h"] >= 0)
    ph_cbf = (ph["Lf2h"] + ph["Lg2Lfh"]*u[1] + ph["Lg3Lfh"]*u[2] + ph["Lg4Lfh"]*u[3] + k1_ph*ph["Lfh"] + k0_ph*ph["h"] >= 0)

    constraints = [
        D @ u == rhs + delta4,
        th_cbf,
        ph_cbf
    ]
    for j_id, pair in pair_cbf_terms.items():
        constraint = (pair["Lg1_iLf3B"] * u[0] + pair["Lg2_iLf3B"] * u[1] + pair["Lg3_iLf3B"] * u[2] + pair["Lg4_iLf3B"] * u[3]
                      >= -0.5*(pair["Lf4B"] + alpha3 * pair["Lf3B"] + alpha2 * pair["Lf2B"] + alpha1 * pair["LfB"] + alpha0 * pair["B"]))
        constraints.append(constraint)
        

    prob = cp.Problem(objective, constraints)
    # You used MOSEK; keep it consistent:
    prob.solve(solver=cp.MOSEK, verbose=False)

    if u.value is None:
        raise RuntimeError("QP failed to solve (u is None). Check feasibility / solver availability.")

    return np.array(u.value).reshape(-1), np.array(delta.value).reshape(-1)


# ----------------------------
# Quadrotor Agent
# ----------------------------

class QuadrotorAgent:
    def __init__(self, agent_id: int, cfg: QuadConfig, x0_full, motor_state0, env: Environment):
        """
        x0_full: you currently use 14 states: [12 physical + zeta1 + zeta2]
        We'll store:
          - physical state (12)
          - zeta1, zeta2
          - motor_state (4)
        """
        self.id = int(agent_id)
        self.cfg = cfg
        self.env = env

        x0_full = np.array(x0_full, dtype=float).reshape(-1)
        assert x0_full.size == 14, "Expected x0_full of length 14: 12 states + zeta1 + zeta2"

        self.state = x0_full[:12].copy()
        self.zeta1 = float(x0_full[12])
        self.zeta2 = float(x0_full[13])

        self.motor_state = np.array(motor_state0, dtype=float).reshape(4)

        # logs
        self.log = {
            "t": [],
            "state": [],
            "zeta1": [],
            "zeta2": [],
            "u": [],
            "control_inputs": [],
            "motor_cmd": [],
            "eta1_2": [],
            "eta2_1": [],
            "xi1_1": [],
            "xi2_1": [],
        }

    def step(self, t, dt, world_snapshot):
        terms = compute_tracking_terms(
            agent_id=self.id,
            state=self.state,
            zeta1=self.zeta1,
            zeta2=self.zeta2,
            cfg=self.cfg,
            env=self.env,
            t=t,
            world_snapshot=world_snapshot,
        )
        D = terms["D"]
        rhs = terms["rhs"]
        cbf_terms = terms["cbf_terms"]
        pair_cbf_terms = terms["pair_cbf_terms"]  
        tracking_vars = terms["tracking_vars"]

        u, delta = solve_qp_for_u(D, rhs, cbf_terms, pair_cbf_terms, self.cfg)

        # u_new, up, uq, ur = u.value
        u_new = float(u[0])
        up = float(u[1])
        uq = float(u[2])
        ur = float(u[3])

        # 3) Update zetas
        self.zeta1 += dt * self.zeta2
        self.zeta2 += dt * u_new

        # 4) Directly propagate the 12 physical states with the QP inputs
        ControlInputs = np.array([up, uq, ur, self.zeta1], dtype=float)

        self.state = PelicanModelDirect(
            time=t, dt=dt,
            state=self.state,
            control_inputs=ControlInputs,
            system_parameters=self.env.system,
            sim_parameters=self.env.sim
        )

        # 5) log
        self.log["t"].append(t)
        self.log["state"].append(self.state.copy())
        self.log["zeta1"].append(self.zeta1)
        self.log["zeta2"].append(self.zeta2)
        self.log["u"].append(u.copy())
        self.log["control_inputs"].append(ControlInputs.copy())
        # self.log["motor_cmd"].append(MotorSpeed.copy())
        self.log["eta1_2"].append(tracking_vars["eta1_2"])
        self.log["eta2_1"].append(tracking_vars["eta2_1"])
        self.log["xi1_1"].append(tracking_vars["xi1_1"])
        self.log["xi2_1"].append(tracking_vars["xi2_1"])
        
        self.log.setdefault("pair_cbf_terms", []).append(pair_cbf_terms)

# ----------------------------
# Multi-agent simulation "server"
# ----------------------------

class MultiQuadSim:
    def __init__(self, agents):
        self.agents = list(agents)
        
    def snapshot(self):
        """
        Returns a dict: id -> dict(state=..., zeta1=..., zeta2=...)
        This is what we'll pass into controllers for pairwise computations.
        """
        snap = {}
        for a in self.agents:
            snap[a.id] = {
                "state": a.state.copy(),   # (12,)
                "zeta1": float(a.zeta1),
                "zeta2": float(a.zeta2),
                "name": a.cfg.name,
            }
        return snap


    def run(self, Tmax, dt):
        T = np.arange(0.0, Tmax + dt, dt)
        for t in T:
            snap = self.snapshot()  # same snapshot for all agents at time t
            for agent in self.agents:
                agent.step(t, dt, world_snapshot=snap)
        return T





# ----------------------------
# Example setup
# ----------------------------

def make_env():
    # system params (shared)
    L = 0.2656
    Ix = 0.01152
    Iy = 0.01152
    Iz = 0.0218
    g = 9.8
    m = 1.923
    d = 1.0

    system_params = {
        "m": m, "g": g, "L": L, "Ix": Ix, "Iy": Iy, "Iz": Iz,
        "d": d,
        "kt": 0.00,
        "kr": 0.00
    }

    sim_params = {
        "quantize2int": 0,
        "disturbanceX": 0,
        "disturbanceY": 0,
        "disturbanceZ": 0
    }

    # mapping matrix for motor mixer
    M_map = np.array([
        [0,  0,  -L,  L],
        [L, -L,   0,  0],
        [-d, -d,  d,  d],
        [1,  1,   1,  1]
    ], dtype=float)

    return Environment(system_params, sim_params, M_map)

    

 
# ── Global style ──────────────────────────────────────────────────────────────
 
# Paul Tol "Bright" — colorblind-safe, prints well in greyscale
_COLORS = [
    "#EE6677",   # rose-red
    "#4477AA",   # steel-blue
    "#228833",   # green
    "#CCBB44",   # gold
    "#66CCEE",   # sky-blue
    "#AA3377",   # purple
    "#BBBBBB",   # grey
]
 
def _apply_base_style():
    matplotlib.rcParams.update({
        # ── Font (matches reference script exactly) ──
        "pdf.fonttype":           42,        # TrueType fonts embedded in PDF
        "ps.fonttype":            42,        # TrueType fonts embedded in EPS
        "font.family":            "serif",
        "font.serif":             ["Times New Roman", "DejaVu Serif"],
        "mathtext.fontset":       "cm",      # Computer Modern math
        "mathtext.default":       "it",      # italic math default
 
        # ── Sizes ──
        "font.size":              13,
        "axes.titlesize":         13,
        "axes.labelsize":         15,
        "xtick.labelsize":        12,
        "ytick.labelsize":        12,
        "legend.fontsize":        12,
 
        # ── Lines ──
        "lines.linewidth":        2.2,
        "lines.markersize":       7,
 
        # ── Axes ──
        "axes.spines.top":        False,
        "axes.spines.right":      False,
        "axes.linewidth":         1.1,
        "axes.edgecolor":         "#333333",
        "axes.facecolor":         "white",
        "figure.facecolor":       "white",
 
        # ── Grid ──
        "axes.grid":              True,
        "grid.color":             "#DDDDDD",
        "grid.linewidth":         0.8,
        "grid.linestyle":         "-",
 
        # ── Ticks ──
        "xtick.direction":        "out",
        "ytick.direction":        "out",
        "xtick.major.width":      1.0,
        "ytick.major.width":      1.0,
        "xtick.minor.visible":    False,
        "ytick.minor.visible":    False,
 
        # ── Legend ──
        "legend.frameon":         True,
        "legend.framealpha":      0.92,
        "legend.edgecolor":       "#CCCCCC",
        "legend.handlelength":    2.4,
 
        # ── Save ──
        "savefig.dpi":            300,
        "savefig.bbox":           "tight",
        "savefig.pad_inches":     0.05,
    })
 
_apply_base_style()
 
 
def _color(i):
    return _COLORS[i % len(_COLORS)]
 
 
def _circle(cx, cy, r, n=300):
    t = np.linspace(0, 2 * np.pi, n)
    return r * np.cos(t) + cx, r * np.sin(t) + cy
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 1.  3-D trajectory
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_trajectories_3d(
    agents,
    save_path="traj_3d.pdf",
    width_in=7.0,
    aspect=0.78,
    elev=22,
    azim=-55,
):
    _apply_base_style()
    fig = plt.figure(figsize=(width_in, width_in * aspect))
    ax  = fig.add_subplot(111, projection="3d")
 
    pane_color = (0.97, 0.97, 0.97, 1.0)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = True
        pane.set_facecolor(pane_color)
        pane.set_edgecolor("#CCCCCC")
    ax.grid(True, linewidth=0.6, color="#CCCCCC")
    ax.view_init(elev=elev, azim=azim)
 
    legend_handles = []
 
    for k, agent in enumerate(agents):
        st = np.array(agent.log["state"])
        x, y, z = st[:, 6], st[:, 7], st[:, 8]
        c = _color(k)
 
        ax.plot3D(x, y, z, color=c, linewidth=2.2, zorder=5)
        ax.scatter([x[0]], [y[0]], [z[0]],
                   color=c, s=60, marker="o", zorder=10,
                   edgecolors="white", linewidths=0.8)
 
        cx_des, cy_des = _circle(agent.cfg.des_x, agent.cfg.des_y, agent.cfg.des_radius)
        ax.plot3D(cx_des, cy_des,
                  np.full_like(cx_des, agent.cfg.des_height),
                  color=c, linewidth=4,
                  linestyle=(0, (5, 2)), alpha=0.9, zorder=3)
 
        legend_handles.append(
            Line2D([0], [0], color=c, linewidth=2.2, label=agent.cfg.name)
        )
 
    ax.set_xlabel(r"$x$ [m]", labelpad=8)
    ax.set_ylabel(r"$y$ [m]", labelpad=8)
    ax.set_zlabel(r"$z$ [m]", labelpad=6)
    ax.tick_params(labelsize=10)
    ax.legend(handles=legend_handles, loc="upper left",
              fontsize=11, frameon=True, framealpha=0.92,
              edgecolor="#CCCCCC", handlelength=2.2)
 
    fig.tight_layout(pad=0.4)
    fmt = save_path.rsplit(".", 1)[-1]
    fig.savefig(save_path, format=fmt, bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 2.  XY (top-down) trajectory
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_xy(
    agents,
    save_path="traj_xy.pdf",
    width_in=4.8,
    height_in=4.5,
):
    _apply_base_style()
    fig, ax = plt.subplots(figsize=(width_in, height_in))
 
    for k, agent in enumerate(agents):
        st = np.array(agent.log["state"])
        x, y = st[:, 6], st[:, 7]
        c = _color(k)
 
        ax.plot(x, y, color=c, linewidth=2.2, label=agent.cfg.name, zorder=4)
        ax.plot(x[0], y[0], "o", color=c,
                markersize=8, markeredgecolor="white",
                markeredgewidth=0.9, zorder=6)
 
        cx_des, cy_des = _circle(agent.cfg.des_x, agent.cfg.des_y, agent.cfg.des_radius)
        ax.plot(cx_des, cy_des,
                color=c, linewidth=4,
                linestyle=(0, (5, 2)), alpha=0.9, zorder=2)
 
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$x$ [m]")
    ax.set_ylabel(r"$y$ [m]")
    ax.legend(loc="best", fontsize=11)
 
    fig.tight_layout(pad=0.4)
    fmt = save_path.rsplit(".", 1)[-1]
    fig.savefig(save_path, format=fmt, bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 3.  Pairwise barrier  h_ij = ||p_i - p_j||²  (plotted as squared distance)
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_pairwise_B(
    agents,
    D_s,
    save_path="pairwise_B.pdf",
    width_in=6.5,
    height_in=2.8,
):
    _apply_base_style()
 
    if len(agents) < 2:
        print("Need ≥ 2 agents.")
        return
 
    T     = np.array(agents[0].log["t"], dtype=float)
    names = [a.cfg.name for a in agents]
    positions = [np.array(a.log["state"])[:, 6:9] for a in agents]
 
    fig, ax = plt.subplots(figsize=(width_in, height_in))
 
    for idx, (i, j) in enumerate(itertools.combinations(range(len(agents)), 2)):
        dp    = positions[i] - positions[j]
        dist2 = np.sum(dp * dp, axis=1)
        ax.plot(T, dist2,
                color=_COLORS[idx % len(_COLORS)],
                linewidth=2.0,
                label=rf"$\bar{{h}}_{{{names[i][-1]}{names[j][-1]}}}$",
                zorder=4)
 
    # ax.axhline(D_s**2, color="#333333", linewidth=1.3,
    #            linestyle=(0, (3, 2)), zorder=5,
    #            label=rf"$D_s^2 = {D_s**2:.2f}$")
 
    ax.set_xlabel(r"$t$ [s]")
    ax.set_ylabel(r"$\bar{h}_{ij}(t)$")
    ax.set_xlim(T[0], T[-1])
    ax.set_ylim(bottom=0)
    ax.legend(loc="best", ncol=2, fontsize=11)
 
    fig.tight_layout(pad=0.4)
    fmt = save_path.rsplit(".", 1)[-1]
    fig.savefig(save_path, format=fmt, bbox_inches="tight")
    plt.show()
 

 
def plot_tracking_summary(
    agents,
    save_path="tracking_summary.pdf",
    width_in=7.0,
    height_in=5.6,
):
    _apply_base_style()
    T = np.array(agents[0].log["t"], dtype=float)
 
    fig, axes = plt.subplots(2, 2, figsize=(width_in, height_in), sharex=True)
 
    panels = [
        (axes[0, 0], "xi1_1",  r"$\xi_{1}^{i}(t)$"),
        (axes[0, 1], "xi2_1",  r"$\zeta_{1}^{i}(t)$"),
        (axes[1, 0], "eta1_2", r"$\eta_{2}^{i}(t)$"),
        (axes[1, 1], "eta2_1", r"$\mu_{1}^{i}(t)$"),
    ]
 
    legend_handles = [
        Line2D([0], [0], color=_color(k), linewidth=2.2, label=ag.cfg.name)
        for k, ag in enumerate(agents)
    ]
 
    for ax, key, ylabel in panels:
        for k, ag in enumerate(agents):
            ax.plot(T, np.array(ag.log[key]),
                    color=_color(k), linewidth=2.0)
            if key=="eta1_2":
                ax.axhline(ag.cfg.des_speed, color=_color(k), linewidth=1, linestyle="--")
        if key != "eta1_2":
            ax.axhline(0, color="#888888", linewidth=0.7, linestyle="--")
        
        ax.set_ylabel(ylabel)
        ax.set_xlim(T[0], T[-1])
        ax.yaxis.set_major_locator(ticker.MaxNLocator(4))
 
    for ax in axes[1]:
        ax.set_xlabel(r"$t$ [s]")
 
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(agents),
        fontsize=11,
        frameon=True,
        framealpha=0.92,
        edgecolor="#CCCCCC",
        bbox_to_anchor=(0.5, -0.02),
    )
 
    fig.tight_layout(pad=0.5, rect=[0, 0.06, 1, 1])
    fmt = save_path.rsplit(".", 1)[-1]
    fig.savefig(save_path, format=fmt, bbox_inches="tight")
    plt.show()    

def plot_xy_and_barrier(
    agents,
    D_s,
    save_path="xy_and_barrier.eps",
    width_in=7.0,
    height_in=3.0,
):
    _apply_base_style()
    fig, (ax_xy, ax_h) = plt.subplots(1, 2, figsize=(width_in, height_in))

    # ── left: XY ──────────────────────────────────────────────
    for k, agent in enumerate(agents):
        st = np.array(agent.log["state"])
        x, y = st[:, 6], st[:, 7]
        c = _color(k)
        ax_xy.plot(x, y, color=c, linewidth=2.0, label=agent.cfg.name, zorder=4)
        ax_xy.plot(x[0], y[0], "o", color=c,
                   markersize=7, markeredgecolor="white",
                   markeredgewidth=0.8, zorder=6)
        cx_des, cy_des = _circle(agent.cfg.des_x, agent.cfg.des_y, agent.cfg.des_radius)
        ax_xy.plot(cx_des, cy_des, color=c, linewidth=4,
                   linestyle=(0, (5, 2)), alpha=0.9, zorder=2)

    ax_xy.set_aspect("equal", adjustable="box")
    ax_xy.set_xlabel(r"$x$ [m]", fontsize=16)
    ax_xy.set_ylabel(r"$y$ [m]", fontsize=16)
    ax_xy.tick_params(labelsize=13)
    ax_xy.legend(fontsize=10)
    ax_xy.grid(True, color="#DDDDDD", linewidth=0.8)
    ax_xy.spines['top'].set_visible(False)
    ax_xy.spines['right'].set_visible(False)

    # ── right: barrier ────────────────────────────────────────
    T         = np.array(agents[0].log["t"], dtype=float)
    names     = [a.cfg.name for a in agents]
    positions = [np.array(a.log["state"])[:, 6:9] for a in agents]

    for idx, (i, j) in enumerate(itertools.combinations(range(len(agents)), 2)):
        dp    = positions[i] - positions[j]
        dist2 = np.sum(dp * dp, axis=1)
        ax_h.plot(T, dist2,
                  color=_COLORS[idx % len(_COLORS)],
                  linewidth=2.0,
                  label=rf"$\bar{{h}}_{{{names[i][-1]}{names[j][-1]}}}$",
                  zorder=4)

    ax_h.set_xlabel(r"$t$ [s]", fontsize=16)
    ax_h.set_ylabel(r"$\bar{h}_{ij}(t)$", fontsize=16)
    ax_h.set_xlim(T[0], T[-1])
    ax_h.set_ylim(bottom=0)
    ax_h.tick_params(labelsize=13)
    ax_h.legend(fontsize=10)
    ax_h.grid(True, color="#DDDDDD", linewidth=0.8)
    ax_h.spines['top'].set_visible(False)
    ax_h.spines['right'].set_visible(False)

    fig.tight_layout(pad=0.4)
    fig.savefig(save_path, format="eps", bbox_inches="tight")
    plt.show()
    
    
# def _disp_poly_to_data(ax, pts_disp):
#     pts_data = ax.transData.inverted().transform(np.asarray(pts_disp, dtype=float))
#     return pts_data[:, 0], pts_data[:, 1]


# def _set_line_from_disp(line, ax, pts_disp):
#     xdat, ydat = _disp_poly_to_data(ax, pts_disp)
#     line.set_data(xdat, ydat)


# def _circle_disp(center_disp, radius_px, n=40):
#     ang = np.linspace(0.0, 2.0 * np.pi, n, endpoint=True)
#     return np.column_stack([
#         center_disp[0] + radius_px * np.cos(ang),
#         center_disp[1] + radius_px * np.sin(ang),
#     ])


# def _quadrotor_xy_primitives_realistic(
#     ax,
#     x,
#     y,
#     yaw,
#     rotor_phase=0.0,
#     arm_len_px=22.0,
#     hub_r_px=7.0,
#     motor_r_px=4.2,
#     rotor_r_px=9.0,
# ):
#     """
#     Build a realistic top-view quadrotor entirely in DISPLAY coordinates first,
#     then map back to data coordinates when plotting.

#     This is what prevents distortion when x/y scales are very different.
#     """
#     center = ax.transData.transform((x, y))

#     # forward direction from yaw, projected into display coordinates
#     test_fwd = ax.transData.transform((x + np.cos(yaw), y + np.sin(yaw))) - center
#     nf = np.linalg.norm(test_fwd)
#     if nf < 1e-12:
#         fwd = np.array([1.0, 0.0])
#     else:
#         fwd = test_fwd / nf

#     # display-space orthogonal lateral direction
#     lat = np.array([-fwd[1], fwd[0]])

#     # X configuration
#     d1 = (fwd + lat) / np.sqrt(2.0)
#     d2 = (-fwd + lat) / np.sqrt(2.0)
#     d3 = (-fwd - lat) / np.sqrt(2.0)
#     d4 = (fwd - lat) / np.sqrt(2.0)
#     motor_dirs = [d1, d2, d3, d4]

#     motor_centers = [center + arm_len_px * d for d in motor_dirs]

#     # arms
#     arms = [
#         np.vstack([center + hub_r_px * d, mc])
#         for d, mc in zip(motor_dirs, motor_centers)
#     ]

#     # central hub
#     hub = _circle_disp(center, hub_r_px, n=34)

#     # motor pods
#     motors = [_circle_disp(mc, motor_r_px, n=28) for mc in motor_centers]

#     # rotor rings
#     rotors = [_circle_disp(mc, rotor_r_px, n=34) for mc in motor_centers]

#     # spinning blades: two crossing blade lines per rotor gives richer look
#     spin_sign = [+1.0, -1.0, +1.0, -1.0]
#     blades_main = []
#     blades_cross = []

#     for k, mc in enumerate(motor_centers):
#         ang = spin_sign[k] * rotor_phase + 0.35 * k
#         bdir1 = np.cos(ang) * fwd + np.sin(ang) * lat
#         bdir2 = np.cos(ang + np.pi/2.0) * fwd + np.sin(ang + np.pi/2.0) * lat

#         blade1 = np.vstack([
#             mc - 0.20 * rotor_r_px * bdir1,
#             mc + 0.95 * rotor_r_px * bdir1,
#         ])
#         blade2 = np.vstack([
#             mc - 0.20 * rotor_r_px * bdir2,
#             mc + 0.70 * rotor_r_px * bdir2,
#         ])

#         blades_main.append(blade1)
#         blades_cross.append(blade2)

#     return {
#         "hub": hub,
#         "arms": arms,
#         "motors": motors,
#         "rotors": rotors,
#         "blades_main": blades_main,
#         "blades_cross": blades_cross,
#     }


def _desired_circle_path(cfg, n=280):
    t = np.linspace(0.0, 2.0 * np.pi, n)
    x = cfg.des_x + cfg.des_radius * np.cos(t)
    y = cfg.des_y + cfg.des_radius * np.sin(t)
    return x, y




def animate_xy_quadrotors_gif_circle(
    agents,
    save_path="traj_xy_circle_anim.gif",
    fps=25,
    stride=5,
    trail_length=120,
    figsize=(7.2, 6.4),
    dpi=100,
    dark_theme=True,
    arm_len_px=22.0,
    hub_r_px=7.0,
    motor_r_px=4.2,
    rotor_r_px=9.0,
):
    """
    Aesthetic 2D GIF animation for circular-path simulation.

    Uses the same display-space quadrotor rendering so the airframe remains
    visually consistent and detailed.
    """
    if len(agents) == 0:
        print("No agents provided.")
        return

    all_states = [np.array(ag.log["state"], dtype=float)[::stride] for ag in agents]
    n_frames = min(st.shape[0] for st in all_states)
    all_states = [st[:n_frames] for st in all_states]
    n_agents = len(agents)

    cmap = plt.get_cmap("tab10")
    colors = [cmap(i) for i in range(n_agents)]

    T_full = np.array(agents[0].log["t"], dtype=float)
    dt_sim = float(T_full[1] - T_full[0]) if T_full.size >= 2 else 0.01

    # bounds from actual trajectories + desired circles
    all_x = []
    all_y = []

    for st in all_states:
        all_x.append(st[:, 6])
        all_y.append(st[:, 7])

    for ag in agents:
        xd, yd = _desired_circle_path(ag.cfg)
        all_x.append(xd)
        all_y.append(yd)

    all_x = np.concatenate(all_x)
    all_y = np.concatenate(all_y)

    xr = float(np.max(all_x) - np.min(all_x))
    yr = float(np.max(all_y) - np.min(all_y))
    pad = max(0.08 * max(xr, yr), 0.45)

    bg = "#16213e" if dark_theme else "white"
    grid_c = "#223355" if dark_theme else "#DDDDDD"
    tick_c = "#AFC0D4" if dark_theme else "#333333"
    label_c = "white" if dark_theme else "#111111"
    legend_face = "#16213e" if dark_theme else "white"
    legend_edge = "#445566" if dark_theme else "#CCCCCC"

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, facecolor=bg)
    ax.set_facecolor(bg)

    ax.set_xlim(np.min(all_x) - pad, np.max(all_x) + pad)
    ax.set_ylim(np.min(all_y) - pad, np.max(all_y) + pad)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(r"$x$ [m]", fontsize=12, color=label_c)
    ax.set_ylabel(r"$y$ [m]", fontsize=12, color=label_c)
    ax.tick_params(colors=tick_c, labelsize=9)
    ax.grid(True, linewidth=0.35, color=grid_c, linestyle="--")

    for spine in ax.spines.values():
        spine.set_color("#334466" if dark_theme else "#CCCCCC")

    # desired circles
    for ag, col in zip(agents, colors):
        xd, yd = _desired_circle_path(ag.cfg)
        ax.plot(
            xd, yd,
            linestyle="--",
            color=col,
            linewidth=1.0,
            alpha=0.36 if dark_theme else 0.55,
            zorder=1
        )

    fig.canvas.draw()

    trail_lines = []
    agent_artists = []
    legend_handles = []

    for k, ag in enumerate(agents):
        c = colors[k]
        dark = tuple(0.35 * np.array(c[:3]))

        trail, = ax.plot([], [], "-", color=c, linewidth=1.15, alpha=0.62, zorder=2)
        hub, = ax.plot([], [], color=c, linewidth=1.2, zorder=6)

        arms = [ax.plot([], [], color=c, linewidth=2.4, zorder=5, solid_capstyle="round")[0] for _ in range(4)]
        motors = [ax.plot([], [], color=dark, linewidth=0.85, zorder=7)[0] for _ in range(4)]
        rotors = [ax.plot([], [], color=c, linewidth=1.3, alpha=0.72, zorder=4)[0] for _ in range(4)]
        blades_main = [ax.plot([], [], color=dark, linewidth=1.7, zorder=8, solid_capstyle="round")[0] for _ in range(4)]
        blades_cross = [ax.plot([], [], color=dark, linewidth=1.15, alpha=0.80, zorder=8, solid_capstyle="round")[0] for _ in range(4)]

        trail_lines.append(trail)
        agent_artists.append({
            "trail": trail,
            "hub": hub,
            "arms": arms,
            "motors": motors,
            "rotors": rotors,
            "blades_main": blades_main,
            "blades_cross": blades_cross,
        })

        legend_handles.append(Line2D([0], [0], color=c, linewidth=3, label=ag.cfg.name))

    ax.legend(
        handles=legend_handles,
        fontsize=10,
        loc="upper right",
        facecolor=legend_face,
        edgecolor=legend_edge,
        labelcolor=label_c if dark_theme else "#111111",
        framealpha=0.92
    )

    time_text = ax.text(
        0.03, 0.95, "",
        transform=ax.transAxes,
        fontsize=12,
        color=label_c,
        va="top",
        fontfamily="monospace"
    )

    rotor_phases = np.zeros(n_agents, dtype=float)
    rad_per_frame = (150.0 / 60.0) * 2.0 * np.pi * (stride * dt_sim)

    def _init():
        out = [time_text]
        time_text.set_text("")

        for art in agent_artists:
            art["trail"].set_data([], [])
            art["hub"].set_data([], [])
            for ln in art["arms"] + art["motors"] + art["rotors"] + art["blades_main"] + art["blades_cross"]:
                ln.set_data([], [])
            out.extend(
                [art["trail"], art["hub"]]
                + art["arms"] + art["motors"] + art["rotors"]
                + art["blades_main"] + art["blades_cross"]
            )
        return out

    def _update(frame):
        nonlocal rotor_phases
        rotor_phases += rad_per_frame

        out = []
        for i, (ag, st, c) in enumerate(zip(agents, all_states, colors)):
            s = st[frame]
            x, y = s[6], s[7]
            yaw = s[2]

            if trail_length is None or trail_length <= 0:
                s0 = 0
            else:
                s0 = max(0, frame - int(trail_length) + 1)

            trail = st[s0:frame+1, 6:8]
            agent_artists[i]["trail"].set_data(trail[:, 0], trail[:, 1])

            prim = _quadrotor_xy_primitives_realistic(
                ax=ax,
                x=x,
                y=y,
                yaw=yaw,
                rotor_phase=rotor_phases[i],
                arm_len_px=arm_len_px,
                hub_r_px=hub_r_px,
                motor_r_px=motor_r_px,
                rotor_r_px=rotor_r_px,
            )

            _set_line_from_disp(agent_artists[i]["hub"], ax, prim["hub"])

            for ln, pts in zip(agent_artists[i]["arms"], prim["arms"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["motors"], prim["motors"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["rotors"], prim["rotors"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["blades_main"], prim["blades_main"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["blades_cross"], prim["blades_cross"]):
                _set_line_from_disp(ln, ax, pts)

            out.extend(
                [agent_artists[i]["trail"], agent_artists[i]["hub"]]
                + agent_artists[i]["arms"] + agent_artists[i]["motors"]
                + agent_artists[i]["rotors"] + agent_artists[i]["blades_main"]
                + agent_artists[i]["blades_cross"]
            )

        time_text.set_text(f"t = {frame * stride * dt_sim:6.2f} s")
        out.append(time_text)
        return out

    print(f"[animate_xy_quadrotors_gif_circle] {n_frames} frames | stride={stride} | fps={fps} -> {save_path}")

    anim = FuncAnimation(
        fig,
        _update,
        frames=n_frames,
        init_func=_init,
        interval=1000 / fps,
        blit=False
    )

    anim.save(
        save_path,
        writer=PillowWriter(fps=fps),
        progress_callback=lambda i, n: print(f"\r  rendering {i+1}/{n}", end="", flush=True)
    )

    plt.close(fig)
    print(f"\n[animate_xy_quadrotors_gif_circle] Done -> {save_path}")
    
# ─────────────────────────────────────────────────────────────────────────────
# Rotation / geometry helpers
# ─────────────────────────────────────────────────────────────────────────────

def _R_quad(ph, th, ps):
    """ZYX Euler rotation: body -> world."""
    cph, sph = np.cos(ph), np.sin(ph)
    cth, sth = np.cos(th), np.sin(th)
    cps, sps = np.cos(ps), np.sin(ps)

    return np.array([
        [cth * cps,  sph * sth * cps - cph * sps,  cph * sth * cps + sph * sps],
        [cth * sps,  sph * sth * sps + cph * cps,  cph * sth * sps - sph * cps],
        [-sth,       sph * cth,                    cph * cth]
    ], dtype=float)


def _body_to_world(pts_body, R, origin):
    pts_body = np.asarray(pts_body, dtype=float)
    origin = np.asarray(origin, dtype=float).reshape(1, 3)
    return (R @ pts_body.T).T + origin


def _circle_pts_body(cx, cy, cz, r, n=24):
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=True)
    return np.column_stack([
        cx + r * np.cos(t),
        cy + r * np.sin(t),
        np.full_like(t, cz)
    ])


def _cylinder_rings_body(cx, cy, cz_bot, r, h, n=24):
    bot = _circle_pts_body(cx, cy, cz_bot, r, n=n)
    top = _circle_pts_body(cx, cy, cz_bot + h, r, n=n)
    verticals = [np.array([bot[i], top[i]]) for i in range(0, n, 4)]
    return bot, top, verticals


def _plot3d_line(ax, pts_world, color, lw, alpha=1.0, ls="-", zorder=None):
    line, = ax.plot(
        pts_world[:, 0], pts_world[:, 1], pts_world[:, 2],
        color=color,
        linewidth=lw,
        alpha=alpha,
        linestyle=ls,
        solid_capstyle="round",
        zorder=zorder
    )
    return line


def _desired_circle_path_3d(cfg, n=220):
    lam = np.linspace(0.0, 2.0 * np.pi, n)
    x = cfg.des_radius * np.cos(lam) + cfg.des_x
    y = cfg.des_radius * np.sin(lam) + cfg.des_y
    z = np.full_like(x, cfg.des_height)
    return x, y, z


# ─────────────────────────────────────────────────────────────────────────────
# Realistic line-only 3D quadrotor
# ─────────────────────────────────────────────────────────────────────────────

def _draw_quad_3d_realistic(
    ax,
    x, y, z, ph, th, ps,
    color,
    rotor_phase,
    scale=1.0,
    arm_lw=2.4,
    hub_lw=1.15,
    mot_lw=0.75,
    disc_lw=1.35,
    blade_lw=1.9,
):
    """
    Pelican-like line-only quadrotor rendering.

    Pure ax.plot() lines only, so it avoids most matplotlib 3D depth-sort issues.
    """
    # Pelican-ish geometry
    ARM   = 0.133 * scale
    RDISC = 0.060 * scale
    RMOT  = 0.018 * scale
    HMOT  = 0.022 * scale
    HR    = 0.028 * scale
    HH    = 0.016 * scale

    SQ2 = np.sqrt(2.0) / 2.0
    motor_pos = ARM * np.array([
        [ SQ2,  SQ2, 0.0],
        [-SQ2,  SQ2, 0.0],
        [-SQ2, -SQ2, 0.0],
        [ SQ2, -SQ2, 0.0],
    ], dtype=float)

    rotor_spin = [+1, -1, +1, -1]

    origin = np.array([x, y, z], dtype=float)
    R = _R_quad(ph, th, ps)

    c = color[:3] if len(color) >= 3 else color
    dark = tuple(0.35 * np.array(c))

    arts = []

    def _add(pts_body, col, lw, alpha=1.0, ls="-", zorder=None):
        pts_world = _body_to_world(pts_body, R, origin)
        arts.append(_plot3d_line(ax, pts_world, col, lw, alpha=alpha, ls=ls, zorder=zorder))

    # Arms
    for mp in motor_pos:
        start = mp * (HR / ARM)
        _add(np.array([start, mp]), c, arm_lw, zorder=8)

    # Hub cylinder
    hub_bot, hub_top, hub_verticals = _cylinder_rings_body(0.0, 0.0, -HH/2, HR, HH, n=24)
    _add(hub_bot, c, hub_lw, zorder=9)
    _add(hub_top, c, hub_lw, zorder=9)
    for v in hub_verticals:
        _add(v, c, 0.7 * hub_lw, zorder=9)

    # Motors + rotors
    for k, mp in enumerate(motor_pos):
        mx, my, mz = mp

        mot_bot, mot_top, mot_verticals = _cylinder_rings_body(mx, my, mz - HMOT, RMOT, HMOT, n=14)
        _add(mot_bot, dark, mot_lw, zorder=10)
        _add(mot_top, dark, mot_lw, zorder=10)
        for v in mot_verticals:
            _add(v, dark, 0.6 * mot_lw, zorder=10)

        disc_z = mz + 0.002 * scale
        disc = _circle_pts_body(mx, my, disc_z, RDISC, n=24)
        _add(disc, c, disc_lw, alpha=0.72, zorder=7)

        phase = rotor_phase * rotor_spin[k]
        for angle in [phase, phase + np.pi]:
            blade = np.array([
                [mx - RDISC * 0.15 * np.cos(angle), my - RDISC * 0.15 * np.sin(angle), disc_z],
                [mx + RDISC * 1.00 * np.cos(angle), my + RDISC * 1.00 * np.sin(angle), disc_z],
            ])
            _add(blade, dark, blade_lw, zorder=11)

    return arts


# ─────────────────────────────────────────────────────────────────────────────
# Main 3D GIF animation
# ─────────────────────────────────────────────────────────────────────────────

def animate_trajectories_3d_gif_circle(
    agents,
    gif_path="quad_anim_circle.gif",
    fps=25,
    skip=6,
    trail_len=50,
    ax_padding=0.4,
    figsize=(10.5, 8.5),
    dpi=100,
    elev=28.0,
    azim=-55.0,
    rotor_rpm=150.0,
    quad_scale=1.0,
    dark_theme=True,
):
    """
    3D GIF animation for the circular-path simulation.

    Parameters
    ----------
    agents : list
        Agents after sim.run(...)
    gif_path : str
        Output gif path
    fps : int
        Frames per second for gif
    skip : int
        Use every `skip`-th logged sample as one rendered frame
    trail_len : int or None
        Number of latest rendered points to show in the trail.
        Example:
            if dt = 0.02 and skip = 5, one rendered frame = 0.1 s
            so trail_len = 120 means last 12 seconds of trajectory.
        If None or <= 0, show full history.
    """
    if len(agents) == 0:
        print("No agents provided.")
        return

    all_states = [np.array(ag.log["state"], dtype=float)[::skip] for ag in agents]
    n_frames = min(st.shape[0] for st in all_states)
    all_states = [st[:n_frames] for st in all_states]
    n_agents = len(agents)

    cmap = plt.get_cmap("tab10")
    colors = [cmap(i) for i in range(n_agents)]

    # infer simulation dt from log
    T_full = np.array(agents[0].log["t"], dtype=float)
    dt_sim = float(T_full[1] - T_full[0]) if T_full.size >= 2 else 0.01

    # axis limits from states + desired circles
    all_xyz = np.vstack([st[:, 6:9] for st in all_states])
    xmin, ymin, zmin = all_xyz.min(axis=0) - ax_padding
    xmax, ymax, zmax = all_xyz.max(axis=0) + ax_padding

    for ag in agents:
        xd, yd, zd = _desired_circle_path_3d(ag.cfg, n=90)
        xmin = min(xmin, xd.min() - ax_padding)
        xmax = max(xmax, xd.max() + ax_padding)
        ymin = min(ymin, yd.min() - ax_padding)
        ymax = max(ymax, yd.max() + ax_padding)
        zmin = min(zmin, zd.min() - ax_padding)
        zmax = max(zmax, zd.max() + ax_padding)

    bg = "#16213e" if dark_theme else "white"
    grid_c = "#223355" if dark_theme else "#DDDDDD"
    pane_edge = "#334466" if dark_theme else "#CCCCCC"
    tick_c = "#AFC0D4" if dark_theme else "#333333"
    label_c = "white" if dark_theme else "#111111"
    legend_face = "#16213e" if dark_theme else "white"
    legend_edge = "#445566" if dark_theme else "#CCCCCC"

    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor=bg)
    ax = fig.add_subplot(111, projection="3d", facecolor=bg)

    # panes / grid
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False if dark_theme else True
        if not dark_theme:
            pane.set_facecolor((0.97, 0.97, 0.97, 1.0))
        pane.set_edgecolor(pane_edge)

    ax.set_xlabel(r"$x$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.set_ylabel(r"$y$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.set_zlabel(r"$z$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.tick_params(colors=tick_c, labelsize=9)
    ax.grid(True, linewidth=0.35, color=grid_c, linestyle="--")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_zlim(zmin, zmax)
    ax.view_init(elev=elev, azim=azim)

    # balanced box aspect
    ax.set_box_aspect((xmax - xmin, ymax - ymin, max(zmax - zmin, 0.8)))

    # desired circles
    for ag, col in zip(agents, colors):
        xd, yd, zd = _desired_circle_path_3d(ag.cfg, n=220)
        ax.plot(
            xd, yd, zd,
            linestyle="--",
            color=col,
            linewidth=0.9,
            alpha=0.35 if dark_theme else 0.55,
            zorder=2
        )

    # trail artists
    trail_lines = []
    for col in colors:
        ln, = ax.plot([], [], [], "-", color=col, linewidth=1.15, alpha=0.62)
        trail_lines.append(ln)

    time_text = ax.text2D(
        0.03, 0.95, "",
        transform=ax.transAxes,
        fontsize=12,
        color=label_c,
        va="top",
        fontfamily="monospace"
    )

    handles = [
        Line2D([0], [0], color=colors[i], linewidth=3, label=agents[i].cfg.name)
        for i in range(n_agents)
    ]
    ax.legend(
        handles=handles,
        fontsize=10,
        loc="upper right",
        facecolor=legend_face,
        edgecolor=legend_edge,
        labelcolor=label_c if dark_theme else "#111111",
        framealpha=0.92
    )

    fig.tight_layout()

    quad_artists = []
    rotor_phases = np.zeros(n_agents, dtype=float)
    rad_per_frame = (rotor_rpm / 60.0) * 2.0 * np.pi * (skip * dt_sim)

    def _init():
        for ln in trail_lines:
            ln.set_data([], [])
            ln.set_3d_properties([])
        time_text.set_text("")
        return trail_lines + [time_text]

    def _update(frame):
        nonlocal quad_artists, rotor_phases

        # remove old quadrotor drawings
        for ln in quad_artists:
            try:
                ln.remove()
            except Exception:
                pass
        quad_artists = []

        rotor_phases += rad_per_frame

        for i, (ag, st, col) in enumerate(zip(agents, all_states, colors)):
            s = st[frame]
            ph, th, ps = s[0], s[1], s[2]
            x, y, z = s[6], s[7], s[8]

            # moving tail: only latest trail_len rendered points
            if trail_len is None or trail_len <= 0:
                s0 = 0
            else:
                s0 = max(0, frame - int(trail_len) + 1)

            trail = st[s0:frame+1, 6:9]
            trail_lines[i].set_data(trail[:, 0], trail[:, 1])
            trail_lines[i].set_3d_properties(trail[:, 2])

            arts = _draw_quad_3d_realistic(
                ax,
                x, y, z,
                ph, th, ps,
                color=col,
                rotor_phase=rotor_phases[i],
                scale=quad_scale
            )
            quad_artists.extend(arts)

        time_text.set_text(f"t = {frame * skip * dt_sim:6.2f} s")
        return trail_lines + [time_text] + quad_artists

    print(f"[animate_trajectories_3d_gif_circle] {n_frames} frames | skip={skip} | fps={fps} -> {gif_path}")

    anim = FuncAnimation(
        fig,
        _update,
        frames=n_frames,
        init_func=_init,
        interval=1000 / fps,
        blit=False
    )

    anim.save(
        gif_path,
        writer=PillowWriter(fps=fps),
        progress_callback=lambda i, n: print(f"\r  rendering {i+1}/{n}", end="", flush=True)
    )

    plt.close(fig)
    print(f"\n[animate_trajectories_3d_gif_circle] Done -> {gif_path}")

In [138]:
def make_agents(env):
    # your gains (shared baseline); can customize per quad
    n = 1
    gains = {
        "k1": 200, "k2": 400, "k3": 90, "k4": 30,
        "k5": 100, "k6": 200, "k7": 60, "k8": 20,
        "k9": 10/n,  "k10": 40/n, "k11": 30/n, "k12": 30/n,
        "k13": 10/n, "k14": 12/n
    }

    th_max = math.pi/2
    ph_max = math.pi/2

    # cbf lambdas
    cbf_lambdas = {"lam_th": 1.1, "lam_ph": 1.1}

    # qp weights
    qp_weights = {"Q_u": 1.0, "P_delta": 10.0}

    cfgA = QuadConfig(
        name="quad_A",
        des_radius=1.5, des_x=0.0, des_y=0.0, des_height=1.0, des_yaw=0.0, des_speed = 0.3,
        gains=gains,
        th_max=th_max, ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights
    )

    # Quad B (different circle + height)
    cfgB = QuadConfig(
        name="quad_B",
        des_radius=1.5, des_x=0.0, des_y=1.0, des_height=1.0, des_yaw=0.0, des_speed = -0.2,
        gains=gains,
        th_max=th_max, ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights
    )

    # Quad C (different circle + height)
    cfgC = QuadConfig(
        name="quad_C",
        des_radius=1.5, des_x=1.0, des_y=0.0, des_height=1.0, des_yaw=0.0, des_speed = 0.2,
        gains=gains,
        th_max=th_max, ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights
    )
    
    # Quad D (different circle + height)
    cfgD = QuadConfig(
        name="quad_D",
        des_radius=1.5, des_x=1.0, des_y=1.0, des_height=1.0, des_yaw=0.0, des_speed = -0.3,
        gains=gains,
        th_max=th_max, ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights
    )

    # Initial conditions (14-state): [12 + zeta1 + zeta2]
    x0A = np.array([0,0,0, 0,0,0,  3,2,0.2,  0,0,0,  9.8, 0], dtype=float)
    x0B = np.array([0,0,0, 0,0,0,  -2.0,-2.0,0.3,  0,0,0,  9.8, 0], dtype=float)
    x0C = np.array([0,0,0, 0,0,0,  -2.0,2.0,0.2,  0,0,0,  9.8, 0], dtype=float)
    x0D = np.array([0,0,0, 0,0,0,  3.0,-1.0,0.2,  0,0,0,  9.8, 0], dtype=float)

    MotorState0 = np.array([10.0,10.0,10.0,10.0], dtype=float)

    return [
        QuadrotorAgent(0, cfgA, x0A, MotorState0, env),
        QuadrotorAgent(1, cfgB, x0B, MotorState0, env),
        QuadrotorAgent(2, cfgC, x0C, MotorState0, env),
        QuadrotorAgent(3, cfgD, x0D, MotorState0, env),
    ]





In [141]:
def main():
    env = make_env()
    agents = make_agents(env)

    sim = MultiQuadSim(agents)
    Tmax = 65
    dt = 0.02

    sim.run(Tmax=Tmax, dt=dt)

    # plot_trajectories_3d(agents,        save_path="fig1_3d.eps")
    # plot_xy_and_barrier(agents, D_s=Ds, save_path="fig2_xy_barrier.eps")
    # plot_tracking_summary(agents,       save_path="fig3_summary.eps")

#     animate_xy_quadrotors_gif_circle(
#         agents,
#         save_path="circle_xy_anim.gif",
#         fps=25,
#         stride=5,
#         trail_length=None,
#         hub_r_px=7.0/3,
#         arm_len_px=22.0/3,
#         motor_r_px=4.2/3,
#         rotor_r_px=9.0/3,
#     )

#     animate_trajectories_3d_gif_circle(
#         agents,
#         save_path="circle_3d_anim.gif",
#         fps=20,
#         stride=5,
#         trail_length=None,
#         quad_scale=0.25,
#         elev=22,
#         azim=-55,
#     )
    
    # animate_trajectories_3d_gif_circle(
    #     agents,
    #     gif_path="circle_quad_anim.gif",
    #     fps=25,
    #     skip=6,
    #     trail_len=20,
    #     figsize=(11, 9),
    #     dpi=100,
    #     elev=24,
    #     azim=-52,
    #     rotor_rpm=150,
    #     quad_scale=0.9,
    #     dark_theme=False,
    # )
    animate_xy_quadrotors_gif_circle(
        agents,
        save_path="traj_xy_circle_anim.gif",
        fps=25,
        stride=5,
        trail_length=120,
        figsize=(7.4, 6.6),
        dpi=100,
        dark_theme=False,
        arm_len_px=22.0/2,
        hub_r_px=7.0/2,
        motor_r_px=4.2/2,
        rotor_r_px=9.0/2,
    )
    
    
if __name__ == "__main__":
    main()

[animate_xy_quadrotors_gif_circle] 651 frames | stride=5 | fps=25 -> traj_xy_circle_anim.gif
  rendering 651/651
[animate_xy_quadrotors_gif_circle] Done -> traj_xy_circle_anim.gif


In [145]:
import numpy as np
import math
import cvxpy as cp
import matplotlib.pyplot as plt
import itertools
import matplotlib
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.animation import FuncAnimation, PillowWriter

sin = math.sin
cos = math.cos
tan = math.tan
th_max = np.pi/2
ph_max = np.pi/2
Ds=1.4

# coefficients for pairwise ECBF constraint
lam_pair = 3.75  # tune
alpha3 = 4.0 * lam_pair
alpha2 = 6.0 * lam_pair**2
alpha1 = 4.0 * lam_pair**3
alpha0 = lam_pair**4
# ----------------------------
# Dynamics + motor mapping
# ----------------------------

def PelicanModelDirect(time, dt, state, control_inputs, system_parameters, sim_parameters):
    """
    One-step quad dynamics driven directly by control inputs:
    control_inputs = [up, uq, ur, u_thrust]
    state = [ph, th, ps, p, q, r, x, y, z, vx, vy, vz]
    """
    m = system_parameters["m"]
    g = system_parameters["g"]
    Ix = system_parameters["Ix"]
    Iy = system_parameters["Iy"]
    Iz = system_parameters["Iz"]
    kr = system_parameters.get("kr", 0.01)
    kt = system_parameters.get("kt", 0.01)

    dist_x = 0.1 * np.sin(time) if sim_parameters.get("disturbanceX", 0) else 0.0
    dist_y = 0.1 * np.sin(time) if sim_parameters.get("disturbanceY", 0) else 0.0
    dist_z = 0.1 * np.sin(time) if sim_parameters.get("disturbanceZ", 0) else 0.0

    up, uq, ur, u_thrust = control_inputs
    ph, th, ps, p, q, r, x, y, z, vx, vy, vz = state

    # attitude kinematics
    ph += dt * (p + (r * np.cos(ph) * np.sin(th)) / np.cos(th) + (q * np.sin(ph) * np.sin(th)) / np.cos(th))
    th += dt * (q * np.cos(ph) - r * np.sin(ph))
    ps += dt * ((r * np.cos(ph)) / np.cos(th) + (q * np.sin(ph)) / np.cos(th))

    # angular dynamics
    p += dt * (up - kr * p + (Iy - Iz) * q * r) / Ix
    q += dt * (uq - kr * q + (Iz - Ix) * p * r) / Iy
    r += dt * (ur - kr * r + (Ix - Iy) * p * q) / Iz

    # position
    x += dt * vx
    y += dt * vy
    z += dt * vz

    # translational dynamics
    vx += dt * (
        dist_x / m
        - (kt * vx) / m
        + (u_thrust * (np.sin(ph) * np.sin(ps) + np.cos(ph) * np.cos(ps) * np.sin(th))) / m
    )
    vy += -dt * (
        (kt * vy) / m
        - dist_y / m
        + (u_thrust * (np.cos(ps) * np.sin(ph) - np.cos(ph) * np.sin(ps) * np.sin(th))) / m
    )
    vz += -dt * (
        g
        - dist_z / m
        + (kt * vz) / m
        - (u_thrust * np.cos(ph) * np.cos(th)) / m
    )

    return np.array([ph, th, ps, p, q, r, x, y, z, vx, vy, vz], dtype=float)



# ----------------------------
# Per-quad configuration
# ----------------------------

class QuadConfig:
    def __init__(
        self,
        name,
        A,
        w,
        des_x,
        des_y,
        des_height,
        des_yaw,
        des_speed,
        gains,
        th_max,
        ph_max,
        cbf_lambdas,
        qp_weights,
        resp_w,
        plot_x_min=-4.0,
        plot_x_max=4.0,
    ):
        self.name = name

        # desired sine-path parameters: y = des_y + A*sin(w*(x - des_x))
        self.A = float(A)
        self.w = float(w)
        self.des_x = float(des_x)
        self.des_y = float(des_y)
        self.des_height = float(des_height)
        self.des_yaw = float(des_yaw)
        self.des_speed = float(des_speed)
        self.resp_w = float(resp_w)

        # plotting range for desired path
        self.plot_x_min = float(plot_x_min)
        self.plot_x_max = float(plot_x_max)

        self.gains = dict(gains)
        self.th_max = float(th_max)
        self.ph_max = float(ph_max)
        self.cbf_lambdas = dict(cbf_lambdas)
        self.qp_weights = dict(qp_weights)


# ----------------------------
# Environment (shared)
# ----------------------------

class Environment:
    def __init__(self, system_params, sim_params, M_map):
        self.system = dict(system_params)
        self.sim = dict(sim_params)
        self.M_map = np.array(M_map, dtype=float)

    def disturbances(self, t):
        # You can expand this to spatially varying or agent-dependent disturbances
        return np.array([0.0, 0.0, 0.0], dtype=float)


# ----------------------------
# Controller wrapper
# ----------------------------

def compute_tracking_terms(agent_id, state, zeta1, zeta2, cfg: QuadConfig, env: Environment, t, world_snapshot):
    """
    YOU IMPLEMENT THIS ONCE by moving your long Lie-derivative block here.

    Must return a dictionary with:
      - D: (4,4) numpy array
      - rhs: (4,) numpy array
      - cbf_terms: dict with the needed scalars to build your constraints:
            th: Lf2h_th, Lg3Lfh_th, Lg4Lfh_th, Lfh_th, h_th
            ph: Lf2h_ph, Lg2Lfh_ph, Lg3Lfh_ph, Lg4Lfh_ph, Lfh_ph, h_ph

    Also anything else you want to log.
    """
    # ---- unpack ----
    ph, th, ps, p, q, r, x, y, z, vx, vy, vz = state
    m = env.system["m"]
    kt = env.system.get("kt", 0.01)
    kr = env.system.get("kr", 0.01)
    Ix = env.system["Ix"]
    Iy = env.system["Iy"]
    Iz = env.system["Iz"]
    gains = cfg.gains
    g = env.system["g"]  
    

    # desired parameters
    A = cfg.A
    w = cfg.w
    des_x = cfg.des_x
    des_y = cfg.des_y
    des_height = cfg.des_height
    des_yaw = cfg.des_yaw
    des_speed = cfg.des_speed

    # gains
    k1, k2, k3, k4 = gains["k1"], gains["k2"], gains["k3"], gains["k4"]
    k5, k6, k7, k8 = gains["k5"], gains["k6"], gains["k7"], gains["k8"]
    k9, k10, k11, k12 = gains["k9"], gains["k10"], gains["k11"], gains["k12"]
    k13, k14 = gains["k13"], gains["k14"]

    # NOTE:
    # Disturbances in your long expressions were dist_x/dist_y/dist_z.
    # Use env.disturbances(t) or keep them zero.
    dist_x, dist_y, dist_z = env.disturbances(t)

    # Transformed states
    xi1_1 = y - des_y - A * sin(w * (x - des_x))
    xi1_2 = -A*vx*w*cos(w*(-des_x + x)) + vy
    xi1_3 = A*vx**2*w**2*sin(w*(-des_x + x)) - A*w*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))*cos(w*(-des_x + x))/m + dist_y/m - kt*vy/m - zeta1*(sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m
    xi1_4 = A*dist_x*kt*w*cos(w*(des_x - x))/m**2 - 3*A*dist_x*vx*w**2*sin(w*(des_x - x))/m - A*kt**2*vx*w*cos(w*(des_x - x))/m**2 + 3*A*kt*vx**2*w**2*sin(w*(des_x - x))/m + A*kt*w*zeta1*sin(ph)*sin(ps)*cos(w*(des_x - x))/m**2 + A*kt*w*zeta1*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m**2 + A*vx**3*w**3*cos(w*(des_x - x)) + A*p*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/m - A*p*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/m - A*q*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/m - 3*A*vx*w**2*zeta1*sin(ph)*sin(ps)*sin(w*(des_x - x))/m - 3*A*vx*w**2*zeta1*sin(th)*sin(w*(des_x - x))*cos(ph)*cos(ps)/m - A*w*zeta2*sin(ph)*sin(ps)*cos(w*(des_x - x))/m - A*w*zeta2*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m - dist_y*kt/m**2 + kt**2*vy/m**2 + kt*zeta1*sin(ph)*cos(ps)/m**2 - kt*zeta1*sin(ps)*sin(th)*cos(ph)/m**2 - p*zeta1*sin(ph)*sin(ps)*sin(th)/m - p*zeta1*cos(ph)*cos(ps)/m + q*zeta1*sin(ps)*cos(th)/m - zeta2*sin(ph)*cos(ps)/m + zeta2*sin(ps)*sin(th)*cos(ph)/m

    Lg1Lf3S1 = -A*w*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))*cos(w*(-des_x + x))/m - (sin(ph)*cos(ps) - sin(ps)*sin(th)*cos(ph))/m

    Lg2Lf3S1 = (-A*w*zeta1*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))*cos(w*(-des_x + x))/m - zeta1*(sin(ph)*sin(ps)*sin(th) + cos(ph)*cos(ps))/m)/Ix

    Lg3Lf3S1 = zeta1*(-A*w*cos(ps)*cos(w*(des_x - x)) + sin(ps))*cos(th)/(Iy*m)
    
    Lg4Lf3S1 = 0
    
    Lf4S1 = A*Ix*p*r*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/(Iy*m) - 3*A*dist_x**2*w**2*sin(w*(des_x - x))/m**2 - A*dist_x*kt**2*w*cos(w*(des_x - x))/m**3 + 10*A*dist_x*kt*vx*w**2*sin(w*(des_x - x))/m**2 + 6*A*dist_x*vx**2*w**3*cos(w*(des_x - x))/m - 6*A*dist_x*w**2*zeta1*sin(ph)*sin(ps)*sin(w*(des_x - x))/m**2 - 6*A*dist_x*w**2*zeta1*sin(th)*sin(w*(des_x - x))*cos(ph)*cos(ps)/m**2 + A*kt**3*vx*w*cos(w*(des_x - x))/m**3 - 7*A*kt**2*vx**2*w**2*sin(w*(des_x - x))/m**2 - A*kt**2*w*zeta1*sin(ph)*sin(ps)*cos(w*(des_x - x))/m**3 - A*kt**2*w*zeta1*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m**3 - 6*A*kt*vx**3*w**3*cos(w*(des_x - x))/m - A*kt*p*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/m**2 + A*kt*p*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/m**2 + A*kt*q*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/m**2 + 10*A*kt*vx*w**2*zeta1*sin(ph)*sin(ps)*sin(w*(des_x - x))/m**2 + 10*A*kt*vx*w**2*zeta1*sin(th)*sin(w*(des_x - x))*cos(ph)*cos(ps)/m**2 + A*kt*w*zeta2*sin(ph)*sin(ps)*cos(w*(des_x - x))/m**2 + A*kt*w*zeta2*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m**2 + A*vx**4*w**4*sin(w*(des_x - x)) + A*p**2*w*zeta1*sin(ph)*sin(ps)*cos(w*(des_x - x))/m + A*p**2*w*zeta1*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m - A*p*r*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/m + 4*A*p*vx*w**2*zeta1*sin(ph)*sin(th)*sin(w*(des_x - x))*cos(ps)/m - 4*A*p*vx*w**2*zeta1*sin(ps)*sin(w*(des_x - x))*cos(ph)/m + 2*A*p*w*zeta2*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/m - 2*A*p*w*zeta2*sin(ps)*cos(ph)*cos(w*(des_x - x))/m + A*q**2*w*zeta1*sin(ph)*sin(ps)*cos(w*(des_x - x))/m + A*q**2*w*zeta1*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m - A*q*r*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/m + A*q*r*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/m - 4*A*q*vx*w**2*zeta1*sin(w*(des_x - x))*cos(ps)*cos(th)/m - 2*A*q*w*zeta2*cos(ps)*cos(th)*cos(w*(des_x - x))/m + 6*A*vx**2*w**3*zeta1*sin(ph)*sin(ps)*cos(w*(des_x - x))/m + 6*A*vx**2*w**3*zeta1*sin(th)*cos(ph)*cos(ps)*cos(w*(des_x - x))/m - 4*A*vx*w**2*zeta2*sin(ph)*sin(ps)*sin(w*(des_x - x))/m - 4*A*vx*w**2*zeta2*sin(th)*sin(w*(des_x - x))*cos(ph)*cos(ps)/m - 6*A*w**2*zeta1**2*sin(ph)*sin(ps)*sin(th)*sin(w*(des_x - x))*cos(ph)*cos(ps)/m**2 + 3*A*w**2*zeta1**2*sin(w*(des_x - x))*cos(ph)**2*cos(ps)**2*cos(th)**2/m**2 - 6*A*w**2*zeta1**2*sin(w*(des_x - x))*cos(ph)**2*cos(ps)**2/m**2 + 3*A*w**2*zeta1**2*sin(w*(des_x - x))*cos(ph)**2/m**2 + 3*A*w**2*zeta1**2*sin(w*(des_x - x))*cos(ps)**2/m**2 - 3*A*w**2*zeta1**2*sin(w*(des_x - x))/m**2 - A*Iz*p*r*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/(Iy*m) + A*kr*q*w*zeta1*cos(ps)*cos(th)*cos(w*(des_x - x))/(Iy*m) + A*Iy*q*r*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/(Ix*m) - A*Iy*q*r*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/(Ix*m) - A*Iz*q*r*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/(Ix*m) + A*Iz*q*r*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/(Ix*m) - A*kr*p*w*zeta1*sin(ph)*sin(th)*cos(ps)*cos(w*(des_x - x))/(Ix*m) + A*kr*p*w*zeta1*sin(ps)*cos(ph)*cos(w*(des_x - x))/(Ix*m) - Ix*p*r*zeta1*sin(ps)*cos(th)/(Iy*m) + dist_y*kt**2/m**3 - kt**3*vy/m**3 - kt**2*zeta1*sin(ph)*cos(ps)/m**3 + kt**2*zeta1*sin(ps)*sin(th)*cos(ph)/m**3 + kt*p*zeta1*sin(ph)*sin(ps)*sin(th)/m**2 + kt*p*zeta1*cos(ph)*cos(ps)/m**2 - kt*q*zeta1*sin(ps)*cos(th)/m**2 + kt*zeta2*sin(ph)*cos(ps)/m**2 - kt*zeta2*sin(ps)*sin(th)*cos(ph)/m**2 + p**2*zeta1*sin(ph)*cos(ps)/m - p**2*zeta1*sin(ps)*sin(th)*cos(ph)/m + p*r*zeta1*sin(ps)*cos(th)/m - 2*p*zeta2*sin(ph)*sin(ps)*sin(th)/m - 2*p*zeta2*cos(ph)*cos(ps)/m + q**2*zeta1*sin(ph)*cos(ps)/m - q**2*zeta1*sin(ps)*sin(th)*cos(ph)/m + q*r*zeta1*sin(ph)*sin(ps)*sin(th)/m + q*r*zeta1*cos(ph)*cos(ps)/m + 2*q*zeta2*sin(ps)*cos(th)/m + Iz*p*r*zeta1*sin(ps)*cos(th)/(Iy*m) - kr*q*zeta1*sin(ps)*cos(th)/(Iy*m) - Iy*q*r*zeta1*sin(ph)*sin(ps)*sin(th)/(Ix*m) - Iy*q*r*zeta1*cos(ph)*cos(ps)/(Ix*m) + Iz*q*r*zeta1*sin(ph)*sin(ps)*sin(th)/(Ix*m) + Iz*q*r*zeta1*cos(ph)*cos(ps)/(Ix*m) + kr*p*zeta1*sin(ph)*sin(ps)*sin(th)/(Ix*m) + kr*p*zeta1*cos(ph)*cos(ps)/(Ix*m)
    
    xi2_1 = z - des_height
    xi2_2 = vz
    xi2_3 = (dist_z - g*m - kt*vz + zeta1*cos(ph)*cos(th))/m
    xi2_4 = (-dist_z*kt + g*kt*m + kt**2*vz - kt*zeta1*cos(ph)*cos(th) - m*p*zeta1*sin(ph)*cos(th) - m*q*zeta1*sin(th) + m*zeta2*cos(ph)*cos(th))/m**2
    
    Lg1Lf3S2 = cos(ph)*cos(th)/m
    Lg2Lf3S2 = -(zeta1*cos(th)*sin(ph))/(Ix*m)
    Lg3Lf3S2 = -(zeta1*sin(th))/(Iy*m)
    Lg4Lf3S2 = 0
    Lf4S2 = kt**2*(dist_z/m - g - kt*vz/m + zeta1*cos(ph)*cos(th)/m)/m**2 + zeta2*(-kt*cos(ph)*cos(th) - m*p*sin(ph)*cos(th) - m*q*sin(th))/m**2 + (q*cos(ph) - r*sin(ph))*(kt*zeta1*sin(th)*cos(ph) + m*p*zeta1*sin(ph)*sin(th) - m*q*zeta1*cos(th) - m*zeta2*sin(th)*cos(ph))/m**2 + (p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))*(kt*zeta1*sin(ph)*cos(th) - m*p*zeta1*cos(ph)*cos(th) - m*zeta2*sin(ph)*cos(th))/m**2 - zeta1*(-Ix*p*r + Iz*p*r - kr*q)*sin(th)/(Iy*m) - zeta1*(Iy*q*r - Iz*q*r - kr*p)*sin(ph)*cos(th)/(Ix*m)
    
    eta1_1 = w * (x - des_x)
    eta1_2 = vx*w
    eta1_3 = w*(dist_x - kt*vx + zeta1*sin(ph)*sin(ps) + zeta1*sin(th)*cos(ph)*cos(ps))/m
    eta1_4 = w*(-dist_x*kt/m + kt**2*vx/m - kt*zeta1*sin(ph)*sin(ps)/m - kt*zeta1*sin(th)*cos(ph)*cos(ps)/m - p*zeta1*sin(ph)*sin(th)*cos(ps) + p*zeta1*sin(ps)*cos(ph) + q*zeta1*cos(ps)*cos(th) + zeta2*sin(ph)*sin(ps) + zeta2*sin(th)*cos(ph)*cos(ps))/m
    
    Lg1Lf3P1 = w*(sin(ph)*sin(ps) + sin(th)*cos(ph)*cos(ps))/m
    Lg2Lf3P1 = w*zeta1*(-sin(ph)*sin(th)*cos(ps) + sin(ps)*cos(ph))/(Ix*m)
    Lg3Lf3P1 = w*zeta1*cos(ps)*cos(th)/(Iy*m)
    Lg4Lf3P1 = 0
    Lf4P1 = w*(-Ix*p*r*zeta1*cos(ps)*cos(th)/Iy + dist_x*kt**2/m**2 - kt**3*vx/m**2 + kt**2*zeta1*sin(ph)*sin(ps)/m**2 + kt**2*zeta1*sin(th)*cos(ph)*cos(ps)/m**2 + kt*p*zeta1*sin(ph)*sin(th)*cos(ps)/m - kt*p*zeta1*sin(ps)*cos(ph)/m - kt*q*zeta1*cos(ps)*cos(th)/m - kt*zeta2*sin(ph)*sin(ps)/m - kt*zeta2*sin(th)*cos(ph)*cos(ps)/m - p**2*zeta1*sin(ph)*sin(ps) - p**2*zeta1*sin(th)*cos(ph)*cos(ps) + p*r*zeta1*cos(ps)*cos(th) - 2*p*zeta2*sin(ph)*sin(th)*cos(ps) + 2*p*zeta2*sin(ps)*cos(ph) - q**2*zeta1*sin(ph)*sin(ps) - q**2*zeta1*sin(th)*cos(ph)*cos(ps) + q*r*zeta1*sin(ph)*sin(th)*cos(ps) - q*r*zeta1*sin(ps)*cos(ph) + 2*q*zeta2*cos(ps)*cos(th) + Iz*p*r*zeta1*cos(ps)*cos(th)/Iy - kr*q*zeta1*cos(ps)*cos(th)/Iy - Iy*q*r*zeta1*sin(ph)*sin(th)*cos(ps)/Ix + Iy*q*r*zeta1*sin(ps)*cos(ph)/Ix + Iz*q*r*zeta1*sin(ph)*sin(th)*cos(ps)/Ix - Iz*q*r*zeta1*sin(ps)*cos(ph)/Ix + kr*p*zeta1*sin(ph)*sin(th)*cos(ps)/Ix - kr*p*zeta1*sin(ps)*cos(ph)/Ix)/m
    
    eta2_1 = ps - des_yaw;
    eta2_2 = (r*cos(ph) + q*sin(ph))/cos(th)
    
    Lg1LfP2 = 0
    Lg2LfP2 = 0
    Lg3LfP2 = sin(ph)/(Iy*cos(th))
    Lg4LfP2 = cos(ph)/(Iz*cos(th))
    Lf2P2 = (q*sin(ph) + r*cos(ph))*(q*cos(ph) - r*sin(ph))*sin(th)/cos(th)**2 + (q*cos(ph) - r*sin(ph))*(p + q*sin(ph)*sin(th)/cos(th) + r*sin(th)*cos(ph)/cos(th))/cos(th) + (Ix*p*q - Iy*p*q - kr*r)*cos(ph)/(Iz*cos(th)) + (-Ix*p*r + Iz*p*r - kr*q)*sin(ph)/(Iy*cos(th))
    
    # Control inputs
    D = np.array([
        [Lg1Lf3S1, Lg2Lf3S1, Lg3Lf3S1, Lg4Lf3S1],
        [Lg1Lf3S2, Lg2Lf3S2, Lg3Lf3S2, Lg4Lf3S2],
        [Lg1Lf3P1, Lg2Lf3P1, Lg3Lf3P1, Lg4Lf3P1],
        [Lg1LfP2,  Lg2LfP2,  Lg3LfP2,  Lg4LfP2]
    ])


    v1_tran = -Lf4S1 - k1*xi1_1 - k2*xi1_2 - k3*xi1_3 - k4*xi1_4
    v2_tran = -Lf4S2 - k5*xi2_1 - k6*xi2_2 - k7*xi2_3 - k8*xi2_4

    v1_tang = -Lf4P1 - k10*(eta1_2 - des_speed) - k11*eta1_3 - k12*eta1_4 
    # v1_tang = -Lf4P1 - k9*(eta1_1 + np.pi/80) - k10*(eta1_2) - k11*eta1_3 - k12*eta1_4

    v2_tang = -Lf2P2 - k13*(eta2_1) - k14*eta2_2
    # v2_tang = -Lf2P2

    rhs = np.array([v1_tran, v2_tran, v1_tang, v2_tang])
    
    # Singularity Avoidance ECBF Terms
    h_th = th_max**2 - th**2
    Lfh_th = -2*th*(q*cos(ph) - r*sin(ph))
    Lf2h_th = 2*th*(q*sin(ph) + r*cos(ph))*(p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th)) - 2*(q*cos(ph) - r*sin(ph))**2 + 2*th*(Ix*p*q - Iy*p*q - kr*r)*sin(ph)/Iz - 2*th*(-Ix*p*r + Iz*p*r - kr*q)*cos(ph)/Iy
    Lg3Lfh_th = -2*th*cos(ph)/Iy
    Lg4Lfh_th = 2*th*sin(ph)/Iz
    
    h_ph = ph_max**2 - ph**2
    Lfh_ph = -2*ph*(p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th))
    Lf2h_ph = -2*ph*(q*sin(ph) + r*cos(ph))*(q*cos(ph) - r*sin(ph))/cos(th)**2 + (p + q*sin(ph)*tan(th) + r*cos(ph)*tan(th))*(-2*p - 2*ph*(q*cos(ph) - r*sin(ph))*tan(th) - 2*q*sin(ph)*tan(th) - 2*r*cos(ph)*tan(th)) - 2*ph*(Ix*p*q - Iy*p*q - kr*r)*cos(ph)*tan(th)/Iz - 2*ph*(-Ix*p*r + Iz*p*r - kr*q)*sin(ph)*tan(th)/Iy - 2*ph*(Iy*q*r - Iz*q*r - kr*p)/Ix
    Lg2Lfh_ph = -2*ph/Ix
    Lg3Lfh_ph = -2*ph*sin(ph)*tan(th)/Iy
    Lg4Lfh_ph = -2*ph*cos(ph)*tan(th)/Iz
    
    
    # -------------------------
    # Pairwise rd=4 collision ECBF terms
    # -------------------------

    ph_i, th_i, ps_i, p_i, q_i, r_i, x_i, y_i, z_i, vx_i, vy_i, vz_i = state
    zeta1_i = float(zeta1)
    zeta2_i = float(zeta2)

    pair_cbf_terms = {}

    for j_id, sj in world_snapshot.items():
        if j_id == agent_id:
            continue

        xj = np.array(sj["state"], dtype=float).reshape(12)
        ph_j, th_j, ps_j, p_j, q_j, r_j, x_j, y_j, z_j, vx_j, vy_j, vz_j = xj
        zeta1_j = float(sj["zeta1"])
        zeta2_j = float(sj["zeta2"])

        B = (x_i - x_j)**2 + (y_i - y_j)**2 + (z_i - z_j)**2 - Ds**2

        LfB = 2*vx_i*(x_i - x_j) - 2*vx_j*(x_i - x_j) + 2*vy_i*(y_i - y_j) - 2*vy_j*(y_i - y_j) + 2*vz_i*(z_i - z_j) - 2*vz_j*(z_i - z_j)
        Lf2B = 2*(m*(vx_i*(vx_i - vx_j) - vx_j*(vx_i - vx_j) + vy_i*(vy_i - vy_j) - vy_j*(vy_i - vy_j) + vz_i*(vz_i - vz_j) - vz_j*(vz_i - vz_j)) + (-x_i + x_j)*(kt*vx_i - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i))) + (x_i - x_j)*(kt*vx_j - zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + (-y_i + y_j)*(kt*vy_i + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i))) + (y_i - y_j)*(kt*vy_j + zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) + (-z_i + z_j)*(g*m + kt*vz_i - zeta1_i*cos(ph_i)*cos(th_i)) + (z_i - z_j)*(g*m + kt*vz_j - zeta1_j*cos(ph_j)*cos(th_j)))/m
        Lf3B = 2*(m*zeta1_i*((q_i*sin(ph_i) + r_i*cos(ph_i))*((x_i - x_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i))) - (p_i*cos(th_i) + q_i*sin(ph_i)*sin(th_i) + r_i*sin(th_i)*cos(ph_i))*((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i)))*cos(th_j) + m*zeta1_j*(-(q_j*sin(ph_j) + r_j*cos(ph_j))*((x_i - x_j)*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)) + (y_i - y_j)*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + (p_j*cos(th_j) + q_j*sin(ph_j)*sin(th_j) + r_j*sin(th_j)*cos(ph_j))*((x_i - x_j)*(sin(ph_j)*sin(th_j)*cos(ps_j) - sin(ps_j)*cos(ph_j)) + (y_i - y_j)*(sin(ph_j)*sin(ps_j)*sin(th_j) + cos(ph_j)*cos(ps_j)) + (z_i - z_j)*sin(ph_j)*cos(th_j)))*cos(th_i) + m*(-vx_i*(kt*vx_i - kt*vx_j - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) + vx_j*(kt*vx_i - kt*vx_j - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j))) - vy_i*(kt*vy_i - kt*vy_j + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) - zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) + vy_j*(kt*vy_i - kt*vy_j + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) - zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j))) - vz_i*(kt*vz_i - kt*vz_j - zeta1_i*cos(ph_i)*cos(th_i) + zeta1_j*cos(ph_j)*cos(th_j)) + vz_j*(kt*vz_i - kt*vz_j - zeta1_i*cos(ph_i)*cos(th_i) + zeta1_j*cos(ph_j)*cos(th_j)) + zeta1_i*(q_i*cos(ph_i) - r_i*sin(ph_i))*((x_i - x_j)*cos(ps_i)*cos(th_i) + (y_i - y_j)*sin(ps_i)*cos(th_i) + (-z_i + z_j)*sin(th_i))*cos(ph_i) - zeta1_j*(q_j*cos(ph_j) - r_j*sin(ph_j))*((x_i - x_j)*cos(ps_j)*cos(th_j) + (y_i - y_j)*sin(ps_j)*cos(th_j) + (-z_i + z_j)*sin(th_j))*cos(ph_j) + zeta2_i*((x_i - x_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + (-y_i + y_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (z_i - z_j)*cos(ph_i)*cos(th_i)) - zeta2_j*((x_i - x_j)*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j)) + (-y_i + y_j)*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)) + (z_i - z_j)*cos(ph_j)*cos(th_j)))*cos(th_i)*cos(th_j) + ((kt*vx_i - zeta1_i*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)))*(kt*(x_i - x_j) - 2*m*(vx_i - vx_j)) + (kt*vx_j - zeta1_j*(sin(ph_j)*sin(ps_j) + sin(th_j)*cos(ph_j)*cos(ps_j)))*(-kt*(x_i - x_j) + 2*m*(vx_i - vx_j)) + (kt*vy_i + zeta1_i*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)))*(kt*(y_i - y_j) - 2*m*(vy_i - vy_j)) + (kt*vy_j + zeta1_j*(sin(ph_j)*cos(ps_j) - sin(ps_j)*sin(th_j)*cos(ph_j)))*(-kt*(y_i - y_j) + 2*m*(vy_i - vy_j)) + (-kt*(z_i - z_j) + 2*m*(vz_i - vz_j))*(g*m + kt*vz_j - zeta1_j*cos(ph_j)*cos(th_j)) + (kt*(z_i - z_j) - 2*m*(vz_i - vz_j))*(g*m + kt*vz_i - zeta1_i*cos(ph_i)*cos(th_i)))*cos(th_i)*cos(th_j))/(m**2*cos(th_i)*cos(th_j))
        Lf4B = -2*Ix*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) - 2*Ix*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) + 2*Ix*zeta1_i*p_i*r_i*z_i*sin(th_i)/(Iy*m) - 2*Ix*zeta1_i*p_i*r_i*z_j*sin(th_i)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) - 2*Ix*zeta1_j*p_j*r_j*z_i*sin(th_j)/(Iy*m) + 2*Ix*zeta1_j*p_j*r_j*z_j*sin(th_j)/(Iy*m) + 6*zeta1_i**2/m**2 - 12*zeta1_i*zeta1_j*sin(ph_i)*sin(ph_j)*cos(ps_i - ps_j)/m**2 - 12*zeta1_i*zeta1_j*sin(ph_i)*sin(th_j)*sin(ps_i - ps_j)*cos(ph_j)/m**2 + 12*zeta1_i*zeta1_j*sin(ph_j)*sin(th_i)*sin(ps_i - ps_j)*cos(ph_i)/m**2 - 12*zeta1_i*zeta1_j*sin(th_i)*sin(th_j)*cos(ph_i)*cos(ph_j)*cos(ps_i - ps_j)/m**2 - 12*zeta1_i*zeta1_j*cos(ph_i)*cos(ph_j)*cos(th_i)*cos(th_j)/m**2 + 2*zeta1_i*kt**2*x_i*sin(ph_i)*sin(ps_i)/m**3 + 2*zeta1_i*kt**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*x_j*sin(ph_i)*sin(ps_i)/m**3 - 2*zeta1_i*kt**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*y_i*sin(ph_i)*cos(ps_i)/m**3 + 2*zeta1_i*kt**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**3 + 2*zeta1_i*kt**2*y_j*sin(ph_i)*cos(ps_i)/m**3 - 2*zeta1_i*kt**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**3 + 2*zeta1_i*kt**2*z_i*cos(ph_i)*cos(th_i)/m**3 - 2*zeta1_i*kt**2*z_j*cos(ph_i)*cos(th_i)/m**3 + 2*zeta1_i*kt*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m**2 - 2*zeta1_i*kt*p_i*x_i*sin(ps_i)*cos(ph_i)/m**2 - 2*zeta1_i*kt*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m**2 + 2*zeta1_i*kt*p_i*x_j*sin(ps_i)*cos(ph_i)/m**2 + 2*zeta1_i*kt*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m**2 + 2*zeta1_i*kt*p_i*y_i*cos(ph_i)*cos(ps_i)/m**2 - 2*zeta1_i*kt*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m**2 - 2*zeta1_i*kt*p_i*y_j*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta1_i*kt*p_i*z_i*sin(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*p_i*z_j*sin(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*q_i*x_i*cos(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*x_j*cos(ps_i)*cos(th_i)/m**2 - 2*zeta1_i*kt*q_i*y_i*sin(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*y_j*sin(ps_i)*cos(th_i)/m**2 + 2*zeta1_i*kt*q_i*z_i*sin(th_i)/m**2 - 2*zeta1_i*kt*q_i*z_j*sin(th_i)/m**2 - 20*zeta1_i*kt*vx_i*sin(ph_i)*sin(ps_i)/m**2 - 20*zeta1_i*kt*vx_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vx_j*sin(ph_i)*sin(ps_i)/m**2 + 20*zeta1_i*kt*vx_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vy_i*sin(ph_i)*cos(ps_i)/m**2 - 20*zeta1_i*kt*vy_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 20*zeta1_i*kt*vy_j*sin(ph_i)*cos(ps_i)/m**2 + 20*zeta1_i*kt*vy_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 20*zeta1_i*kt*vz_i*cos(ph_i)*cos(th_i)/m**2 + 20*zeta1_i*kt*vz_j*cos(ph_i)*cos(th_i)/m**2 - 2*zeta1_i*p_i**2*x_i*sin(ph_i)*sin(ps_i)/m - 2*zeta1_i*p_i**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*x_j*sin(ph_i)*sin(ps_i)/m + 2*zeta1_i*p_i**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*y_i*sin(ph_i)*cos(ps_i)/m - 2*zeta1_i*p_i**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*p_i**2*y_j*sin(ph_i)*cos(ps_i)/m + 2*zeta1_i*p_i**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*p_i**2*z_i*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*p_i**2*z_j*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/m + 2*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/m - 2*zeta1_i*p_i*r_i*z_i*sin(th_i)/m + 2*zeta1_i*p_i*r_i*z_j*sin(th_i)/m - 8*zeta1_i*p_i*vx_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 8*zeta1_i*p_i*vx_i*sin(ps_i)*cos(ph_i)/m + 8*zeta1_i*p_i*vx_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 8*zeta1_i*p_i*vx_j*sin(ps_i)*cos(ph_i)/m - 8*zeta1_i*p_i*vy_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 8*zeta1_i*p_i*vy_i*cos(ph_i)*cos(ps_i)/m + 8*zeta1_i*p_i*vy_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 8*zeta1_i*p_i*vy_j*cos(ph_i)*cos(ps_i)/m - 8*zeta1_i*p_i*vz_i*sin(ph_i)*cos(th_i)/m + 8*zeta1_i*p_i*vz_j*sin(ph_i)*cos(th_i)/m - 2*zeta1_i*q_i**2*x_i*sin(ph_i)*sin(ps_i)/m - 2*zeta1_i*q_i**2*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*x_j*sin(ph_i)*sin(ps_i)/m + 2*zeta1_i*q_i**2*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*y_i*sin(ph_i)*cos(ps_i)/m - 2*zeta1_i*q_i**2*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*q_i**2*y_j*sin(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i**2*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m - 2*zeta1_i*q_i**2*z_i*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*q_i**2*z_j*cos(ph_i)*cos(th_i)/m + 2*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 2*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/m - 2*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 2*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/m + 2*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 2*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/m - 2*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 2*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/m + 2*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/m - 2*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/m + 8*zeta1_i*q_i*vx_i*cos(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vx_j*cos(ps_i)*cos(th_i)/m + 8*zeta1_i*q_i*vy_i*sin(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vy_j*sin(ps_i)*cos(th_i)/m - 8*zeta1_i*q_i*vz_i*sin(th_i)/m + 8*zeta1_i*q_i*vz_j*sin(th_i)/m + 6*zeta1_j**2/m**2 - 2*zeta1_j*kt**2*x_i*sin(ph_j)*sin(ps_j)/m**3 - 2*zeta1_j*kt**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*x_j*sin(ph_j)*sin(ps_j)/m**3 + 2*zeta1_j*kt**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*y_i*sin(ph_j)*cos(ps_j)/m**3 - 2*zeta1_j*kt**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**3 - 2*zeta1_j*kt**2*y_j*sin(ph_j)*cos(ps_j)/m**3 + 2*zeta1_j*kt**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**3 - 2*zeta1_j*kt**2*z_i*cos(ph_j)*cos(th_j)/m**3 + 2*zeta1_j*kt**2*z_j*cos(ph_j)*cos(th_j)/m**3 - 2*zeta1_j*kt*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m**2 + 2*zeta1_j*kt*p_j*x_i*sin(ps_j)*cos(ph_j)/m**2 + 2*zeta1_j*kt*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m**2 - 2*zeta1_j*kt*p_j*x_j*sin(ps_j)*cos(ph_j)/m**2 - 2*zeta1_j*kt*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m**2 - 2*zeta1_j*kt*p_j*y_i*cos(ph_j)*cos(ps_j)/m**2 + 2*zeta1_j*kt*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m**2 + 2*zeta1_j*kt*p_j*y_j*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta1_j*kt*p_j*z_i*sin(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*p_j*z_j*sin(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*q_j*x_i*cos(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*x_j*cos(ps_j)*cos(th_j)/m**2 + 2*zeta1_j*kt*q_j*y_i*sin(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*y_j*sin(ps_j)*cos(th_j)/m**2 - 2*zeta1_j*kt*q_j*z_i*sin(th_j)/m**2 + 2*zeta1_j*kt*q_j*z_j*sin(th_j)/m**2 + 20*zeta1_j*kt*vx_i*sin(ph_j)*sin(ps_j)/m**2 + 20*zeta1_j*kt*vx_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vx_j*sin(ph_j)*sin(ps_j)/m**2 - 20*zeta1_j*kt*vx_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vy_i*sin(ph_j)*cos(ps_j)/m**2 + 20*zeta1_j*kt*vy_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 20*zeta1_j*kt*vy_j*sin(ph_j)*cos(ps_j)/m**2 - 20*zeta1_j*kt*vy_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 20*zeta1_j*kt*vz_i*cos(ph_j)*cos(th_j)/m**2 - 20*zeta1_j*kt*vz_j*cos(ph_j)*cos(th_j)/m**2 + 2*zeta1_j*p_j**2*x_i*sin(ph_j)*sin(ps_j)/m + 2*zeta1_j*p_j**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*x_j*sin(ph_j)*sin(ps_j)/m - 2*zeta1_j*p_j**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*y_i*sin(ph_j)*cos(ps_j)/m + 2*zeta1_j*p_j**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*p_j**2*y_j*sin(ph_j)*cos(ps_j)/m - 2*zeta1_j*p_j**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*p_j**2*z_i*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*p_j**2*z_j*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/m - 2*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/m + 2*zeta1_j*p_j*r_j*z_i*sin(th_j)/m - 2*zeta1_j*p_j*r_j*z_j*sin(th_j)/m + 8*zeta1_j*p_j*vx_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 8*zeta1_j*p_j*vx_i*sin(ps_j)*cos(ph_j)/m - 8*zeta1_j*p_j*vx_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 8*zeta1_j*p_j*vx_j*sin(ps_j)*cos(ph_j)/m + 8*zeta1_j*p_j*vy_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 8*zeta1_j*p_j*vy_i*cos(ph_j)*cos(ps_j)/m - 8*zeta1_j*p_j*vy_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 8*zeta1_j*p_j*vy_j*cos(ph_j)*cos(ps_j)/m + 8*zeta1_j*p_j*vz_i*sin(ph_j)*cos(th_j)/m - 8*zeta1_j*p_j*vz_j*sin(ph_j)*cos(th_j)/m + 2*zeta1_j*q_j**2*x_i*sin(ph_j)*sin(ps_j)/m + 2*zeta1_j*q_j**2*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*x_j*sin(ph_j)*sin(ps_j)/m - 2*zeta1_j*q_j**2*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*y_i*sin(ph_j)*cos(ps_j)/m + 2*zeta1_j*q_j**2*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*q_j**2*y_j*sin(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j**2*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m + 2*zeta1_j*q_j**2*z_i*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*q_j**2*z_j*cos(ph_j)*cos(th_j)/m - 2*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 2*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/m + 2*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 2*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/m - 2*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 2*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/m + 2*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 2*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/m - 2*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/m + 2*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/m - 8*zeta1_j*q_j*vx_i*cos(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vx_j*cos(ps_j)*cos(th_j)/m - 8*zeta1_j*q_j*vy_i*sin(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vy_j*sin(ps_j)*cos(th_j)/m + 8*zeta1_j*q_j*vz_i*sin(th_j)/m - 8*zeta1_j*q_j*vz_j*sin(th_j)/m - 2*zeta2_i*kt*x_i*sin(ph_i)*sin(ps_i)/m**2 - 2*zeta2_i*kt*x_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*x_j*sin(ph_i)*sin(ps_i)/m**2 + 2*zeta2_i*kt*x_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*y_i*sin(ph_i)*cos(ps_i)/m**2 - 2*zeta2_i*kt*y_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 2*zeta2_i*kt*y_j*sin(ph_i)*cos(ps_i)/m**2 + 2*zeta2_i*kt*y_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m**2 - 2*zeta2_i*kt*z_i*cos(ph_i)*cos(th_i)/m**2 + 2*zeta2_i*kt*z_j*cos(ph_i)*cos(th_i)/m**2 - 4*zeta2_i*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/m + 4*zeta2_i*p_i*x_i*sin(ps_i)*cos(ph_i)/m + 4*zeta2_i*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/m - 4*zeta2_i*p_i*x_j*sin(ps_i)*cos(ph_i)/m - 4*zeta2_i*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/m - 4*zeta2_i*p_i*y_i*cos(ph_i)*cos(ps_i)/m + 4*zeta2_i*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/m + 4*zeta2_i*p_i*y_j*cos(ph_i)*cos(ps_i)/m - 4*zeta2_i*p_i*z_i*sin(ph_i)*cos(th_i)/m + 4*zeta2_i*p_i*z_j*sin(ph_i)*cos(th_i)/m + 4*zeta2_i*q_i*x_i*cos(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*x_j*cos(ps_i)*cos(th_i)/m + 4*zeta2_i*q_i*y_i*sin(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*y_j*sin(ps_i)*cos(th_i)/m - 4*zeta2_i*q_i*z_i*sin(th_i)/m + 4*zeta2_i*q_i*z_j*sin(th_i)/m + 8*zeta2_i*vx_i*sin(ph_i)*sin(ps_i)/m + 8*zeta2_i*vx_i*sin(th_i)*cos(ph_i)*cos(ps_i)/m - 8*zeta2_i*vx_j*sin(ph_i)*sin(ps_i)/m - 8*zeta2_i*vx_j*sin(th_i)*cos(ph_i)*cos(ps_i)/m - 8*zeta2_i*vy_i*sin(ph_i)*cos(ps_i)/m + 8*zeta2_i*vy_i*sin(ps_i)*sin(th_i)*cos(ph_i)/m + 8*zeta2_i*vy_j*sin(ph_i)*cos(ps_i)/m - 8*zeta2_i*vy_j*sin(ps_i)*sin(th_i)*cos(ph_i)/m + 8*zeta2_i*vz_i*cos(ph_i)*cos(th_i)/m - 8*zeta2_i*vz_j*cos(ph_i)*cos(th_i)/m + 2*zeta2_j*kt*x_i*sin(ph_j)*sin(ps_j)/m**2 + 2*zeta2_j*kt*x_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*x_j*sin(ph_j)*sin(ps_j)/m**2 - 2*zeta2_j*kt*x_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*y_i*sin(ph_j)*cos(ps_j)/m**2 + 2*zeta2_j*kt*y_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 2*zeta2_j*kt*y_j*sin(ph_j)*cos(ps_j)/m**2 - 2*zeta2_j*kt*y_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m**2 + 2*zeta2_j*kt*z_i*cos(ph_j)*cos(th_j)/m**2 - 2*zeta2_j*kt*z_j*cos(ph_j)*cos(th_j)/m**2 + 4*zeta2_j*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/m - 4*zeta2_j*p_j*x_i*sin(ps_j)*cos(ph_j)/m - 4*zeta2_j*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/m + 4*zeta2_j*p_j*x_j*sin(ps_j)*cos(ph_j)/m + 4*zeta2_j*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/m + 4*zeta2_j*p_j*y_i*cos(ph_j)*cos(ps_j)/m - 4*zeta2_j*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/m - 4*zeta2_j*p_j*y_j*cos(ph_j)*cos(ps_j)/m + 4*zeta2_j*p_j*z_i*sin(ph_j)*cos(th_j)/m - 4*zeta2_j*p_j*z_j*sin(ph_j)*cos(th_j)/m - 4*zeta2_j*q_j*x_i*cos(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*x_j*cos(ps_j)*cos(th_j)/m - 4*zeta2_j*q_j*y_i*sin(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*y_j*sin(ps_j)*cos(th_j)/m + 4*zeta2_j*q_j*z_i*sin(th_j)/m - 4*zeta2_j*q_j*z_j*sin(th_j)/m - 8*zeta2_j*vx_i*sin(ph_j)*sin(ps_j)/m - 8*zeta2_j*vx_i*sin(th_j)*cos(ph_j)*cos(ps_j)/m + 8*zeta2_j*vx_j*sin(ph_j)*sin(ps_j)/m + 8*zeta2_j*vx_j*sin(th_j)*cos(ph_j)*cos(ps_j)/m + 8*zeta2_j*vy_i*sin(ph_j)*cos(ps_j)/m - 8*zeta2_j*vy_i*sin(ps_j)*sin(th_j)*cos(ph_j)/m - 8*zeta2_j*vy_j*sin(ph_j)*cos(ps_j)/m + 8*zeta2_j*vy_j*sin(ps_j)*sin(th_j)*cos(ph_j)/m - 8*zeta2_j*vz_i*cos(ph_j)*cos(th_j)/m + 8*zeta2_j*vz_j*cos(ph_j)*cos(th_j)/m - 2*kt**3*vx_i*x_i/m**3 + 2*kt**3*vx_i*x_j/m**3 + 2*kt**3*vx_j*x_i/m**3 - 2*kt**3*vx_j*x_j/m**3 - 2*kt**3*vy_i*y_i/m**3 + 2*kt**3*vy_i*y_j/m**3 + 2*kt**3*vy_j*y_i/m**3 - 2*kt**3*vy_j*y_j/m**3 - 2*kt**3*vz_i*z_i/m**3 + 2*kt**3*vz_i*z_j/m**3 + 2*kt**3*vz_j*z_i/m**3 - 2*kt**3*vz_j*z_j/m**3 + 14*kt**2*vx_i**2/m**2 - 28*kt**2*vx_i*vx_j/m**2 + 14*kt**2*vx_j**2/m**2 + 14*kt**2*vy_i**2/m**2 - 28*kt**2*vy_i*vy_j/m**2 + 14*kt**2*vy_j**2/m**2 + 14*kt**2*vz_i**2/m**2 - 28*kt**2*vz_i*vz_j/m**2 + 14*kt**2*vz_j**2/m**2 + 2*Iz*zeta1_i*p_i*r_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) + 2*Iz*zeta1_i*p_i*r_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) - 2*Iz*zeta1_i*p_i*r_i*z_i*sin(th_i)/(Iy*m) + 2*Iz*zeta1_i*p_i*r_i*z_j*sin(th_i)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) + 2*Iz*zeta1_j*p_j*r_j*z_i*sin(th_j)/(Iy*m) - 2*Iz*zeta1_j*p_j*r_j*z_j*sin(th_j)/(Iy*m) - 2*zeta1_i*kr*q_i*x_i*cos(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*x_j*cos(ps_i)*cos(th_i)/(Iy*m) - 2*zeta1_i*kr*q_i*y_i*sin(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*y_j*sin(ps_i)*cos(th_i)/(Iy*m) + 2*zeta1_i*kr*q_i*z_i*sin(th_i)/(Iy*m) - 2*zeta1_i*kr*q_i*z_j*sin(th_i)/(Iy*m) + 2*zeta1_j*kr*q_j*x_i*cos(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*x_j*cos(ps_j)*cos(th_j)/(Iy*m) + 2*zeta1_j*kr*q_j*y_i*sin(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*y_j*sin(ps_j)*cos(th_j)/(Iy*m) - 2*zeta1_j*kr*q_j*z_i*sin(th_j)/(Iy*m) + 2*zeta1_j*kr*q_j*z_j*sin(th_j)/(Iy*m) - 2*Iy*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*Iy*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) + 2*Iy*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*Iy*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) - 2*Iy*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*Iz*zeta1_i*q_i*r_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) - 2*Iz*zeta1_i*q_i*r_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*Iz*zeta1_j*q_j*r_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) + 2*Iz*zeta1_j*q_j*r_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m) + 2*zeta1_i*kr*p_i*x_i*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) - 2*zeta1_i*kr*p_i*x_i*sin(ps_i)*cos(ph_i)/(Ix*m) - 2*zeta1_i*kr*p_i*x_j*sin(ph_i)*sin(th_i)*cos(ps_i)/(Ix*m) + 2*zeta1_i*kr*p_i*x_j*sin(ps_i)*cos(ph_i)/(Ix*m) + 2*zeta1_i*kr*p_i*y_i*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) + 2*zeta1_i*kr*p_i*y_i*cos(ph_i)*cos(ps_i)/(Ix*m) - 2*zeta1_i*kr*p_i*y_j*sin(ph_i)*sin(ps_i)*sin(th_i)/(Ix*m) - 2*zeta1_i*kr*p_i*y_j*cos(ph_i)*cos(ps_i)/(Ix*m) + 2*zeta1_i*kr*p_i*z_i*sin(ph_i)*cos(th_i)/(Ix*m) - 2*zeta1_i*kr*p_i*z_j*sin(ph_i)*cos(th_i)/(Ix*m) - 2*zeta1_j*kr*p_j*x_i*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) + 2*zeta1_j*kr*p_j*x_i*sin(ps_j)*cos(ph_j)/(Ix*m) + 2*zeta1_j*kr*p_j*x_j*sin(ph_j)*sin(th_j)*cos(ps_j)/(Ix*m) - 2*zeta1_j*kr*p_j*x_j*sin(ps_j)*cos(ph_j)/(Ix*m) - 2*zeta1_j*kr*p_j*y_i*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) - 2*zeta1_j*kr*p_j*y_i*cos(ph_j)*cos(ps_j)/(Ix*m) + 2*zeta1_j*kr*p_j*y_j*sin(ph_j)*sin(ps_j)*sin(th_j)/(Ix*m) + 2*zeta1_j*kr*p_j*y_j*cos(ph_j)*cos(ps_j)/(Ix*m) - 2*zeta1_j*kr*p_j*z_i*sin(ph_j)*cos(th_j)/(Ix*m) + 2*zeta1_j*kr*p_j*z_j*sin(ph_j)*cos(th_j)/(Ix*m)

        Lg1_iLf3B = 2*((x_i - x_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)) + (-y_i + y_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (z_i - z_j)*cos(ph_i)*cos(th_i))/m
        Lg2_iLf3B = -2*zeta1_i*((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i))/(Ix*m)
        Lg3_iLf3B = 2*(zeta1_i*m*(((x_i - x_j)*(sin(ph_i)*cos(ps_i) - sin(ps_i)*sin(th_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i) + sin(th_i)*cos(ph_i)*cos(ps_i)))*sin(ph_i) - ((x_i - x_j)*(sin(ph_i)*sin(th_i)*cos(ps_i) - sin(ps_i)*cos(ph_i)) + (y_i - y_j)*(sin(ph_i)*sin(ps_i)*sin(th_i) + cos(ph_i)*cos(ps_i)) + (z_i - z_j)*sin(ph_i)*cos(th_i))*sin(ph_i)*sin(th_i))*cos(th_j) + zeta1_i*m*((x_i - x_j)*cos(ps_i)*cos(th_i) + (y_i - y_j)*sin(ps_i)*cos(th_i) + (-z_i + z_j)*sin(th_i))*cos(ph_i)**2*cos(th_i)*cos(th_j))/(Iy*m**2*cos(th_i)*cos(th_j))
        Lg4_iLf3B = 0

        # ------------------------------------------------
        pair_cbf_terms[j_id] = {

            # rd=4 ECBF core terms
            "B": float(B),
            "LfB": float(LfB),
            "Lf2B": float(Lf2B),
            "Lf3B": float(Lf3B),
            "Lf4B": float(Lf4B),

            # input directions (agent i only, decentralized form)
            "Lg1_iLf3B": float(Lg1_iLf3B),
            "Lg2_iLf3B": float(Lg2_iLf3B),
            "Lg3_iLf3B": float(Lg3_iLf3B),
            "Lg4_iLf3B": float(Lg4_iLf3B),

            # (optional but very useful for debugging)
            "j_name": sj.get("name", f"agent_{j_id}"),
        }
    
    
    return {
    "D": np.array(D, dtype=float).reshape(4, 4),
    "rhs": np.array(rhs, dtype=float).reshape(4,),
    "cbf_terms": {
        "th": {
            "Lf2h": float(Lf2h_th),
            "Lg3Lfh": float(Lg3Lfh_th),
            "Lg4Lfh": float(Lg4Lfh_th),
            "Lfh": float(Lfh_th),
            "h": float(h_th),
        },
        "ph": {
            "Lf2h": float(Lf2h_ph),
            "Lg2Lfh": float(Lg2Lfh_ph),
            "Lg3Lfh": float(Lg3Lfh_ph),
            "Lg4Lfh": float(Lg4Lfh_ph),
            "Lfh": float(Lfh_ph),
            "h": float(h_ph),
        },
    },
    "pair_cbf_terms": pair_cbf_terms,
    "tracking_vars": {
        "eta1_2": float(eta1_2),
        "eta2_1": float(eta2_1),
        "xi1_1": float(xi1_1),
        "xi2_1": float(xi2_1),
    },
}

def solve_qp_for_u(D, rhs, cbf_terms, pair_cbf_terms, cfg: QuadConfig):
    """
    Solves your per-quad QP:
      min u^T Q u + delta^T P delta
      s.t. D u == rhs + [0,0,delta0,delta1]
           roll/pitch CBF constraints
    Returns:
      u (4,), delta(2,)
    """
    lam_th = float(cfg.cbf_lambdas["lam_th"])
    lam_ph = float(cfg.cbf_lambdas["lam_ph"])
    k1_th, k0_th = 2*lam_th, lam_th**2
    k1_ph, k0_ph = 2*lam_ph, lam_ph**2
    resp_w = cfg.resp_w

    Q_u = 1
    P_delta = 5
    # P_delta1 = np.array([
    #     [0.5,0],
    #     [0,1]
    # ])

    u = cp.Variable(4)
    delta = cp.Variable(2)

    objective = cp.Minimize(Q_u*cp.sum_squares(u) + P_delta*cp.sum_squares(delta))
    #objective = cp.Minimize(Q_u*cp.sum_squares(u) + cp.quad_form(delta, P_delta1))

    delta4 = cp.hstack([0, 0, delta[0], delta[1]])

    # roll/pitch cbf constraints (same structure as your code)
    th = cbf_terms["th"]
    ph = cbf_terms["ph"]

    th_cbf = (th["Lf2h"] + th["Lg3Lfh"]*u[2] + th["Lg4Lfh"]*u[3] + k1_th*th["Lfh"] + k0_th*th["h"] >= 0)
    ph_cbf = (ph["Lf2h"] + ph["Lg2Lfh"]*u[1] + ph["Lg3Lfh"]*u[2] + ph["Lg4Lfh"]*u[3] + k1_ph*ph["Lfh"] + k0_ph*ph["h"] >= 0)

    constraints = [
        D @ u == rhs + delta4
        # th_cbf,
        # ph_cbf
    ]
    for j_id, pair in pair_cbf_terms.items():
        constraint = (pair["Lg1_iLf3B"] * u[0] + pair["Lg2_iLf3B"] * u[1] + pair["Lg3_iLf3B"] * u[2] + pair["Lg4_iLf3B"] * u[3]
                      >= -resp_w*(pair["Lf4B"] + alpha3 * pair["Lf3B"] + alpha2 * pair["Lf2B"] + alpha1 * pair["LfB"] + alpha0 * pair["B"]))
        constraints.append(constraint)
        
    
    
    prob = cp.Problem(objective, constraints)
    
    prob.solve(solver=cp.CLARABEL, verbose=False)

    if u.value is None:
        raise RuntimeError("QP failed to solve (u is None). Check feasibility / solver availability.")

    return np.array(u.value).reshape(-1), np.array(delta.value).reshape(-1)


# ----------------------------
# Quadrotor Agent
# ----------------------------

class QuadrotorAgent:
    def __init__(self, agent_id: int, cfg: QuadConfig, x0_full, motor_state0, env: Environment):
        """
        x0_full: you currently use 14 states: [12 physical + zeta1 + zeta2]
        We'll store:
          - physical state (12)
          - zeta1, zeta2
          - motor_state (4)
        """
        self.id = int(agent_id)
        self.cfg = cfg
        self.env = env

        x0_full = np.array(x0_full, dtype=float).reshape(-1)
        assert x0_full.size == 14, "Expected x0_full of length 14: 12 states + zeta1 + zeta2"

        self.state = x0_full[:12].copy()
        self.zeta1 = float(x0_full[12])
        self.zeta2 = float(x0_full[13])

        self.motor_state = np.array(motor_state0, dtype=float).reshape(4)

        # logs
        self.log = {
            "t": [],
            "state": [],
            "zeta1": [],
            "zeta2": [],
            "u": [],
            "control_inputs": [],
            "motor_cmd": [],
            "eta1_2": [],
            "eta2_1": [],
            "xi1_1": [],
            "xi2_1": [],
        }

    def step(self, t, dt, world_snapshot):
        terms = compute_tracking_terms(
            agent_id=self.id,
            state=self.state,
            zeta1=self.zeta1,
            zeta2=self.zeta2,
            cfg=self.cfg,
            env=self.env,
            t=t,
            world_snapshot=world_snapshot,
        )
        D = terms["D"]
        rhs = terms["rhs"]
        cbf_terms = terms["cbf_terms"]
        pair_cbf_terms = terms["pair_cbf_terms"]  
        tracking_vars = terms["tracking_vars"]

        u, delta = solve_qp_for_u(D, rhs, cbf_terms, pair_cbf_terms, self.cfg)

        # u_new, up, uq, ur = u.value
        u_new = float(u[0])
        up = float(u[1])
        uq = float(u[2])
        ur = float(u[3])

        # 3) Update zetas
        self.zeta1 += dt * self.zeta2
        self.zeta2 += dt * u_new

        # 4) Convert to motor speeds and simulate dynamics
        ControlInputs = np.array([up, uq, ur, self.zeta1], dtype=float)
        MotorSpeed = Torques2AsctecMotorSpeeds(ControlInputs, self.env.M_map, self.env.sim)

        self.state = PelicanModelDirect(
            time=t, dt=dt,
            state=self.state,
            control_inputs=ControlInputs,
            system_parameters=self.env.system,
            sim_parameters=self.env.sim
        )

        # 5) log
        self.log["t"].append(t)
        self.log["state"].append(self.state.copy())
        self.log["zeta1"].append(self.zeta1)
        self.log["zeta2"].append(self.zeta2)
        self.log["u"].append(u.copy())
        self.log["control_inputs"].append(ControlInputs.copy())
        # self.log["motor_cmd"].append(MotorSpeed.copy())
        self.log["eta1_2"].append(tracking_vars["eta1_2"])
        self.log["eta2_1"].append(tracking_vars["eta2_1"])
        self.log["xi1_1"].append(tracking_vars["xi1_1"])
        self.log["xi2_1"].append(tracking_vars["xi2_1"])
        
        self.log.setdefault("pair_cbf_terms", []).append(pair_cbf_terms)

# ----------------------------
# Multi-agent simulation "server"
# ----------------------------

class MultiQuadSim:
    def __init__(self, agents):
        self.agents = list(agents)
        
    def snapshot(self):
        """
        Returns a dict: id -> dict(state=..., zeta1=..., zeta2=...)
        This is what we'll pass into controllers for pairwise computations.
        """
        snap = {}
        for a in self.agents:
            snap[a.id] = {
                "state": a.state.copy(),   # (12,)
                "zeta1": float(a.zeta1),
                "zeta2": float(a.zeta2),
                "name": a.cfg.name,
            }
        return snap


    def run(self, Tmax, dt):
        T = np.arange(0.0, Tmax + dt, dt)
        for t in T:
            snap = self.snapshot()  # same snapshot for all agents at time t
            for agent in self.agents:
                agent.step(t, dt, world_snapshot=snap)
        return T



def plot_trajectories_3d(
    agents,
    save_path="traj_3d.eps",
    width_in=10.0,
    height_scale=0.6,
    label_fontsize=18,
    tick_fontsize=14,
    legend_fontsize=14,
):
    height_in = width_in * height_scale

    fig = plt.figure(figsize=(width_in, height_in))
    ax = fig.add_subplot(111, projection='3d')

    for agent in agents:
        st = np.array(agent.log["state"])
        x = st[:, 6]
        y = st[:, 7]
        z = st[:, 8]

        ax.plot3D([x[0]], [y[0]], [z[0]], 'o', markersize=6)
        ax.plot3D(x, y, z, linewidth=2, label=agent.cfg.name)

        x_path = np.linspace(agent.cfg.plot_x_min, agent.cfg.plot_x_max, 500)
        y_path = agent.cfg.des_y + agent.cfg.A * np.sin(agent.cfg.w * (x_path - agent.cfg.des_x))
        z_path = np.full_like(x_path, agent.cfg.des_height)

        ax.plot3D(x_path, y_path, z_path, '--', linewidth=1)

    ax.set_xlabel("x", fontsize=label_fontsize, labelpad=10)
    ax.set_ylabel("y", fontsize=label_fontsize, labelpad=10)
    ax.set_zlabel("z", fontsize=label_fontsize, labelpad=10)

    ax.tick_params(labelsize=tick_fontsize)
    ax.legend(fontsize=legend_fontsize, frameon=True, loc="best", handlelength=2.5)
    ax.grid(True)

    fig.tight_layout()
    fig.savefig(save_path, format="eps", bbox_inches="tight")
    plt.show()


def plot_xy(agents, save_path="traj_xy.eps"):
    fig = plt.figure()

    for agent in agents:
        st = np.array(agent.log["state"])
        x = st[:, 6]
        y = st[:, 7]
        plt.plot(x, y, linewidth=2, label=agent.cfg.name)

        x_path = np.linspace(agent.cfg.plot_x_min, agent.cfg.plot_x_max, 500)
        y_path = agent.cfg.des_y + agent.cfg.A * np.sin(agent.cfg.w * (x_path - agent.cfg.des_x))
        plt.plot(x_path, y_path, '--', linewidth=1)

        plt.plot([x[0]], [y[0]], 'o')

    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.title("XY Trajectories (Multi-Quad)")

    fig.tight_layout()
    fig.savefig(save_path, format="eps", bbox_inches="tight")
    plt.show()

# ----------------------------
# Example setup
# ----------------------------

def make_env():
    # system params (shared)
    L = 0.2656
    Ix = 0.01152
    Iy = 0.01152
    Iz = 0.0218
    g = 9.8
    m = 1.923
    d = 1.0

    system_params = {
        "m": m, "g": g, "L": L, "Ix": Ix, "Iy": Iy, "Iz": Iz,
        "d": d,
        "kt": 0.01,
        "kr": 0.01
    }

    sim_params = {
        "quantize2int": 0,
        "disturbanceX": 0,
        "disturbanceY": 0,
        "disturbanceZ": 0
    }

    # mapping matrix for motor mixer
    M_map = np.array([
        [0,  0,  -L,  L],
        [L, -L,   0,  0],
        [-d, -d,  d,  d],
        [1,  1,   1,  1]
    ], dtype=float)

    return Environment(system_params, sim_params, M_map)





def plot_pairwise_B(
    agents, D_s,
    save_path="pairwise_B.eps",
    width_in=10.0,          # choose width in inches
    label_fontsize=18,
    tick_fontsize=14,
    legend_fontsize=14
):
    if len(agents) < 2:
        print("Need at least 2 agents to plot pairwise distances/barriers.")
        return

    T = np.array(agents[0].log["t"], dtype=float)
    if T.size == 0:
        print("No logged data found. Run the simulation first.")
        return

    positions = []
    names = []
    for ag in agents:
        st = np.array(ag.log["state"], dtype=float)
        if st.shape[0] != T.size:
            raise ValueError("Agent logs have mismatched time lengths.")
        positions.append(st[:, 6:9])  # x,y,z
        names.append(ag.cfg.name)

    height_in = width_in / 3.0  # 1/3 height-to-width
    fig, ax = plt.subplots(figsize=(width_in, height_in))

    for i, j in itertools.combinations(range(len(agents)), 2):
        dp = positions[i] - positions[j]
        dist2 = np.sum(dp * dp, axis=1)
        B = dist2 - float(D_s)**2
        ax.plot(T, B, linewidth=2, label=f"{names[i]}-{names[j]}")

    ax.grid(True)
    ax.set_xlabel("t [s]", fontsize=label_fontsize)
    ax.set_ylabel(r"$h_{ij}(t)=\|p_i-p_j\|^2-D_s^2$", fontsize=label_fontsize)

    ax.tick_params(axis="both", labelsize=tick_fontsize)

    ax.legend(
        fontsize=legend_fontsize,
        frameon=True,
        loc="best",
        handlelength=2.5
    )

    # no title (intentionally)

    fig.tight_layout()
    fig.savefig(save_path, format="eps", bbox_inches="tight")
    plt.show()

def plot_eta_terms_separately(
    agents,
    save_prefix="eta_terms",
    width_in=10.0,
    height_in=3.2,
    label_fontsize=18,
    tick_fontsize=14,
    legend_fontsize=14
):
    if len(agents) == 0:
        print("No agents provided.")
        return

    T = np.array(agents[0].log["t"], dtype=float)
    if T.size == 0:
        print("No logged data found. Run the simulation first.")
        return

    # -------- eta1_2 --------
    fig1, ax1 = plt.subplots(figsize=(width_in, height_in))
    for ag in agents:
        eta1_2_log = np.array(ag.log["eta1_2"], dtype=float)
        ax1.plot(T, eta1_2_log, linewidth=2, label=ag.cfg.name)

    ax1.grid(True)
    ax1.set_xlabel("t [s]", fontsize=label_fontsize)
    ax1.set_ylabel(r"$\eta_{1,2}$", fontsize=label_fontsize)
    ax1.tick_params(axis="both", labelsize=tick_fontsize)
    ax1.legend(fontsize=legend_fontsize, frameon=True, loc="best", handlelength=2.5)
    fig1.tight_layout()
    fig1.savefig(f"{save_prefix}_eta1_2.eps", format="eps", bbox_inches="tight")
    plt.show()

    # -------- eta2_1 --------
    fig2, ax2 = plt.subplots(figsize=(width_in, height_in))
    for ag in agents:
        eta2_1_log = np.array(ag.log["eta2_1"], dtype=float)
        ax2.plot(T, eta2_1_log, linewidth=2, label=ag.cfg.name)

    ax2.grid(True)
    ax2.set_xlabel("t [s]", fontsize=label_fontsize)
    ax2.set_ylabel(r"$\eta_{2,1}$", fontsize=label_fontsize)
    ax2.tick_params(axis="both", labelsize=tick_fontsize)
    ax2.legend(fontsize=legend_fontsize, frameon=True, loc="best", handlelength=2.5)
    fig2.tight_layout()
    fig2.savefig(f"{save_prefix}_eta2_1.eps", format="eps", bbox_inches="tight")
    plt.show()
    
    
def plot_xi_terms_separately(
    agents,
    save_prefix="xi_terms",
    width_in=10.0,
    height_in=3.2,
    label_fontsize=18,
    tick_fontsize=14,
    legend_fontsize=14
):
    if len(agents) == 0:
        print("No agents provided.")
        return

    T = np.array(agents[0].log["t"], dtype=float)
    if T.size == 0:
        print("No logged data found. Run the simulation first.")
        return

    # -------- xi1_1 --------
    fig1, ax1 = plt.subplots(figsize=(width_in, height_in))
    for ag in agents:
        xi1_1_log = np.array(ag.log["xi1_1"], dtype=float)
        ax1.plot(T, xi1_1_log, linewidth=2, label=ag.cfg.name)

    ax1.grid(True)
    ax1.set_xlabel("t [s]", fontsize=label_fontsize)
    ax1.set_ylabel(r"$\xi_{1,1}$", fontsize=label_fontsize)
    ax1.tick_params(axis="both", labelsize=tick_fontsize)
    ax1.legend(fontsize=legend_fontsize, frameon=True, loc="best", handlelength=2.5)
    fig1.tight_layout()
    fig1.savefig(f"{save_prefix}xi1_1.eps", format="eps", bbox_inches="tight")
    plt.show()

    # -------- xi2_1 --------
    fig2, ax2 = plt.subplots(figsize=(width_in, height_in))
    for ag in agents:
        xi2_1_log = np.array(ag.log["xi2_1"], dtype=float)
        ax2.plot(T, xi2_1_log, linewidth=2, label=ag.cfg.name)

    ax2.grid(True)
    ax2.set_xlabel("t [s]", fontsize=label_fontsize)
    ax2.set_ylabel(r"$\xi_{2,1}$", fontsize=label_fontsize)
    ax2.tick_params(axis="both", labelsize=tick_fontsize)
    ax2.legend(fontsize=legend_fontsize, frameon=True, loc="best", handlelength=2.5)
    fig2.tight_layout()
    fig2.savefig(f"{save_prefix}_xi2_1.eps", format="eps", bbox_inches="tight")
    plt.show()
    

 
# ── Global style ──────────────────────────────────────────────────────────────
 
# Paul Tol "Bright" — colorblind-safe, prints well in greyscale
_COLORS = [
    "#EE6677",   # rose-red
    "#4477AA",   # steel-blue
    "#228833",   # green
    "#CCBB44",   # gold
    "#66CCEE",   # sky-blue
    "#AA3377",   # purple
    "#BBBBBB",   # grey
]
 
def _apply_base_style():
    matplotlib.rcParams.update({
        "font.family":          "serif",
        "font.serif":           ["Times New Roman", "DejaVu Serif"],
        "mathtext.fontset":     "stix",
        "font.size":            13,
        "axes.titlesize":       13,
        "axes.labelsize":       15,
        "xtick.labelsize":      12,
        "ytick.labelsize":      12,
        "legend.fontsize":      12,
        "lines.linewidth":      2.2,
        "lines.markersize":     7,
        "axes.spines.top":      False,
        "axes.spines.right":    False,
        "axes.linewidth":       1.1,
        "axes.edgecolor":       "#333333",
        "axes.facecolor":       "white",
        "figure.facecolor":     "white",
        "axes.grid":            True,
        "grid.color":           "#DDDDDD",
        "grid.linewidth":       0.8,
        "grid.linestyle":       "-",
        "xtick.direction":      "out",
        "ytick.direction":      "out",
        "xtick.major.width":    1.0,
        "ytick.major.width":    1.0,
        "xtick.minor.visible":  False,
        "ytick.minor.visible":  False,
        "legend.frameon":       True,
        "legend.framealpha":    0.92,
        "legend.edgecolor":     "#CCCCCC",
        "legend.handlelength":  2.4,
        "savefig.dpi":          300,
        "savefig.bbox":         "tight",
        "savefig.pad_inches":   0.05,
    })
 
_apply_base_style()
 
 
def _color(i):
    return _COLORS[i % len(_COLORS)]
 
 
def _sine_path(cfg, n=600):
    """Return (x, y) arrays for the desired sine path of a QuadConfig."""
    x = np.linspace(cfg.plot_x_min, cfg.plot_x_max, n)
    y = cfg.des_y + cfg.A * np.sin(cfg.w * (x - cfg.des_x))
    return x, y
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 1.  3-D trajectory
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_trajectories_3d(
    agents,
    save_path="traj_3d.pdf",
    width_in=7.0,
    aspect=0.72,
    elev=24,
    azim=-50,
):
    _apply_base_style()
    fig = plt.figure(figsize=(width_in, width_in * aspect))
    ax  = fig.add_subplot(111, projection="3d")
 
    # pane colours
    pane_col = (0.97, 0.97, 0.97, 1.0)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = True
        pane.set_facecolor(pane_col)
        pane.set_edgecolor("#CCCCCC")
    ax.grid(True, linewidth=0.6, color="#CCCCCC")
    ax.view_init(elev=elev, azim=azim)
 
    legend_handles = []
 
    for k, agent in enumerate(agents):
        st = np.array(agent.log["state"])
        x, y, z = st[:, 6], st[:, 7], st[:, 8]
        c = _color(k)
 
        # actual trajectory
        ax.plot3D(x, y, z, color=c, linewidth=2, zorder=5)
 
        # start marker
        ax.scatter([x[0]], [y[0]], [z[0]],
                   color=c, s=60, marker="o", zorder=10,
                   edgecolors="white", linewidths=0.8)
 
        # desired sine path at constant height (dashed, lighter)
        xp, yp = _sine_path(agent.cfg)
        zp = np.full_like(xp, agent.cfg.des_height)
        ax.plot3D(xp, yp, zp,
                  color=c, linewidth=4,
                  linestyle=(0, (5, 2)), alpha=0.8, zorder=3)
 
        legend_handles.append(
            Line2D([0], [0], color=c, linewidth=2.2, label=agent.cfg.name)
        )
 
    ax.set_xlabel(r"$x$ [m]", labelpad=8)
    ax.set_ylabel(r"$y$ [m]", labelpad=8)
    ax.set_zlabel(r"$z$ [m]", labelpad=6)
    ax.tick_params(labelsize=10)
    ax.legend(handles=legend_handles, loc="upper left",
              fontsize=11, frameon=True, framealpha=0.92,
              edgecolor="#CCCCCC", handlelength=2.2)
 
    fig.tight_layout(pad=0.4)
    fig.savefig(save_path, format=save_path.rsplit(".", 1)[-1], bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 2.  XY (top-down) trajectory
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_xy(
    agents,
    save_path="traj_xy.pdf",
    width_in=6.5,
    height_in=4.2,
):
    _apply_base_style()
    fig, ax = plt.subplots(figsize=(width_in, height_in))
 
    for k, agent in enumerate(agents):
        st = np.array(agent.log["state"])
        x, y = st[:, 6], st[:, 7]
        c = _color(k)
 
        ax.plot(x, y, color=c, linewidth=2, label=agent.cfg.name, zorder=4)
 
        # start marker
        ax.plot(x[0], y[0], "o", color=c,
                markersize=8, markeredgecolor="white",
                markeredgewidth=0.9, zorder=6)
 
        # desired sine path
        xp, yp = _sine_path(agent.cfg)
        ax.plot(xp, yp,
                color=c, linewidth=4,
                linestyle=(0, (5, 2)), alpha=0.8, zorder=2)
 
    ax.set_xlabel(r"$x$ [m]")
    ax.set_ylabel(r"$y$ [m]")
    ax.legend(loc="best", fontsize=11)
    fig.tight_layout(pad=0.4)
    fig.savefig(save_path, format=save_path.rsplit(".", 1)[-1], bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 3.  Pairwise barrier  h_ij = ||p_i - p_j||² - D_s²
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_pairwise_B(
    agents,
    D_s,
    save_path="pairwise_B.pdf",
    width_in=6.5,
    height_in=2.8,
):
    _apply_base_style()
 
    if len(agents) < 2:
        print("Need ≥ 2 agents.")
        return
 
    T     = np.array(agents[0].log["t"], dtype=float)
    names = [a.cfg.name for a in agents]
    positions = [np.array(a.log["state"])[:, 6:9] for a in agents]
 
    fig, ax = plt.subplots(figsize=(width_in, height_in))
 
    for idx, (i, j) in enumerate(itertools.combinations(range(len(agents)), 2)):
        dp    = positions[i] - positions[j]
        dist2 = np.sum(dp * dp, axis=1)
        B     = dist2 - float(D_s) ** 2
        ax.plot(T, B,
                color=_COLORS[idx % len(_COLORS)],
                linewidth=2.0,
                label=rf"$h_{{{names[i][-1]}{names[j][-1]}}}$",
                zorder=4)
 
    # zero line = safety boundary
    # ax.axhline(0, color="#333333", linewidth=1.3,
    #            linestyle=(0, (3, 2)), zorder=5,
    #            label=r"$h_{ij}=0$")
 
    ax.set_xlabel(r"$t$ [s]")
    ax.set_ylabel(r"$h_{ij}(t)=\|p_i-p_j\|^2-D_s^2$")
    ax.set_xlim(T[0], T[-1])
    ax.set_ylim(bottom=min(0, ax.get_ylim()[0]) - 0.05)
 
    ax.legend(loc="best", ncol=2, fontsize=11)
    fig.tight_layout(pad=0.4)
    fig.savefig(save_path, format=save_path.rsplit(".", 1)[-1], bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 4.  η (tangential) tracking terms  — two panels side-by-side
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_eta_terms_separately(
    agents,
    save_prefix="eta",
    width_in=6.5,
    height_in=2.6,
):
    _apply_base_style()
 
    T = np.array(agents[0].log["t"], dtype=float)
 
    # ── η₂ (angular speed tracking)
    fig1, ax1 = plt.subplots(figsize=(width_in, height_in))
    for k, ag in enumerate(agents):
        ax1.plot(T, np.array(ag.log["eta1_2"]),
                 color=_color(k), linewidth=2.1,
                 label=ag.cfg.name)
        ax1.axhline(ag.cfg.des_speed, color=_color(k), linewidth=1, linestyle="--")
    # ax1.axhline(0, color="#888888", linewidth=0.8, linestyle="--")
    ax1.set_xlabel(r"$t$ [s]")
    ax1.set_ylabel(r"$\eta_{2}^{i}(t)$")
    ax1.set_xlim(T[0], T[-1])
    ax1.legend(loc="best", fontsize=11)
    fig1.tight_layout(pad=0.4)
    fig1.savefig(f"{save_prefix}_eta1_2.pdf", bbox_inches="tight")
    plt.show()
 
    # ── μ₁ (yaw error)
    fig2, ax2 = plt.subplots(figsize=(width_in, height_in))
    for k, ag in enumerate(agents):
        ax2.plot(T, np.array(ag.log["eta2_1"]),
                 color=_color(k), linewidth=2.1,
                 label=ag.cfg.name)
    ax2.axhline(0, color="#888888", linewidth=0.8, linestyle="--")
    ax2.set_xlabel(r"$t$ [s]")
    ax2.set_ylabel(r"$\mu_{1}^{i}(t)$")
    ax2.set_xlim(T[0], T[-1])
    ax2.legend(loc="best", fontsize=11)
    fig2.tight_layout(pad=0.4)
    fig2.savefig(f"{save_prefix}_eta2_1.pdf", bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 5.  ξ (radial / altitude) tracking terms
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_xi_terms_separately(
    agents,
    save_prefix="xi",
    width_in=6.5,
    height_in=2.6,
):
    _apply_base_style()
 
    T = np.array(agents[0].log["t"], dtype=float)
 
    # ── ξ₁ (radial error)
    fig1, ax1 = plt.subplots(figsize=(width_in, height_in))
    for k, ag in enumerate(agents):
        ax1.plot(T, np.array(ag.log["xi1_1"]),
                 color=_color(k), linewidth=2.1,
                 label=ag.cfg.name)
    ax1.axhline(0, color="#888888", linewidth=0.8, linestyle="--")
    ax1.set_xlabel(r"$t$ [s]")
    ax1.set_ylabel(r"$\xi_{1}^{i}(t)$")
    ax1.set_xlim(T[0], T[-1])
    ax1.legend(loc="best", fontsize=11)
    fig1.tight_layout(pad=0.4)
    fig1.savefig(f"{save_prefix}_xi1_1.pdf", bbox_inches="tight")
    plt.show()
 
    # ── ζ₁ (altitude error)
    fig2, ax2 = plt.subplots(figsize=(width_in, height_in))
    for k, ag in enumerate(agents):
        ax2.plot(T, np.array(ag.log["xi2_1"]),
                 color=_color(k), linewidth=2.1,
                 label=ag.cfg.name)
    ax2.axhline(0, color="#888888", linewidth=0.8, linestyle="--")
    ax2.set_xlabel(r"$t$ [s]")
    ax2.set_ylabel(r"$\zeta_{1}^{i}(t)$")
    ax2.set_xlim(T[0], T[-1])
    ax2.legend(loc="best", fontsize=11)
    fig2.tight_layout(pad=0.4)
    fig2.savefig(f"{save_prefix}_xi2_1.pdf", bbox_inches="tight")
    plt.show()
 
 
# ══════════════════════════════════════════════════════════════════════════════
# BONUS — combined 2×2 summary figure (all tracking errors in one shot)
# ══════════════════════════════════════════════════════════════════════════════
 
def plot_tracking_summary(
    agents,
    save_path="tracking_summary.pdf",
    width_in=7.0,
    height_in=5.6,
):
    """
    2×2 grid:  top-left ξ₁,  top-right ξ₂(=altitude),
               bottom-left η₂, bottom-right μ₁.
    Perfect for a single full-column figure in a CDC paper.
    """
    _apply_base_style()
 
    T = np.array(agents[0].log["t"], dtype=float)
 
    fig, axes = plt.subplots(2, 2, figsize=(width_in, height_in),
                             sharex=True)
 
    panels = [
        (axes[0, 0], "xi1_1",  r"$\xi_{1}^{i}(t)$"),
        (axes[0, 1], "xi2_1",  r"$\zeta_{1}^{i}(t)$"),
        (axes[1, 0], "eta1_2", r"$\eta_{2}^{i}(t)$"),
        (axes[1, 1], "eta2_1", r"$\mu_{1}^{i}(t)$"),
    ]
 
    legend_handles = [
        Line2D([0], [0], color=_color(k), linewidth=2, label=ag.cfg.name)
        for k, ag in enumerate(agents)
    ]
 
    for ax, key, ylabel in panels:
        for k, ag in enumerate(agents):
            ax.plot(T, np.array(ag.log[key]),
                    color=_color(k), linewidth=2.0)
            if key=="eta1_2":
                ax.axhline(ag.cfg.des_speed, color=_color(k), linewidth=1, linestyle="--")
        if key != "eta1_2":
            ax.axhline(0, color="#888888", linewidth=0.7, linestyle="--")
        ax.set_ylabel(ylabel)
        ax.set_xlim(T[0], T[-1])
        ax.yaxis.set_major_locator(ticker.MaxNLocator(4))
 
    for ax in axes[1]:
        ax.set_xlabel(r"$t$ [s]")
 
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(agents),
        fontsize=11,
        frameon=True,
        framealpha=0.92,
        edgecolor="#CCCCCC",
        bbox_to_anchor=(0.5, -0.02),
    )
 
    fig.tight_layout(pad=0.5, rect=[0, 0.06, 1, 1])
    fig.savefig(save_path, format=save_path.rsplit(".", 1)[-1], bbox_inches="tight")
    plt.show()
    
def _quadrotor_screen_geometry(
    ax,
    x,
    y,
    yaw,
    body_half_len_px=18,
    arm_half_len_px=12,
    nose_offset_px=20,
):
    """
    Build a top-view quadrotor shape centered at (x,y), using yaw = psi.

    Important:
    The geometry is constructed in DISPLAY/SREEN coordinates and then mapped
    back to data coordinates. This keeps the quadrotor visually well-shaped
    even when the XY axes are very skewed (large x-range, small y-range).
    """
    center_disp = ax.transData.transform((x, y))
    inv = ax.transData.inverted()

    # Forward direction in display space, computed from the physical yaw in data space
    fwd_disp = ax.transData.transform((x + np.cos(yaw), y + np.sin(yaw))) - center_disp
    lat_disp = ax.transData.transform((x - np.sin(yaw), y + np.cos(yaw))) - center_disp

    nf = np.linalg.norm(fwd_disp)
    nl = np.linalg.norm(lat_disp)

    if nf < 1e-12:
        fwd_disp = np.array([1.0, 0.0])
        nf = 1.0
    if nl < 1e-12:
        lat_disp = np.array([0.0, 1.0])
        nl = 1.0

    fwd_hat = fwd_disp / nf
    lat_hat = lat_disp / nl

    # Main body bar
    body_a = inv.transform(center_disp - body_half_len_px * fwd_hat)
    body_b = inv.transform(center_disp + body_half_len_px * fwd_hat)

    # Perpendicular arm bar (gives an actual quadrotor-like top view)
    arm_a = inv.transform(center_disp - arm_half_len_px * lat_hat)
    arm_b = inv.transform(center_disp + arm_half_len_px * lat_hat)

    # Nose marker to show heading clearly
    nose = inv.transform(center_disp + nose_offset_px * fwd_hat)

    return body_a, body_b, arm_a, arm_b, nose


def _disp_poly_to_data(ax, pts_disp):
    pts_data = ax.transData.inverted().transform(np.asarray(pts_disp, dtype=float))
    return pts_data[:, 0], pts_data[:, 1]


def _circle_disp(center_disp, radius_px, n=40):
    ang = np.linspace(0.0, 2.0 * np.pi, n)
    return np.column_stack([
        center_disp[0] + radius_px * np.cos(ang),
        center_disp[1] + radius_px * np.sin(ang),
    ])


def _set_line_from_disp(line, ax, pts_disp):
    xdat, ydat = _disp_poly_to_data(ax, pts_disp)
    line.set_data(xdat, ydat)


def _quadrotor_xy_primitives(
    ax,
    x,
    y,
    yaw,
    rotor_phase=0.0,
    hub_r_px=7.0,
    arm_len_px=22.0,
    motor_r_px=4.2,
    rotor_r_px=9.0,
    spine_len_px=11.0,
):
    """
    Build a realistic-looking 2D top-view quadrotor in DISPLAY coordinates first,
    then map it back to data coordinates later.

    This avoids visual distortion when x and y axis scales are very different.
    """
    center = ax.transData.transform((x, y))

    # Forward direction from physical yaw, projected into screen space
    fwd_try = ax.transData.transform((x + np.cos(yaw), y + np.sin(yaw))) - center
    nf = np.linalg.norm(fwd_try)
    if nf < 1e-12:
        fwd = np.array([1.0, 0.0])
    else:
        fwd = fwd_try / nf

    # Use an orthogonal screen-space lateral vector so the body keeps its shape
    lat = np.array([-fwd[1], fwd[0]])

    # X-configuration arms (top-view quadrotor look)
    d1 = (fwd + lat) / np.sqrt(2.0)
    d2 = (-fwd + lat) / np.sqrt(2.0)
    d3 = (-fwd - lat) / np.sqrt(2.0)
    d4 = (fwd - lat) / np.sqrt(2.0)
    motor_dirs = [d1, d2, d3, d4]

    motor_centers = [center + arm_len_px * d for d in motor_dirs]

    arms = [
        np.vstack([center + hub_r_px * d, mc])
        for d, mc in zip(motor_dirs, motor_centers)
    ]

    hub = _circle_disp(center, hub_r_px, n=36)

    # Short body spine inside the hub to suggest front/back without using a marker
    # spine = np.vstack([
    #     center - 0.45 * spine_len_px * fwd,
    #     center + 1.00 * spine_len_px * fwd,
    # ])

    motors = [_circle_disp(mc, motor_r_px, n=28) for mc in motor_centers]
    rotors = [_circle_disp(mc, rotor_r_px, n=36) for mc in motor_centers]

    # One animated blade diameter per rotor
    spin_sign = [1.0, -1.0, 1.0, -1.0]
    blades = []
    for k, mc in enumerate(motor_centers):
        ang = spin_sign[k] * rotor_phase + k * 0.35
        bdir = np.cos(ang) * fwd + np.sin(ang) * lat
        blade = np.vstack([
            mc - 0.82 * rotor_r_px * bdir,
            mc + 0.82 * rotor_r_px * bdir,
        ])
        blades.append(blade)

    return {
        "hub": hub,
        # "spine": spine,
        "arms": arms,
        "motors": motors,
        "rotors": rotors,
        "blades": blades,
    }

def _desired_sine_path(cfg, n=500):
    x = np.linspace(cfg.plot_x_min, cfg.plot_x_max, n)
    y = cfg.des_y + cfg.A * np.sin(cfg.w * (x - cfg.des_x))
    return x, y


def animate_xy_quadrotors_gif_sine(
    agents,
    save_path="traj_xy_sine_anim.gif",
    fps=25,
    stride=6,
    trail_length=120,
    figsize=(10.0, 4.8),
    dpi=100,
    dark_theme=True,
    arm_len_px=22.0,
    hub_r_px=7.0,
    motor_r_px=4.2,
    rotor_r_px=9.0,
):
    """
    Aesthetic 2D GIF animation for sinusoidal-path simulation.

    Uses display-space quadrotor geometry so the airframe remains visually
    correct even when x and y axes have very different scales.
    """
    if len(agents) == 0:
        print("No agents provided.")
        return

    all_states = [np.array(ag.log["state"], dtype=float)[::stride] for ag in agents]
    n_frames = min(st.shape[0] for st in all_states)
    all_states = [st[:n_frames] for st in all_states]
    n_agents = len(agents)

    cmap = plt.get_cmap("tab10")
    colors = [cmap(i) for i in range(n_agents)]

    T_full = np.array(agents[0].log["t"], dtype=float)
    dt_sim = float(T_full[1] - T_full[0]) if T_full.size >= 2 else 0.01

    # bounds from actual trajectories + desired sine paths
    all_x = []
    all_y = []

    for st in all_states:
        all_x.append(st[:, 6])
        all_y.append(st[:, 7])

    for ag in agents:
        xd, yd = _desired_sine_path(ag.cfg)
        all_x.append(xd)
        all_y.append(yd)

    all_x = np.concatenate(all_x)
    all_y = np.concatenate(all_y)

    xr = float(np.max(all_x) - np.min(all_x))
    yr = float(np.max(all_y) - np.min(all_y))

    x_pad = max(0.03 * xr, 0.8)
    y_pad = max(0.12 * yr, 0.8)

    bg = "#16213e" if dark_theme else "white"
    grid_c = "#223355" if dark_theme else "#DDDDDD"
    tick_c = "#AFC0D4" if dark_theme else "#333333"
    label_c = "white" if dark_theme else "#111111"
    legend_face = "#16213e" if dark_theme else "white"
    legend_edge = "#445566" if dark_theme else "#CCCCCC"

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, facecolor=bg)
    ax.set_facecolor(bg)

    ax.set_xlim(np.min(all_x) - x_pad, np.max(all_x) + x_pad)
    ax.set_ylim(np.min(all_y) - y_pad, np.max(all_y) + y_pad)

    ax.set_xlabel(r"$x$ [m]", fontsize=12, color=label_c)
    ax.set_ylabel(r"$y$ [m]", fontsize=12, color=label_c)
    ax.tick_params(colors=tick_c, labelsize=9)
    ax.grid(True, linewidth=0.35, color=grid_c, linestyle="--")

    for spine in ax.spines.values():
        spine.set_color("#334466" if dark_theme else "#CCCCCC")

    # desired paths
    for ag, col in zip(agents, colors):
        xd, yd = _desired_sine_path(ag.cfg)
        ax.plot(
            xd, yd,
            linestyle="--",
            color=col,
            linewidth=1.0,
            alpha=0.36 if dark_theme else 0.55,
            zorder=1
        )

    # need draw so transforms are valid
    fig.canvas.draw()

    trail_lines = []
    agent_artists = []
    legend_handles = []

    for k, ag in enumerate(agents):
        c = colors[k]
        dark = tuple(0.35 * np.array(c[:3]))

        trail, = ax.plot([], [], "-", color=c, linewidth=1.15, alpha=0.62, zorder=2)
        hub, = ax.plot([], [], color=c, linewidth=1.2, zorder=6)

        arms = [ax.plot([], [], color=c, linewidth=2.4, zorder=5, solid_capstyle="round")[0] for _ in range(4)]
        motors = [ax.plot([], [], color=dark, linewidth=0.85, zorder=7)[0] for _ in range(4)]
        rotors = [ax.plot([], [], color=c, linewidth=1.3, alpha=0.72, zorder=4)[0] for _ in range(4)]
        blades_main = [ax.plot([], [], color=dark, linewidth=1.7, zorder=8, solid_capstyle="round")[0] for _ in range(4)]
        blades_cross = [ax.plot([], [], color=dark, linewidth=1.15, alpha=0.80, zorder=8, solid_capstyle="round")[0] for _ in range(4)]

        trail_lines.append(trail)
        agent_artists.append({
            "trail": trail,
            "hub": hub,
            "arms": arms,
            "motors": motors,
            "rotors": rotors,
            "blades_main": blades_main,
            "blades_cross": blades_cross,
        })

        legend_handles.append(Line2D([0], [0], color=c, linewidth=3, label=ag.cfg.name))

    ax.legend(
        handles=legend_handles,
        fontsize=10,
        loc="upper right",
        facecolor=legend_face,
        edgecolor=legend_edge,
        labelcolor=label_c if dark_theme else "#111111",
        framealpha=0.92
    )

    time_text = ax.text(
        0.03, 0.95, "",
        transform=ax.transAxes,
        fontsize=12,
        color=label_c,
        va="top",
        fontfamily="monospace"
    )

    rotor_phases = np.zeros(n_agents, dtype=float)
    rad_per_frame = (150.0 / 60.0) * 2.0 * np.pi * (stride * dt_sim)

    def _init():
        out = [time_text]
        time_text.set_text("")

        for art in agent_artists:
            art["trail"].set_data([], [])
            art["hub"].set_data([], [])
            for ln in art["arms"] + art["motors"] + art["rotors"] + art["blades_main"] + art["blades_cross"]:
                ln.set_data([], [])
            out.extend(
                [art["trail"], art["hub"]]
                + art["arms"] + art["motors"] + art["rotors"]
                + art["blades_main"] + art["blades_cross"]
            )
        return out

    def _update(frame):
        nonlocal rotor_phases
        rotor_phases += rad_per_frame

        out = []
        for i, (ag, st, c) in enumerate(zip(agents, all_states, colors)):
            s = st[frame]
            x, y = s[6], s[7]
            yaw = s[2]

            if trail_length is None or trail_length <= 0:
                s0 = 0
            else:
                s0 = max(0, frame - int(trail_length) + 1)

            trail = st[s0:frame+1, 6:8]
            agent_artists[i]["trail"].set_data(trail[:, 0], trail[:, 1])

            prim = _quadrotor_xy_primitives_realistic(
                ax=ax,
                x=x,
                y=y,
                yaw=yaw,
                rotor_phase=rotor_phases[i],
                arm_len_px=arm_len_px,
                hub_r_px=hub_r_px,
                motor_r_px=motor_r_px,
                rotor_r_px=rotor_r_px,
            )

            _set_line_from_disp(agent_artists[i]["hub"], ax, prim["hub"])

            for ln, pts in zip(agent_artists[i]["arms"], prim["arms"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["motors"], prim["motors"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["rotors"], prim["rotors"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["blades_main"], prim["blades_main"]):
                _set_line_from_disp(ln, ax, pts)

            for ln, pts in zip(agent_artists[i]["blades_cross"], prim["blades_cross"]):
                _set_line_from_disp(ln, ax, pts)

            out.extend(
                [agent_artists[i]["trail"], agent_artists[i]["hub"]]
                + agent_artists[i]["arms"] + agent_artists[i]["motors"]
                + agent_artists[i]["rotors"] + agent_artists[i]["blades_main"]
                + agent_artists[i]["blades_cross"]
            )

        time_text.set_text(f"t = {frame * stride * dt_sim:6.2f} s")
        out.append(time_text)
        return out

    print(f"[animate_xy_quadrotors_gif_sine] {n_frames} frames | stride={stride} | fps={fps} -> {save_path}")

    anim = FuncAnimation(
        fig,
        _update,
        frames=n_frames,
        init_func=_init,
        interval=1000 / fps,
        blit=False
    )

    anim.save(
        save_path,
        writer=PillowWriter(fps=fps),
        progress_callback=lambda i, n: print(f"\r  rendering {i+1}/{n}", end="", flush=True)
    )

    plt.close(fig)
    print(f"\n[animate_xy_quadrotors_gif_sine] Done -> {save_path}")
    
    
    
    
# ─────────────────────────────────────────────────────────────────────────────
# Rotation / geometry helpers
# ─────────────────────────────────────────────────────────────────────────────

def _R_quad(ph, th, ps):
    """ZYX Euler rotation: body -> world."""
    cph, sph = np.cos(ph), np.sin(ph)
    cth, sth = np.cos(th), np.sin(th)
    cps, sps = np.cos(ps), np.sin(ps)

    return np.array([
        [cth * cps,  sph * sth * cps - cph * sps,  cph * sth * cps + sph * sps],
        [cth * sps,  sph * sth * sps + cph * cps,  cph * sth * sps - sph * cps],
        [-sth,       sph * cth,                    cph * cth]
    ], dtype=float)


def _body_to_world(pts_body, R, origin):
    pts_body = np.asarray(pts_body, dtype=float)
    origin = np.asarray(origin, dtype=float).reshape(1, 3)
    return (R @ pts_body.T).T + origin


def _circle_pts_body(cx, cy, cz, r, n=24):
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=True)
    return np.column_stack([
        cx + r * np.cos(t),
        cy + r * np.sin(t),
        np.full_like(t, cz)
    ])


def _cylinder_rings_body(cx, cy, cz_bot, r, h, n=24):
    bot = _circle_pts_body(cx, cy, cz_bot, r, n=n)
    top = _circle_pts_body(cx, cy, cz_bot + h, r, n=n)
    verticals = [np.array([bot[i], top[i]]) for i in range(0, n, 4)]
    return bot, top, verticals


def _plot3d_line(ax, pts_world, color, lw, alpha=1.0, ls="-", zorder=None):
    line, = ax.plot(
        pts_world[:, 0], pts_world[:, 1], pts_world[:, 2],
        color=color,
        linewidth=lw,
        alpha=alpha,
        linestyle=ls,
        solid_capstyle="round",
        zorder=zorder
    )
    return line


def _desired_sine_path_3d(cfg, n=240):
    x = np.linspace(cfg.plot_x_min, cfg.plot_x_max, n)
    y = cfg.des_y + cfg.A * np.sin(cfg.w * (x - cfg.des_x))
    z = np.full_like(x, cfg.des_height)
    return x, y, z


# ─────────────────────────────────────────────────────────────────────────────
# Realistic line-only 3D quadrotor
# ─────────────────────────────────────────────────────────────────────────────

def _draw_quad_3d_realistic(
    ax,
    x, y, z, ph, th, ps,
    color,
    rotor_phase,
    scale=1.0,
    arm_lw=2.4,
    hub_lw=1.15,
    mot_lw=0.75,
    disc_lw=1.35,
    blade_lw=1.9,
):
    """
    Pelican-like line-only quadrotor rendering.

    Pure ax.plot() lines only, so it avoids most matplotlib 3D depth-sort issues.
    """
    # Pelican-ish geometry
    ARM   = 0.133 * scale
    RDISC = 0.060 * scale
    RMOT  = 0.018 * scale
    HMOT  = 0.022 * scale
    HR    = 0.028 * scale
    HH    = 0.016 * scale

    SQ2 = np.sqrt(2.0) / 2.0
    motor_pos = ARM * np.array([
        [ SQ2,  SQ2, 0.0],
        [-SQ2,  SQ2, 0.0],
        [-SQ2, -SQ2, 0.0],
        [ SQ2, -SQ2, 0.0],
    ], dtype=float)

    rotor_spin = [+1, -1, +1, -1]

    origin = np.array([x, y, z], dtype=float)
    R = _R_quad(ph, th, ps)

    c = color[:3] if len(color) >= 3 else color
    dark = tuple(0.35 * np.array(c))

    arts = []

    def _add(pts_body, col, lw, alpha=1.0, ls="-", zorder=None):
        pts_world = _body_to_world(pts_body, R, origin)
        arts.append(_plot3d_line(ax, pts_world, col, lw, alpha=alpha, ls=ls, zorder=zorder))

    # Arms
    for mp in motor_pos:
        start = mp * (HR / ARM)
        _add(np.array([start, mp]), c, arm_lw, zorder=8)

    # Hub cylinder
    hub_bot, hub_top, hub_verticals = _cylinder_rings_body(0.0, 0.0, -HH/2, HR, HH, n=24)
    _add(hub_bot, c, hub_lw, zorder=9)
    _add(hub_top, c, hub_lw, zorder=9)
    for v in hub_verticals:
        _add(v, c, 0.7 * hub_lw, zorder=9)

    # Motors + rotors
    for k, mp in enumerate(motor_pos):
        mx, my, mz = mp

        mot_bot, mot_top, mot_verticals = _cylinder_rings_body(mx, my, mz - HMOT, RMOT, HMOT, n=14)
        _add(mot_bot, dark, mot_lw, zorder=10)
        _add(mot_top, dark, mot_lw, zorder=10)
        for v in mot_verticals:
            _add(v, dark, 0.6 * mot_lw, zorder=10)

        disc_z = mz + 0.002 * scale
        disc = _circle_pts_body(mx, my, disc_z, RDISC, n=24)
        _add(disc, c, disc_lw, alpha=0.72, zorder=7)

        phase = rotor_phase * rotor_spin[k]
        for angle in [phase, phase + np.pi]:
            blade = np.array([
                [mx - RDISC * 0.15 * np.cos(angle), my - RDISC * 0.15 * np.sin(angle), disc_z],
                [mx + RDISC * 1.00 * np.cos(angle), my + RDISC * 1.00 * np.sin(angle), disc_z],
            ])
            _add(blade, dark, blade_lw, zorder=11)

    return arts


# ─────────────────────────────────────────────────────────────────────────────
# Main 3D GIF animation for sinusoidal-path simulation
# ─────────────────────────────────────────────────────────────────────────────

def animate_trajectories_3d_gif(
    agents,
    save_path="traj_3d_anim.gif",
    fps=25,
    stride=6,
    trail_length=120,
    ax_padding=0.8,
    figsize=(10.5, 8.5),
    dpi=100,
    elev=28.0,
    azim=-55.0,
    rotor_rpm=150.0,
    quad_scale=1.0,
    dark_theme=True,
):
    """
    Aesthetic 3D GIF animation for the sinusoidal-path simulation.

    Parameters
    ----------
    agents : list
        Agents after sim.run(...)
    save_path : str
        Output gif path
    fps : int
        Frames per second
    stride : int
        Use every `stride`-th logged sample as one rendered frame
    trail_length : int or None
        Number of latest rendered points shown in the trail.
        If None or <= 0, full history is shown.
    ax_padding : float
        Extra padding around trajectory bounds
    quad_scale : float
        Visual quadrotor scale
    """
    if len(agents) == 0:
        print("No agents provided.")
        return

    all_states = [np.array(ag.log["state"], dtype=float)[::stride] for ag in agents]
    n_frames = min(st.shape[0] for st in all_states)
    all_states = [st[:n_frames] for st in all_states]
    n_agents = len(agents)

    cmap = plt.get_cmap("tab10")
    colors = [cmap(i) for i in range(n_agents)]

    # infer simulation dt from log
    T_full = np.array(agents[0].log["t"], dtype=float)
    dt_sim = float(T_full[1] - T_full[0]) if T_full.size >= 2 else 0.01

    # axis limits from states + desired sine paths
    all_xyz = np.vstack([st[:, 6:9] for st in all_states])
    xmin, ymin, zmin = all_xyz.min(axis=0) - ax_padding
    xmax, ymax, zmax = all_xyz.max(axis=0) + ax_padding

    for ag in agents:
        xd, yd, zd = _desired_sine_path_3d(ag.cfg, n=220)
        xmin = min(xmin, xd.min() - ax_padding)
        xmax = max(xmax, xd.max() + ax_padding)
        ymin = min(ymin, yd.min() - ax_padding)
        ymax = max(ymax, yd.max() + ax_padding)
        zmin = min(zmin, zd.min() - ax_padding)
        zmax = max(zmax, zd.max() + ax_padding)

    bg = "#16213e" if dark_theme else "white"
    grid_c = "#223355" if dark_theme else "#DDDDDD"
    pane_edge = "#334466" if dark_theme else "#CCCCCC"
    tick_c = "#AFC0D4" if dark_theme else "#333333"
    label_c = "white" if dark_theme else "#111111"
    legend_face = "#16213e" if dark_theme else "white"
    legend_edge = "#445566" if dark_theme else "#CCCCCC"

    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor=bg)
    ax = fig.add_subplot(111, projection="3d", facecolor=bg)

    # panes / grid
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False if dark_theme else True
        if not dark_theme:
            pane.set_facecolor((0.97, 0.97, 0.97, 1.0))
        pane.set_edgecolor(pane_edge)

    ax.set_xlabel(r"$x$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.set_ylabel(r"$y$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.set_zlabel(r"$z$ [m]", fontsize=12, color=label_c, labelpad=8)
    ax.tick_params(colors=tick_c, labelsize=9)
    ax.grid(True, linewidth=0.35, color=grid_c, linestyle="--")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_zlim(zmin, zmax)
    ax.view_init(elev=elev, azim=azim)

    # balanced box aspect
    ax.set_box_aspect((xmax - xmin, ymax - ymin, max(zmax - zmin, 0.8)))

    # desired sinusoidal paths
    for ag, col in zip(agents, colors):
        xd, yd, zd = _desired_sine_path_3d(ag.cfg, n=280)
        ax.plot(
            xd, yd, zd,
            linestyle="--",
            color=col,
            linewidth=2,
            alpha=0.35 if dark_theme else 0.55,
            zorder=2
        )

    # trail artists
    trail_lines = []
    for col in colors:
        ln, = ax.plot([], [], [], "-", color=col, linewidth=1.15, alpha=0.62)
        trail_lines.append(ln)

    time_text = ax.text2D(
        0.03, 0.95, "",
        transform=ax.transAxes,
        fontsize=12,
        color=label_c,
        va="top",
        fontfamily="monospace"
    )

    handles = [
        Line2D([0], [0], color=colors[i], linewidth=3, label=agents[i].cfg.name)
        for i in range(n_agents)
    ]
    ax.legend(
        handles=handles,
        fontsize=10,
        loc="upper right",
        facecolor=legend_face,
        edgecolor=legend_edge,
        labelcolor=label_c if dark_theme else "#111111",
        framealpha=0.92
    )

    fig.tight_layout()

    quad_artists = []
    rotor_phases = np.zeros(n_agents, dtype=float)
    rad_per_frame = (rotor_rpm / 60.0) * 2.0 * np.pi * (stride * dt_sim)

    def _init():
        for ln in trail_lines:
            ln.set_data([], [])
            ln.set_3d_properties([])
        time_text.set_text("")
        return trail_lines + [time_text]

    def _update(frame):
        nonlocal quad_artists, rotor_phases

        # remove old quadrotor drawings
        for ln in quad_artists:
            try:
                ln.remove()
            except Exception:
                pass
        quad_artists = []

        rotor_phases += rad_per_frame

        for i, (ag, st, col) in enumerate(zip(agents, all_states, colors)):
            s = st[frame]
            ph, th, ps = s[0], s[1], s[2]
            x, y, z = s[6], s[7], s[8]

            # moving tail: latest trail_length rendered points only
            if trail_length is None or trail_length <= 0:
                s0 = 0
            else:
                s0 = max(0, frame - int(trail_length) + 1)

            trail = st[s0:frame+1, 6:9]
            trail_lines[i].set_data(trail[:, 0], trail[:, 1])
            trail_lines[i].set_3d_properties(trail[:, 2])

            arts = _draw_quad_3d_realistic(
                ax,
                x, y, z,
                ph, th, ps,
                color=col,
                rotor_phase=rotor_phases[i],
                scale=quad_scale
            )
            quad_artists.extend(arts)

        time_text.set_text(f"t = {frame * stride * dt_sim:6.2f} s")
        return trail_lines + [time_text] + quad_artists

    print(f"[animate_trajectories_3d_gif] {n_frames} frames | stride={stride} | fps={fps} -> {save_path}")

    anim = FuncAnimation(
        fig,
        _update,
        frames=n_frames,
        init_func=_init,
        interval=1000 / fps,
        blit=False
    )

    anim.save(
        save_path,
        writer=PillowWriter(fps=fps),
        progress_callback=lambda i, n: print(f"\r  rendering {i+1}/{n}", end="", flush=True)
    )

    plt.close(fig)
    print(f"\n[animate_trajectories_3d_gif] Done -> {save_path}")
    
    
    
    
def make_agents(env):
    n = 1
    gains = {
        "k1": 200 / n, "k2": 400 / n, "k3": 90 / n, "k4": 30 / n,
        "k5": 100 / n, "k6": 200 / n, "k7": 60 / n, "k8": 20 / n,
        "k9": 10 / n, "k10": 40 / n, "k11": 30 / n, "k12": 30 / n,
        "k13": 10 / n, "k14": 12 / n,
    }

    cbf_lambdas = {"lam_th": 1.1, "lam_ph": 3.5}
    qp_weights = {"Q_u": 1.0, "P_delta": 10.0}

    # desired path: y = des_y + A*sin(w*(x - des_x))
    cfgA = QuadConfig(
        name="quad_A",
        A=5,
        w=0.25,
        des_x=0.0,
        des_y=0.0,
        des_height=1.0,
        des_yaw=0.0,
        des_speed=0.3,
        gains=gains,
        th_max=th_max,
        ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights,
        resp_w=0.5,
        plot_x_min=-5.0,
        plot_x_max=18.0,
        
    )

    # phase-shifted sine path
    cfgB = QuadConfig(
        name="quad_B",
        A=5,
        w=0.25,
        des_x=-np.pi*4,
        des_y=0.0,
        des_height=1.0,
        des_yaw=0.0,
        des_speed=0.25,
        gains=gains,
        th_max=th_max,
        ph_max=ph_max,
        cbf_lambdas=cbf_lambdas,
        qp_weights=qp_weights,
        resp_w=0.5,
        plot_x_min=-5.0,
        plot_x_max=18.0,
        
    )

    x0A = np.array([0, 0, 0, 0, 0, 0, -2.5, -6.0, 0.3, 0, 0, 0, 9.8, 0], dtype=float)
    x0B = np.array([0, 0, 0, 0, 0, 0, 1.25, -7.0, 0.4, 0, 0, 0, 9.8, 0], dtype=float)

    MotorState0 = np.array([10.0, 10.0, 10.0, 10.0], dtype=float)

    return [
        QuadrotorAgent(0, cfgA, x0A, MotorState0, env),
        QuadrotorAgent(1, cfgB, x0B, MotorState0, env),
    ]
    
def main():
    env = make_env()
    agents = make_agents(env)

    sim = MultiQuadSim(agents)
    Tmax = 22
    dt = 0.01

    sim.run(Tmax=Tmax, dt=dt)

    # plots
    # plot_trajectories_3d(agents)
    # plot_xy(agents)
    # plot_pairwise_B(agents, Ds)
    # plot_eta_terms_separately(agents)
    # plot_xi_terms_separately(agents)
    # plot_tracking_summary(agents)
    
#     animate_xy_quadrotors_gif(
#         agents,
#         save_path="traj_xy_anim.gif",
#         fps=25,
#         stride=10,
#         trail_length=None,
#         hub_r_px=7.0/3,
#         arm_len_px=22.0/3,
#         motor_r_px=4.2/3,
#         rotor_r_px=9.0/3,
#     )
    
#     animate_trajectories_3d_gif(
#         agents,
#         save_path="traj_3d_anim.gif",
#         fps=20,
#         stride=10,
#         trail_length=None,
#         quad_scale=0.9,
#         elev=20,
#         azim=-50,
#     )
    
    # animate_trajectories_3d_gif(
    #     agents,
    #     save_path="traj_3d_anim.gif",
    #     fps=25,
    #     stride=7,
    #     trail_length=50,   # moving tail, not full history
    #     figsize=(11, 9),
    #     dpi=100,
    #     elev=24,
    #     azim=-52,
    #     rotor_rpm=150,
    #     quad_scale=3,
    #     dark_theme=False,
    # )
    
    animate_xy_quadrotors_gif_sine(
        agents,
        save_path="traj_xy_sine_anim.gif",
        fps=25,
        stride=6,
        trail_length=50,
        figsize=(11, 5),
        dpi=100,
        dark_theme=False,
        arm_len_px=22.0/2,
        hub_r_px=7.0/2,
        motor_r_px=4.2/2,
        rotor_r_px=9.0/2,
    )


if __name__ == "__main__":
    main()

[animate_xy_quadrotors_gif_sine] 367 frames | stride=6 | fps=25 -> traj_xy_sine_anim.gif
  rendering 367/367
[animate_xy_quadrotors_gif_sine] Done -> traj_xy_sine_anim.gif
